# *Metisa plana* Target Discovery and Candidate Prioritisation

## Computational Analysis Workflow

This notebook presents an integrated computational workflow for analysing the supplied *Metisa plana* predicted proteome and identifying promising candidates for targeted pesticide development. Multiple evidence sources are progressively combined to evaluate sequence quality, homology, functional annotation, expression patterns and evolutionary relationships.

### Overall Workflow

26,490 Proteins → QC → BLASTP → Filtering & Taxonomic Screening → InterProScan → Candidate Mining → Expression Analysis → Phylogeny → Final Ranking → Top 10 Targets


In [1]:
# ============================================================
# ANALYSIS IMPORTS
# ============================================================

import os

import statistics

import subprocess

import re

import pandas as pd

import matplotlib.pyplot as plt

import csv

from collections import defaultdict, Counter

import math

from collections import defaultdict

import numpy as np

from pathlib import Path

import logging

import shutil

import sys



# 01. Input Protein QC & FASTA Preparation

The analysis begins with quality control of the supplied *Metisa plana* predicted proteome.

This stage evaluates the protein sequences for sequence-quality issues such as:

- protein length
- terminal stop characters
- internal stop characters
- ambiguous residues
- invalid characters
- duplicate sequences
- sequence statistics

Terminal stop characters are removed where appropriate and a cleaned protein FASTA is prepared for downstream analysis.

The outputs from this stage provide the validated protein sequence input for the subsequent homology-search workflow.


In [ ]:

RAW_PROTEIN_FASTA = CLEANED_FASTA = FILTERED_FASTA = None
QC_SUMMARY = SEQUENCE_STATISTICS = LENGTH_PLOT = None
BLAST_DATABASE = BLAST_PROGRAM = RAW_BLAST = None

sequences = cleaned_sequences = filtered_sequences = qc_results = None

PROTEIN_ALPHABET = set("ACDEFGHIKLMNPQRSTVWYX")
PROTEIN_DB_EXTS = [".pin", ".psq", ".phr"]

ANNOTATION_OUTFMT = (
    "6 qseqid sseqid pident length "
    "qlen slen qcovs evalue bitscore stitle"
)

DEFAULT_MAX_TARGET_SEQS = "10"
DEFAULT_THREADS = "6"

def expand_path(path):
    return os.path.abspath(
        os.path.expanduser(path.strip().strip('"'))
    )


def check_file_exists(path):
    if not os.path.isfile(path):
        print("\nERROR: File not found:")
        print(path)
        return False
    return True


def section(title):
    print("\n" + "=" * 65)
    print(title)
    print("=" * 65)


def banner():
    print("""
===============================================================
          METISA PLANA — QC + HOMOLOGY SEARCH
===============================================================
""")

def read_fasta(input_file):
    sequences = []
    header = None
    sequence = ""

    try:
        with open(
            input_file,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as file:

            for line in file:
                line = line.strip()

                if not line:
                    continue

                if line.startswith(">"):
                    if header is not None:
                        sequences.append((header, sequence))

                    header = line[1:].strip()
                    sequence = ""

                else:
                    if header is None:
                        raise ValueError(
                            "Sequence data was found before "
                            "the first FASTA header."
                        )

                    sequence += line.upper()

        if header is not None:
            sequences.append((header, sequence))

    except Exception as error:
        print("\nERROR reading FASTA:")
        print(error)
        return None

    return sequences


def calculate_id_statistics(sequences):
    sequence_ids = [
        header.split()[0]
        for header, sequence in sequences
    ]

    counts = {}

    for sequence_id in sequence_ids:
        counts[sequence_id] = counts.get(sequence_id, 0) + 1

    duplicated_groups = {
        sequence_id: count
        for sequence_id, count in counts.items()
        if count > 1
    }

    duplicate_id_groups = len(duplicated_groups)
    duplicate_extra_records = sum(
        count - 1
        for count in duplicated_groups.values()
    )

    return (
        sequence_ids,
        duplicate_id_groups,
        duplicate_extra_records
    )


def calculate_protein_qc_and_prepare(sequences):
    cleaned_records = []
    terminal_stop_count = 0
    internal_stop_count = 0
    sequences_with_x = 0
    total_x = 0
    invalid_counts = {}
    sequences_with_invalid = 0
    empty_after_cleanup = 0

    (
        sequence_ids,
        duplicate_id_groups,
        duplicate_extra_records
    ) = calculate_id_statistics(sequences)

    for header, raw_sequence in sequences:

        sequence = (
            raw_sequence.upper()
            .replace(" ", "")
            .replace("\t", "")
        )

        terminal_stops = 0

        while sequence.endswith("*"):
            sequence = sequence[:-1]
            terminal_stops += 1

        if terminal_stops > 0:
            terminal_stop_count += 1

        if "*" in sequence:
            internal_stop_count += 1

        x_count = sequence.count("X")
        total_x += x_count

        if x_count > 0:
            sequences_with_x += 1

        sequence_invalid = False

        for character in sequence:

            if character == "*":
                continue

            if character not in PROTEIN_ALPHABET:
                invalid_counts[character] = (
                    invalid_counts.get(character, 0) + 1
                )
                sequence_invalid = True

        if sequence_invalid:
            sequences_with_invalid += 1

        cleaned_sequence = sequence

        if len(cleaned_sequence) == 0:
            empty_after_cleanup += 1

        cleaned_records.append(
            (header, cleaned_sequence)
        )

    lengths = [
        len(sequence)
        for header, sequence in cleaned_records
        if len(sequence) > 0
    ]

    if not lengths:
        print("\nERROR: No non-empty protein sequences remain.")
        return None

    min_length = min(lengths)
    max_length = max(lengths)
    mean_length = statistics.mean(lengths)
    median_length = statistics.median(lengths)
    total_length = sum(lengths)

    below_50 = sum(length < 50 for length in lengths)
    below_100 = sum(length < 100 for length in lengths)

    total_invalid = sum(invalid_counts.values())

    results = {
        "Number of sequences": len(sequences),
        "Number of unique IDs": len(set(sequence_ids)),
        "Duplicate ID groups": duplicate_id_groups,
        "Extra duplicate ID records": duplicate_extra_records,
        "Empty input sequences": sum(
            len(sequence) == 0
            for header, sequence in sequences
        ),
        "Terminal stop-containing proteins": terminal_stop_count,
        "Internal stop-containing proteins": internal_stop_count,
        "Empty sequences after cleanup": empty_after_cleanup,
        "Proteins <50 aa": below_50,
        "Proteins <100 aa": below_100,
        "Proteins containing X": sequences_with_x,
        "Total X residues": total_x,
        "Total invalid amino-acid characters": total_invalid,
        "Proteins containing invalid amino-acid characters":
            sequences_with_invalid,
        "Invalid character breakdown": (
            "; ".join(
                f"{character}: {count}"
                for character, count
                in sorted(invalid_counts.items())
            )
            if invalid_counts else "None"
        ),
        "Total protein length (aa)": total_length,
        "Minimum protein length (aa)": min_length,
        "Maximum protein length (aa)": max_length,
        "Mean protein length (aa)": round(mean_length, 2),
        "Median protein length (aa)": median_length,
        "Proteins retained for downstream analysis":
            len(cleaned_records),
    }

    return results, cleaned_records

def save_fasta(sequences, output_file):
    try:
        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as file:

            for header, sequence in sequences:

                file.write(f">{header}\n")

                for i in range(0, len(sequence), 60):
                    file.write(
                        sequence[i:i + 60] + "\n"
                    )

        print("\nCleaned FASTA saved:")
        print(output_file)

        return output_file

    except Exception as error:
        print("\nERROR saving FASTA:")
        print(error)
        return False


def save_qc_summary(qc_results, output_file):
    try:
        qc_table = pd.DataFrame(
            list(qc_results.items()),
            columns=["QC_Measurement", "Value"]
        )

        qc_table.to_csv(
            output_file,
            sep="\t",
            index=False
        )

        print("\nQC summary saved:")
        print(output_file)

        return True

    except Exception as error:
        print("\nERROR saving QC summary:")
        print(error)
        return False


def save_sequence_statistics(sequences, output_file):
    rows = []

    for header, sequence in sequences:

        sequence_id = header.split()[0]

        rows.append({
            "Protein_ID": sequence_id,
            "Header": header,
            "Length_aa": len(sequence),
            "Contains_X":
                "Yes" if "X" in sequence else "No",
            "Contains_internal_stop":
                "Yes" if "*" in sequence else "No"
        })

    df = pd.DataFrame(rows)

    try:
        df.to_csv(
            output_file,
            sep="\t",
            index=False
        )

        print("\nPer-protein statistics saved:")
        print(output_file)

        return True

    except Exception as error:
        print("\nERROR saving per-protein statistics:")
        print(error)
        return False


def save_length_plot(sequences, output_file):
    lengths = [
        len(sequence)
        for header, sequence in sequences
    ]

    if not lengths:
        return False

    try:
        plt.figure(figsize=(10, 6))
        plt.hist(lengths, bins=50)

        plt.xlabel("Protein length (aa)")
        plt.ylabel("Number of proteins")
        plt.title(
            "Metisa plana Protein Length Distribution"
        )

        plt.tight_layout()
        plt.savefig(output_file, dpi=300)
        plt.close()

        print("\nProtein length distribution saved:")
        print(output_file)

        return True

    except Exception as error:
        print("\nERROR creating length plot:")
        print(error)
        return False


def ask_output_name(input_file):
    folder = os.path.dirname(input_file)

    while True:

        filename = input(
            "\nEnter output filename (without extension): "
        ).strip().strip('"')

        if not filename:
            print("\nERROR: Filename cannot be empty.")
            continue

        filename = os.path.splitext(filename)[0]

        return (
            os.path.join(
                folder,
                filename + ".fasta"
            ),
            os.path.join(
                folder,
                filename + "_QC.tsv"
            ),
            os.path.join(
                folder,
                filename + "_protein_statistics.tsv"
            ),
            os.path.join(
                folder,
                filename + "_length_distribution.png"
            )
        )


# 02. Homology Search using BLASTP

The quality-controlled protein sequences are searched against the selected reference protein database using BLASTP.

The purpose of this analysis is to identify homologous proteins and obtain sequence-based evidence for the possible biological identity and function of the *M. plana* proteins.

The BLASTP results contain information including:

- query protein identifier
- subject protein identifier
- percentage sequence identity
- alignment length
- query length
- subject length
- query coverage
- E-value
- bit score
- subject description

The homology results provide the primary sequence-similarity evidence used in the subsequent filtering and candidate-selection stages.


In [3]:
def find_protein_db(db_root):

    protein_files = [
        f"{db_root}{ext}"
        for ext in PROTEIN_DB_EXTS
    ]

    if all(
        os.path.isfile(path)
        for path in protein_files
    ):
        return True

    protein_volume_files = [
        f"{db_root}.00{ext}"
        for ext in PROTEIN_DB_EXTS
    ]

    if all(
        os.path.isfile(path)
        for path in protein_volume_files
    ):
        return True

    if os.path.isfile(f"{db_root}.pal"):
        return True

    return False


def validate_db_root(db_root):
    if not find_protein_db(db_root):

        print(
            "\nERROR: Protein BLAST database "
            "files were not found."
        )

        print("\nExpected database:")
        print(db_root + ".pin/.psq/.phr")

        print("\nOr a BLAST protein alias database:")
        print(db_root + ".pal")

        return False

    print("\n✔ Protein BLAST database detected.")
    return True


def find_blastp_executable():

    print("\nEnter the full path to blastp.exe")

    executable = expand_path(
        input("Path to blastp.exe: ")
    )

    if not os.path.isfile(executable):
        print("\nERROR: blastp.exe not found:")
        print(executable)
        return None

    if os.path.basename(executable).lower() != "blastp.exe":
        print("\nERROR: Selected file is not blastp.exe.")
        return None

    try:

        result = subprocess.run(
            [executable, "-version"],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:

            print(
                "\nERROR: blastp.exe was found "
                "but could not be executed."
            )

            print(result.stderr)
            return None

        print("\n✔ BLASTP executable found:")
        print(executable)

        print("\n✔ BLASTP is working.")

        version_output = (
            result.stdout.strip().splitlines()
        )

        if version_output:
            print("Version:", version_output[0])

        return executable

    except Exception as error:

        print("\nERROR testing BLASTP:")
        print(error)

        return None

def ask_blast_output(query_path):
    folder = os.path.dirname(query_path)

    filename = input(
        "\nOutput filename (without extension): "
    ).strip().strip('"')

    if not filename:
        filename = "raw_blast"

    filename = os.path.splitext(filename)[0]

    output_path = os.path.join(
        folder,
        filename + ".tsv"
    )

    print("\nOutput:")
    print(output_path)

    return output_path


def build_blast_command(
    blast_program,
    query_path,
    db_root,
    output_path,
    max_target_seqs,
    threads
):
    return [
        blast_program,
        "-query", query_path,
        "-db", db_root,
        "-out", output_path,
        "-outfmt", ANNOTATION_OUTFMT,
        "-max_target_seqs", max_target_seqs,
        "-num_threads", threads
    ]


def determine_uniprot_source(sseqid):

    if not sseqid:
        return "Unknown"

    sseqid = sseqid.strip()
    parts = sseqid.split("|")

    if len(parts) < 2:
        return "Unknown"

    database_type = parts[0].lower()
    accession = parts[1]

    if re.search(r"-\d+$", accession):
        return "Isoform"

    if database_type == "sp":
        return "Swiss-Prot"

    if database_type == "tr":
        return "TrEMBL"

    return "Unknown"


def add_header_and_source(output_path):

    header = (
        "qseqid\tsseqid\tpident\tlength\tqlen\tslen\t"
        "qcovs\tevalue\tbitscore\tstitle\tsource\n"
    )

    try:

        with open(
            output_path,
            "r",
            encoding="utf-8"
        ) as file:
            lines = file.readlines()

        new_lines = []

        for line in lines:

            line = line.rstrip("\n")

            if not line.strip():
                continue

            fields = line.split("\t")

            if len(fields) < 10:
                continue

            fields.append(
                determine_uniprot_source(fields[1])
            )

            new_lines.append(
                "\t".join(fields) + "\n"
            )

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as file:

            file.write(header)
            file.writelines(new_lines)

        return True

    except Exception as error:

        print("\nERROR adding output header:")
        print(error)

        return False


def run_blast(command):

    print("\nBLASTP running...")
    print(
        subprocess.list2cmdline(command)
    )

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        if result.returncode == 0:

            print("\n✔ BLASTP completed.")
            return True

        print("\nBLASTP failed.")

        if result.stderr:
            print(result.stderr)

        return False

    except Exception as error:

        print("\nERROR running BLASTP:")
        print(error)

        return False


def print_summary(
    query,
    database,
    output,
    status
):
    print("""
===============================================================
                         SUMMARY
===============================================================
""")

    print("BLAST    : blastp")
    print("QUERY    :", query)
    print("DATABASE :", database)
    print("OUTPUT   :", output)
    print("STATUS   :", status)

banner()

section("STEP 1: INPUT FASTA")

RAW_PROTEIN_FASTA = expand_path(
    input("\nEnter path to raw protein FASTA: ")
)

if not check_file_exists(RAW_PROTEIN_FASTA):
    raise FileNotFoundError(RAW_PROTEIN_FASTA)

print("\nRaw protein FASTA:")
print(RAW_PROTEIN_FASTA)

section("STEP 2: READING FASTA")

sequences = read_fasta(RAW_PROTEIN_FASTA)

if sequences is None or not sequences:
    raise ValueError("FASTA contains no sequences.")

print(
    "\nSequences loaded:",
    f"{len(sequences):,}"
)

section("STEP 3: PROTEIN QC + PREPARATION")

result = calculate_protein_qc_and_prepare(
    sequences
)

if result is None:
    raise ValueError(
        "No non-empty protein sequences remain."
    )

qc_results, cleaned_sequences = result

section("STEP 4: QC RESULTS")

for measurement, value in qc_results.items():
    print(f"{measurement}: {value}")

if qc_results[
    "Internal stop-containing proteins"
] > 0:

    print(
        "\nWARNING: Internal stop-containing "
        "proteins were detected."
    )

    print(
        "Review these proteins before "
        "downstream BLASTP analysis."
    )

section("STEP 5: QC OUTPUT")

(
    CLEANED_FASTA,
    QC_SUMMARY,
    SEQUENCE_STATISTICS,
    LENGTH_PLOT
) = ask_output_name(RAW_PROTEIN_FASTA)


section("STEP 6: SAVING CLEANED FASTA")

if not save_fasta(
    cleaned_sequences,
    CLEANED_FASTA
):
    raise IOError(
        "Failed to save cleaned FASTA."
    )

section("STEP 7: SAVING QC SUMMARY")

save_qc_summary(
    qc_results,
    QC_SUMMARY
)

section("STEP 8: SAVING PROTEIN STATISTICS")

save_sequence_statistics(
    cleaned_sequences,
    SEQUENCE_STATISTICS
)

section("STEP 9: SAVING LENGTH DISTRIBUTION")

save_length_plot(
    cleaned_sequences,
    LENGTH_PLOT
)

section("INPUT QC + FASTA PREPARATION COMPLETE")

print(
    "\nOriginal proteins:",
    f"{len(sequences):,}"
)

print(
    "Prepared proteins:",
    f"{len(cleaned_sequences):,}"
)

print(
    "Terminal '*' removed:",
    qc_results[
        "Terminal stop-containing proteins"
    ]
)

print(
    "Internal '*' detected:",
    qc_results[
        "Internal stop-containing proteins"
    ]
)

print(
    "Proteins <50 aa:",
    qc_results["Proteins <50 aa"]
)

print(
    "Proteins containing X:",
    qc_results["Proteins containing X"]
)

section("STEP 10: BLASTP INPUT")

if (
    CLEANED_FASTA is None
    or not check_file_exists(CLEANED_FASTA)
):
    raise FileNotFoundError(
        "Cleaned FASTA from QC is unavailable."
    )

print("\nUsing cleaned FASTA from QC:")
print(CLEANED_FASTA)

BLAST_DATABASE = expand_path(
    input("\nUniProt database path: ")
)

if not validate_db_root(BLAST_DATABASE):
    raise FileNotFoundError(
        BLAST_DATABASE
    )

BLAST_PROGRAM = find_blastp_executable()

if BLAST_PROGRAM is None:
    raise FileNotFoundError(
        "blastp.exe not found."
    )

section("STEP 11: BLAST PARAMETERS")

while True:

    max_target_seqs = input(
        "Maximum target sequences per query [10]: "
    ).strip()

    if not max_target_seqs:

        max_target_seqs = (
            DEFAULT_MAX_TARGET_SEQS
        )
        break

    if (
        max_target_seqs.isdigit()
        and int(max_target_seqs) > 0
    ):
        break

    print(
        "Please enter a positive whole number."
    )


while True:

    threads = input(
        "CPU threads [6]: "
    ).strip()

    if not threads:

        threads = DEFAULT_THREADS
        break

    if (
        threads.isdigit()
        and int(threads) > 0
    ):
        break

    print(
        "Please enter a positive whole number."
    )


section("STEP 12: BLAST OUTPUT")

RAW_BLAST = ask_blast_output(
    CLEANED_FASTA
)

section("STEP 13: BUILDING BLASTP COMMAND")

command = build_blast_command(
    BLAST_PROGRAM,
    CLEANED_FASTA,
    BLAST_DATABASE,
    RAW_BLAST,
    max_target_seqs,
    threads
)

section("STEP 14: RUNNING BLASTP")

success = run_blast(command)

if not success:

    print_summary(
        CLEANED_FASTA,
        BLAST_DATABASE,
        RAW_BLAST,
        "FAILED"
    )

    raise RuntimeError(
        "BLASTP failed."
    )

section("STEP 15: PROCESSING BLAST RESULTS")

if not add_header_and_source(RAW_BLAST):

    print_summary(
        CLEANED_FASTA,
        BLAST_DATABASE,
        RAW_BLAST,
        "FAILED DURING RESULT PROCESSING"
    )

    raise RuntimeError(
        "Failed to process BLAST results."
    )

print_summary(
    CLEANED_FASTA,
    BLAST_DATABASE,
    RAW_BLAST,
    "SUCCESS"
)

print(
    "\nBLASTP result is ready for downstream "
    "filtering and top-hit ranking."
)

print("\nPipeline variables saved:")
print("  RAW_PROTEIN_FASTA =", RAW_PROTEIN_FASTA)
print("  CLEANED_FASTA =", CLEANED_FASTA)
print("  QC_SUMMARY =", QC_SUMMARY)
print("  SEQUENCE_STATISTICS =", SEQUENCE_STATISTICS)
print("  LENGTH_PLOT =", LENGTH_PLOT)
print("  BLAST_DATABASE =", BLAST_DATABASE)
print("  BLAST_PROGRAM =", BLAST_PROGRAM)
print("  RAW_BLAST =", RAW_BLAST)


          METISA PLANA — QC + HOMOLOGY SEARCH


STEP 1: INPUT FASTA

Raw protein FASTA:
C:\metp\output\testing.fasta

STEP 2: READING FASTA

Sequences loaded: 100

STEP 3: PROTEIN QC + PREPARATION

STEP 4: QC RESULTS
Number of sequences: 100
Number of unique IDs: 100
Duplicate ID groups: 0
Extra duplicate ID records: 0
Empty input sequences: 0
Terminal stop-containing proteins: 93
Internal stop-containing proteins: 0
Empty sequences after cleanup: 0
Proteins <50 aa: 0
Proteins <100 aa: 22
Proteins containing X: 10
Total X residues: 11
Total invalid amino-acid characters: 0
Proteins containing invalid amino-acid characters: 0
Invalid character breakdown: None
Total protein length (aa): 32350
Minimum protein length (aa): 55
Maximum protein length (aa): 1699
Mean protein length (aa): 323.5
Median protein length (aa): 211.5
Proteins retained for downstream analysis: 100

STEP 5: QC OUTPUT

STEP 6: SAVING CLEANED FASTA

Cleaned FASTA saved:
C:\metp\output\testing_cleaned.fasta

STEP 7: SAV


# 03. BLAST Hit Filtering & Taxonomic Screening

The BLASTP results are subsequently evaluated using predefined similarity and alignment-quality thresholds.

Hits with insufficient sequence identity, query coverage or statistical significance are excluded from the qualifying dataset.

The qualifying hits are then examined according to their taxonomic information.

This screening step helps distinguish biologically relevant arthropod homologues from potentially unrelated or contaminating sequences, including bacterial, fungal and other non-target organisms.

The resulting filtered dataset provides a higher-confidence set of proteins for functional annotation and candidate mining.


In [4]:
# ===============================================================
# STEP 16: BLAST TAXONOMY SCREENING
# ===============================================================

MIN_IDENTITY = 30.0
MIN_COVERAGE = 70.0
MAX_EVALUE = 0.05
DOMINANCE_THRESHOLD = 0.70

CATEGORIES = {
    1: "KEEP_ARTHROPOD",
    2: "POSSIBLE_BACTERIAL",
    3: "POSSIBLE_FUNGAL",
    4: "NON_ARTHROPOD_EUKARYOTE",
    5: "OTHER_PROKARYOTE",
    6: "AMBIGUOUS",
    7: "NO_QUALIFYING_HIT",
    8: "UNKNOWN_TAXONOMY",
}


def read_blast_file(path):
    print("\nReading RAW BLAST TSV...")
    rows = []

    with open(path, "r", encoding="utf-8", errors="replace") as file:
        reader = csv.reader(file, delimiter="\t")

        for line_number, row in enumerate(reader, start=1):
            if not row:
                continue

            if row[0].strip().lower() == "qseqid":
                continue

            if len(row) < 10:
                print(
                    f"WARNING: Skipping line {line_number}: "
                    f"only {len(row)} columns."
                )
                continue

            try:
                rows.append({
                    "qseqid": row[0].strip(),
                    "sseqid": row[1].strip(),
                    "pident": float(row[2]),
                    "length": int(float(row[3])),
                    "qlen": int(float(row[4])),
                    "slen": int(float(row[5])),
                    "qcov": float(row[6]),
                    "evalue": float(row[7]),
                    "bitscore": float(row[8]),
                    "stitle": row[9].strip(),
                })
            except ValueError:
                print(
                    f"WARNING: Could not parse line {line_number}."
                )

    print(f"Raw BLAST rows read: {len(rows):,}")
    return rows


def passes_quality_filter(hit):
    return (
        hit["pident"] >= MIN_IDENTITY
        and hit["qcov"] >= MIN_COVERAGE
        and hit["evalue"] <= MAX_EVALUE
    )


def apply_quality_filter(rows):
    print("\nApplying BLAST quality thresholds...")

    qualifying = []
    failed = []

    for hit in rows:
        (qualifying if passes_quality_filter(hit) else failed).append(hit)

    print(f"  Total BLAST hits:       {len(rows):,}")
    print(f"  Passing all thresholds: {len(qualifying):,}")
    print(f"  Failed thresholds:      {len(failed):,}")

    print("\nThresholds used:")
    print(f"  Identity >= {MIN_IDENTITY}%")
    print(f"  Coverage >= {MIN_COVERAGE}%")
    print(f"  E-value  <= {MAX_EVALUE}")

    return qualifying, failed


def extract_taxid(stitle):
    if not stitle:
        return None

    match = re.search(r"\bOX=(\d+)", stitle)

    return int(match.group(1)) if match else None


def extract_species(stitle):
    if not stitle:
        return "Unknown"

    match = re.search(
        r"\bOS=([^=]+?)(?=\s+(?:OX|GN|PE|SV|CC|KW|GO)=|$)",
        stitle
    )

    return match.group(1).strip() if match else "Unknown"


def parse_ranked_lineage_line(line):
    line = line.strip()

    if not line:
        return None

    parts = [x.strip() for x in line.split("|")]

    if len(parts) < 10:
        return None

    try:
        taxid = int(parts[0])
    except ValueError:
        return None

    return {
        "taxid": taxid,
        "scientific_name": parts[1],
        "species": parts[2],
        "genus": parts[3],
        "family": parts[4],
        "order": parts[5],
        "class": parts[6],
        "phylum": parts[7],
        "kingdom": parts[8],
        "superkingdom": parts[9],
    }


def load_required_taxonomy(rankedlineage_path, required_taxids):
    print("\nLoading taxonomy ONLY for qualifying BLAST hits...")

    required_taxids = set(required_taxids)
    taxonomy = {}

    if not required_taxids:
        print("No qualifying TaxIDs to load.")
        return taxonomy

    print(
        f"Qualifying TaxIDs required: "
        f"{len(required_taxids):,}"
    )

    with open(
        rankedlineage_path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as file:

        for line in file:
            record = parse_ranked_lineage_line(line)

            if record is None:
                continue

            taxid = record["taxid"]

            if taxid in required_taxids:
                taxonomy[taxid] = record

                if len(taxonomy) == len(required_taxids):
                    break

    print(
        f"TaxIDs successfully found: "
        f"{len(taxonomy):,}/{len(required_taxids):,}"
    )

    return taxonomy


def classify_taxonomy(tax_record):
    if tax_record is None:
        return "UNKNOWN_TAXONOMY"

    superkingdom = tax_record["superkingdom"].strip().lower()
    kingdom = tax_record["kingdom"].strip().lower()
    phylum = tax_record["phylum"].strip().lower()

    if superkingdom == "bacteria":
        return "BACTERIAL"

    if superkingdom == "archaea":
        return "OTHER_PROKARYOTE"

    if kingdom == "fungi":
        return "FUNGAL"

    if phylum == "arthropoda":
        return "ARTHROPOD"

    if superkingdom == "eukaryota":
        return "NON_ARTHROPOD_EUKARYOTE"

    return "OTHER"


def group_hits_by_query(rows):
    grouped = defaultdict(list)

    for row in rows:
        grouped[row["qseqid"]].append(row)

    return grouped


def get_qualifying_hits(hits):
    qualifying = [
        hit for hit in hits
        if passes_quality_filter(hit)
    ]

    qualifying.sort(
        key=lambda x: (
            x["evalue"],
            -x["bitscore"],
            -x["pident"],
            -x["qcov"],
        )
    )

    return qualifying


def calculate_taxonomic_support(qualifying_hits, taxonomy):
    counts = Counter()
    classified_hits = []

    for hit in qualifying_hits:
        taxid = extract_taxid(hit["stitle"])
        tax_record = taxonomy.get(taxid) if taxid is not None else None
        tax_class = classify_taxonomy(tax_record)

        counts[tax_class] += 1

        classified_hits.append({
            "hit": hit,
            "taxid": taxid,
            "taxonomy": tax_record,
            "tax_class": tax_class,
        })

    return counts, classified_hits


def make_decision(qualifying_hits, counts):
    total = len(qualifying_hits)

    if total == 0:
        return (
            "NO_QUALIFYING_HIT",
            "No BLAST hit passed all three quality thresholds."
        )

    group_counts = {
        "KEEP_ARTHROPOD": counts.get("ARTHROPOD", 0),
        "POSSIBLE_BACTERIAL": counts.get("BACTERIAL", 0),
        "POSSIBLE_FUNGAL": counts.get("FUNGAL", 0),
        "NON_ARTHROPOD_EUKARYOTE":
            counts.get("NON_ARTHROPOD_EUKARYOTE", 0),
        "OTHER_PROKARYOTE":
            counts.get("OTHER_PROKARYOTE", 0),
    }

    highest_count = max(group_counts.values())

    if highest_count == 0:
        return (
            "UNKNOWN_TAXONOMY",
            "All qualifying BLAST hits had unresolved taxonomy."
        )

    highest_groups = [
        category
        for category, count in group_counts.items()
        if count == highest_count
    ]

    if len(highest_groups) > 1:
        tied_groups = ", ".join(highest_groups)

        return (
            "AMBIGUOUS",
            f"Taxonomic evidence is tied between "
            f"{tied_groups} ({highest_count} qualifying "
            f"hit(s) each)."
        )

    winner = highest_groups[0]
    winner_fraction = highest_count / total

    if winner_fraction >= DOMINANCE_THRESHOLD:

        reasons = {
            "KEEP_ARTHROPOD": "Arthropod",
            "POSSIBLE_BACTERIAL": "Bacteria",
            "POSSIBLE_FUNGAL": "Fungi",
            "NON_ARTHROPOD_EUKARYOTE":
                "Non-arthropod eukaryotes",
            "OTHER_PROKARYOTE":
                "Other prokaryotes",
        }

        return (
            winner,
            f"{reasons[winner]} is the dominant taxonomic "
            f"group ({highest_count}/{total} = "
            f"{winner_fraction:.1%} of all qualifying hits)."
        )

    return (
        "AMBIGUOUS",
        f"{winner} has the highest support "
        f"({highest_count}/{total} = "
        f"{winner_fraction:.1%}), but does not reach "
        f"the {DOMINANCE_THRESHOLD:.0%} dominance threshold."
    )


def screen_proteins(raw_rows, taxonomy):
    print("\nGrouping RAW BLAST hits by protein...")

    grouped = group_hits_by_query(raw_rows)

    print(f"Unique proteins: {len(grouped):,}")
    print("\nScreening proteins...")

    results = []

    for query_id, hits in grouped.items():

        total_blast_hits = len(hits)
        qualifying_hits = get_qualifying_hits(hits)
        total = len(qualifying_hits)

        if not qualifying_hits:
            results.append({
                "gene_id": query_id,
                "best_hit_accession": "",
                "best_hit_species": "",
                "best_hit_taxid": "",
                "best_hit_lineage": "",
                "best_identity_pct": "",
                "best_query_coverage_pct": "",
                "best_evalue": "",
                "best_bitscore": "",
                "qualifying_hits": 0,
                "total_blast_hits": total_blast_hits,
                "arthropod_hits": 0,
                "bacterial_hits": 0,
                "fungal_hits": 0,
                "non_arthropod_eukaryote_hits": 0,
                "other_prokaryote_hits": 0,
                "unknown_taxonomy_hits": 0,
                "known_taxonomy_hits": 0,
                "arthropod_fraction": 0,
                "bacterial_fraction": 0,
                "fungal_fraction": 0,
                "non_arthropod_eukaryote_fraction": 0,
                "other_prokaryote_fraction": 0,
                "classification": "NO_QUALIFYING_HIT",
                "decision": "NO_QUALIFYING_HIT",
                "reason":
                    "No BLAST hit passed all three quality thresholds.",
                "best_hit_description": "",
            })
            continue

        counts, classified_hits = calculate_taxonomic_support(
            qualifying_hits,
            taxonomy
        )

        category, reason = make_decision(
            qualifying_hits,
            counts
        )

        best = qualifying_hits[0]
        best_taxid = extract_taxid(best["stitle"])
        best_tax = taxonomy.get(best_taxid)

        if best_tax:
            best_lineage = (
                best_tax["superkingdom"] + "; "
                + best_tax["kingdom"] + "; "
                + best_tax["phylum"] + "; "
                + best_tax["class"] + "; "
                + best_tax["order"]
            )
        else:
            best_lineage = ""

        unknown = counts.get("UNKNOWN_TAXONOMY", 0)

        results.append({
            "gene_id": query_id,
            "best_hit_accession": best["sseqid"],
            "best_hit_species": extract_species(best["stitle"]),
            "best_hit_taxid": best_taxid,
            "best_hit_lineage": best_lineage,
            "best_identity_pct": best["pident"],
            "best_query_coverage_pct": best["qcov"],
            "best_evalue": best["evalue"],
            "best_bitscore": best["bitscore"],
            "qualifying_hits": total,
            "total_blast_hits": total_blast_hits,
            "arthropod_hits": counts.get("ARTHROPOD", 0),
            "bacterial_hits": counts.get("BACTERIAL", 0),
            "fungal_hits": counts.get("FUNGAL", 0),
            "non_arthropod_eukaryote_hits":
                counts.get("NON_ARTHROPOD_EUKARYOTE", 0),
            "other_prokaryote_hits":
                counts.get("OTHER_PROKARYOTE", 0),
            "unknown_taxonomy_hits": unknown,
            "known_taxonomy_hits": total - unknown,
            "arthropod_fraction":
                counts.get("ARTHROPOD", 0) / total,
            "bacterial_fraction":
                counts.get("BACTERIAL", 0) / total,
            "fungal_fraction":
                counts.get("FUNGAL", 0) / total,
            "non_arthropod_eukaryote_fraction":
                counts.get("NON_ARTHROPOD_EUKARYOTE", 0) / total,
            "other_prokaryote_fraction":
                counts.get("OTHER_PROKARYOTE", 0) / total,
            "classification": category,
            "decision": category,
            "reason": reason,
            "best_hit_description": best["stitle"],
        })

    return results


def save_tsv(rows, output_path):
    if not rows:
        print(f"\nNo rows to save: {output_path}")
        return

    fields = list(rows[0].keys())

    with open(
        output_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=fields,
            delimiter="\t"
        )

        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved TSV: {output_path}")


def save_excel(results, output_path):
    from openpyxl import Workbook
    from openpyxl.styles import Font

    if not results:
        print("No results to write to Excel.")
        return

    workbook = Workbook()
    sheet = workbook.active
    sheet.title = "All Results"

    fields = list(results[0].keys())

    for col, field in enumerate(fields, start=1):
        cell = sheet.cell(
            row=1,
            column=col,
            value=field
        )
        cell.font = Font(bold=True)

    for row_number, record in enumerate(results, start=2):
        for col, field in enumerate(fields, start=1):
            sheet.cell(
                row=row_number,
                column=col,
                value=record[field]
            )

    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions

    categories = sorted(
        set(row["classification"] for row in results)
    )

    for category in categories:

        ws = workbook.create_sheet(
            title=category[:31]
        )

        category_rows = [
            row for row in results
            if row["classification"] == category
        ]

        for col, field in enumerate(fields, start=1):
            cell = ws.cell(
                row=1,
                column=col,
                value=field
            )
            cell.font = Font(bold=True)

        for row_number, record in enumerate(
            category_rows,
            start=2
        ):
            for col, field in enumerate(fields, start=1):
                ws.cell(
                    row=row_number,
                    column=col,
                    value=record[field]
                )

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

    workbook.save(output_path)

    print(f"Saved Excel: {output_path}")


def read_fasta_dict(path):
    sequences = {}
    current_id = None
    sequence_parts = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as file:

        for line in file:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):

                if current_id is not None:
                    sequences[current_id] = "".join(
                        sequence_parts
                    )

                current_id = line[1:].split()[0]
                sequence_parts = []

            else:
                sequence_parts.append(line)

    if current_id is not None:
        sequences[current_id] = "".join(sequence_parts)

    return sequences


def write_filtered_fasta(
    sequence_ids,
    sequences,
    output_path
):
    written = 0
    missing = []

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:

        for sequence_id in sequence_ids:

            if sequence_id not in sequences:
                missing.append(sequence_id)
                continue

            file.write(f">{sequence_id}\n")

            sequence = sequences[sequence_id]

            for i in range(0, len(sequence), 80):
                file.write(
                    sequence[i:i + 80] + "\n"
                )

            written += 1

    print(f"\nFASTA sequences written: {written:,}")

    if missing:
        print(
            f"WARNING: {len(missing):,} IDs "
            "were not found in cleaned FASTA."
        )


def ask_categories():
    print("\n" + "=" * 70)
    print("FASTA CATEGORY SELECTION")
    print("=" * 70)

    for number, category in CATEGORIES.items():
        print(f"{number} = {category}")

    print("\nSelect one or multiple categories.")
    print("Example: 1")
    print("Example: 1,4,6")

    while True:
        answer = input(
            "\nEnter category numbers: "
        ).strip()

        if not answer:
            print("Please enter at least one category.")
            continue

        try:
            numbers = [
                int(x.strip())
                for x in answer.split(",")
                if x.strip()
            ]
        except ValueError:
            print(
                "Invalid input. Use numbers such as 1,4,6."
            )
            continue

        invalid = [
            n for n in numbers
            if n not in CATEGORIES
        ]

        if invalid:
            print(
                "Invalid category number(s): "
                + ", ".join(str(x) for x in invalid)
            )
            continue

        numbers = list(dict.fromkeys(numbers))

        return [
            CATEGORIES[n]
            for n in numbers
        ]


def export_selected_fasta(
    results,
    fasta_sequences,
    categories,
    output_dir,
    output_prefix
):
    selected_rows = [
        row for row in results
        if row["classification"] in categories
    ]

    selected_ids = [
        row["gene_id"]
        for row in selected_rows
    ]

    if not selected_ids:
        print(
            "\nNo sequences belong to "
            "the selected categories."
        )
        return

    output_path = os.path.join(
        output_dir,
        f"{output_prefix}_filtered.fasta"
    )

    write_filtered_fasta(
        selected_ids,
        fasta_sequences,
        output_path
    )

    print("\nSelected categories:")

    for category in categories:
        count = sum(
            1
            for row in results
            if row["classification"] == category
        )
        print(f"  {category}: {count:,}")

    print("\nFASTA saved:")
    print(output_path)

    return output_path

def print_screening_summary(results):
    counts = Counter(
        row["classification"]
        for row in results
    )

    print("\n" + "=" * 70)
    print("CONTAMINATION SCREEN SUMMARY")
    print("=" * 70)

    print(
        f"Proteins screened: {len(results):,}"
    )

    for category in CATEGORIES.values():

        count = counts.get(category, 0)

        percentage = (
            count / len(results) * 100
            if results else 0
        )

        print(
            f"{category:30s} "
            f"{count:8,} "
            f"({percentage:6.2f}%)"
        )

section("STEP 16: TAXONOMY SCREENING INPUT")

if (
    RAW_BLAST is None
    or not check_file_exists(RAW_BLAST)
):
    raise FileNotFoundError(
        "Previous RAW_BLAST output is unavailable."
    )

if (
    CLEANED_FASTA is None
    or not check_file_exists(CLEANED_FASTA)
):
    raise FileNotFoundError(
        "Previous CLEANED_FASTA output is unavailable."
    )

print("\nUsing previous RAW BLAST:")
print(RAW_BLAST)

print("\nUsing previous cleaned FASTA:")
print(CLEANED_FASTA)

section("STEP 17: RANKED LINEAGE INPUT")

rankedlineage_file = expand_path(
    input("\nPath to rankedlineage.dmp: ")
)

if not check_file_exists(rankedlineage_file):
    raise FileNotFoundError(rankedlineage_file)

section("STEP 18: SCREENING OUTPUT")

blast_dir = os.path.dirname(RAW_BLAST)

output_prefix = (
    os.path.splitext(
        os.path.basename(RAW_BLAST)
    )[0]
    + "_screening"
)

output_dir = os.path.join(
    blast_dir,
    output_prefix
)

os.makedirs(
    output_dir,
    exist_ok=True
)

print("\nOutput folder:")
print(output_dir)

section("STEP 19: READING BLAST RESULTS")

raw_rows = read_blast_file(RAW_BLAST)

if not raw_rows:
    raise ValueError(
        "No BLAST rows found."
    )

raw_grouped = group_hits_by_query(raw_rows)

print("\nActual BLAST hit distribution:")

hit_counts = Counter(
    len(hits)
    for hits in raw_grouped.values()
)

for number_of_hits in sorted(hit_counts):
    protein_count = hit_counts[number_of_hits]
    print(
        f"  {number_of_hits:>4} hit(s): "
        f"{protein_count:,} protein(s)"
    )

section("STEP 20: BLAST QUALITY FILTER")

qualifying_rows, failed_rows = (
    apply_quality_filter(raw_rows)
)

qualifying_hits_path = os.path.join(
    output_dir,
    f"{output_prefix}_qualifying_hits.tsv"
)

save_tsv(
    qualifying_rows,
    qualifying_hits_path
)

section("STEP 21: TAXONOMY SCREENING")

print(
    "\nExtracting TaxIDs ONLY from "
    "quality-passing hits..."
)

required_taxids = set()

for row in qualifying_rows:
    taxid = extract_taxid(row["stitle"])

    if taxid is not None:
        required_taxids.add(taxid)

print(
    f"Unique qualifying TaxIDs: "
    f"{len(required_taxids):,}"
)

taxonomy = load_required_taxonomy(
    rankedlineage_file,
    required_taxids
)

results = screen_proteins(
    raw_rows,
    taxonomy
)

screening_tsv = os.path.join(
    output_dir,
    f"{output_prefix}_screening_all.tsv"
)

save_tsv(
    results,
    screening_tsv
)

screening_excel = os.path.join(
    output_dir,
    f"{output_prefix}_screening.xlsx"
)

save_excel(
    results,
    screening_excel
)

print_screening_summary(results)

section("STEP 22: FILTERED FASTA")

print("\nReading cleaned FASTA...")

fasta_sequences = read_fasta_dict(
    CLEANED_FASTA
)

print(
    f"FASTA sequences loaded: "
    f"{len(fasta_sequences):,}"
)

categories = ask_categories()

FILTERED_FASTA = export_selected_fasta(
    results,
    fasta_sequences,
    categories,
    output_dir,
    output_prefix
)

section("STEP 23: SCREENING COMPLETE")

print("\nInput files:")
print("  RAW BLAST       :", RAW_BLAST)
print("  CLEANED FASTA   :", CLEANED_FASTA)
print("  RANKED LINEAGE  :", rankedlineage_file)

print("\nOutput folder:")
print(output_dir)

print("\nMain outputs:")
print("  Qualifying BLAST :", qualifying_hits_path)
print("  Screening TSV    :", screening_tsv)
print("  Screening Excel  :", screening_excel)
print(
    "  Filtered FASTA   :",
    os.path.join(
        output_dir,
        f"{output_prefix}_filtered.fasta"
    )
)

print("\nFiltering rules:")
print(f"  Identity >= {MIN_IDENTITY}%")
print(f"  Coverage >= {MIN_COVERAGE}%")
print(f"  E-value <= {MAX_EVALUE}")
print(
    f"  Taxonomic dominance >= "
    f"{DOMINANCE_THRESHOLD:.0%}"
)

print(
    "\nOnly BLAST hits passing all three "
    "quality thresholds were used for taxonomy."
)

print(
    "The selected categories determine which "
    "proteins are included in the filtered FASTA."
)


STEP 16: TAXONOMY SCREENING INPUT

Using previous RAW BLAST:
C:\metp\output\testing_raw_blast.tsv

Using previous cleaned FASTA:
C:\metp\output\testing_cleaned.fasta

STEP 17: RANKED LINEAGE INPUT

STEP 18: SCREENING OUTPUT

Output folder:
C:\metp\output\testing_raw_blast_screening

STEP 19: READING BLAST RESULTS

Reading RAW BLAST TSV...
Raw BLAST rows read: 1,156

Actual BLAST hit distribution:
     1 hit(s): 1 protein(s)
     3 hit(s): 4 protein(s)
     4 hit(s): 1 protein(s)
     5 hit(s): 3 protein(s)
     6 hit(s): 2 protein(s)
     7 hit(s): 1 protein(s)
     8 hit(s): 1 protein(s)
     9 hit(s): 2 protein(s)
    10 hit(s): 66 protein(s)
    11 hit(s): 2 protein(s)
    12 hit(s): 2 protein(s)
    13 hit(s): 1 protein(s)
    14 hit(s): 1 protein(s)
    16 hit(s): 2 protein(s)
    17 hit(s): 2 protein(s)
    18 hit(s): 2 protein(s)
    19 hit(s): 2 protein(s)
    24 hit(s): 1 protein(s)
    37 hit(s): 1 protein(s)
    42 hit(s): 1 protein(s)
   103 hit(s): 1 protein(s)

STEP 20: 


# 04. InterProScan Functional Annotation

The filtered protein sequences are analysed using InterProScan to obtain complementary functional evidence.

Whereas BLASTP identifies sequence similarity to known proteins, InterProScan identifies conserved protein domains and functional signatures.

The analysis can provide information including:

- InterPro entries
- Pfam domains
- protein families
- conserved functional regions
- Gene Ontology terms
- additional functional signatures

This functional annotation is integrated with the homology and taxonomic evidence to improve interpretation and prioritisation of candidate proteins.


In [5]:


def count_fasta_sequences(input_file):
    count = 0

    try:
        with open(
            input_file,
            "r",
            encoding="utf-8"
        ) as file:

            for line in file:
                if line.strip().startswith(">"):
                    count += 1

        return count

    except Exception as error:
        print("\nERROR reading FASTA:")
        print(error)
        return 0


def windows_to_wsl_path(windows_path):
    windows_path = os.path.abspath(windows_path)

    drive = windows_path[0].lower()
    remaining = windows_path[2:]
    remaining = remaining.replace("\\", "/")

    return "/mnt/" + drive + remaining


def check_interproscan(interproscan_dir):
    interproscan_path = (
        interproscan_dir.rstrip("/") +
        "/interproscan.sh"
    )

    result = subprocess.run(
        [
            "wsl",
            "test",
            "-f",
            interproscan_path
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:

        print("\nERROR: InterProScan not found.")
        print("Checked:")
        print(interproscan_path)

        return False

    print("\n✔ InterProScan found.")

    return True


def ask_interpro_output(input_file):
    folder = os.path.dirname(input_file)

    filename = input(
        "\nOutput filename (without extension): "
    ).strip()

    if filename == "":
        filename = "interproscan_result"

    filename = os.path.splitext(filename)[0]

    output_file = os.path.join(
        folder,
        filename + ".tsv"
    )

    print("\nOutput location:")
    print(output_file)

    return output_file

def run_interproscan(
    input_file,
    output_file,
    interproscan_dir,
    sequence_count
):
    input_wsl = windows_to_wsl_path(input_file)
    output_wsl = windows_to_wsl_path(output_file)

    interproscan_path = (
        interproscan_dir.rstrip("/") +
        "/interproscan.sh"
    )

    print("\nInput FASTA:")
    print(input_file)

    print("\nNumber of sequences:")
    print(sequence_count)

    print("\nWSL input path:")
    print(input_wsl)

    print("\nWSL output path:")
    print(output_wsl)

    print("\nInterProScan:")
    print(interproscan_path)

    command = [
        "wsl",
        "bash",
        interproscan_path,
        "-i", input_wsl,
        "-f", "tsv",
        "-o", output_wsl,
        "-goterms",
        "-iprlookup",
        "-pa"
    ]

    print("\nInterProScan running...")
    print(f"Processing {sequence_count:,} protein sequences.")
    print("This may take some time.")

    print("\nCommand:")
    print(" ".join(command))

    print("\nStarting InterProScan...\n")

    env = os.environ.copy()
    env["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    env["PATH"] = env["JAVA_HOME"] + "/bin:" + env["PATH"]

    try:
        result = subprocess.run(
            command,
            text=True,
            env=env
        )

        if result.returncode == 0:
            print("\n✔ InterProScan completed successfully.")
            return True

        print("\nERROR: InterProScan failed.")
        print("Return code:", result.returncode)
        return False

    except Exception as error:
        print("\nERROR:")
        print(error)
        return False

def add_interproscan_header(output_path):

    header = (
        "Protein Accession\t"
        "Sequence MD5 digest\t"
        "Sequence length\t"
        "Analysis\t"
        "Signature accession\t"
        "Signature description\t"
        "Start location\t"
        "Stop location\t"
        "Score\t"
        "Status\t"
        "Date\t"
        "InterPro accession\t"
        "InterPro description\t"
        "GO terms\t"
        "Pathways\n"
    )

    try:
        with open(output_path, "r", encoding="utf-8") as file:
            content = file.read()

        with open(output_path, "w", encoding="utf-8") as file:
            file.write(header)
            file.write(content)

        print("\n✔ InterProScan column header added.")
        return True

    except Exception as error:
        print("\nERROR adding column header:")
        print(error)
        return False

section("STEP 24: INTERPROSCAN INPUT")

if (
    FILTERED_FASTA is None
    or not check_file_exists(FILTERED_FASTA)
):
    raise FileNotFoundError(
        "Filtered FASTA from previous screening step "
        "is unavailable."
    )

print("\nUsing filtered FASTA from previous step:")
print(FILTERED_FASTA)

sequence_count = count_fasta_sequences(
    FILTERED_FASTA
)

if sequence_count == 0:
    raise ValueError(
        "No sequences found in filtered FASTA."
    )

print(
    f"\n✔ Filtered FASTA detected."
)

print(
    f"✔ Sequences found: {sequence_count:,}"
)

section("STEP 25: INTERPROSCAN DIRECTORY")

interproscan_dir = input(
    "\nInterProScan directory path: "
).strip()

if not check_interproscan(interproscan_dir):
    raise FileNotFoundError(
        "InterProScan was not found."
    )

section("STEP 26: INTERPROSCAN OUTPUT")

INTERPROSCAN_RESULT = ask_interpro_output(
    FILTERED_FASTA
)

section("STEP 27: RUNNING INTERPROSCAN")

success = run_interproscan(
    FILTERED_FASTA,
    INTERPROSCAN_RESULT,
    interproscan_dir,
    sequence_count
)

section("STEP 28: INTERPROSCAN RESULT")

if success:

    if add_interproscan_header(
        INTERPROSCAN_RESULT
    ):

        print("\nSTATUS: SUCCESS")

        print(
            "\nInterProScan result saved:"
        )

        print(
            INTERPROSCAN_RESULT
        )

    else:

        print(
            "\nSTATUS: COMPLETED, "
            "BUT HEADER ADDITION FAILED"
        )

else:

    print("\nSTATUS: INTERPROSCAN FAILED")


STEP 24: INTERPROSCAN INPUT

Using filtered FASTA from previous step:
C:\metp\output\testing_raw_blast_screening\testing_raw_blast_screening_filtered.fasta

✔ Filtered FASTA detected.
✔ Sequences found: 76

STEP 25: INTERPROSCAN DIRECTORY

✔ InterProScan found.

STEP 26: INTERPROSCAN OUTPUT

Output location:
C:\metp\output\testing_raw_blast_screening\testing_interpro.tsv

STEP 27: RUNNING INTERPROSCAN

Input FASTA:
C:\metp\output\testing_raw_blast_screening\testing_raw_blast_screening_filtered.fasta

Number of sequences:
76

WSL input path:
/mnt/c/metp/output/testing_raw_blast_screening/testing_raw_blast_screening_filtered.fasta

WSL output path:
/mnt/c/metp/output/testing_raw_blast_screening/testing_interpro.tsv

InterProScan:
/home/nurly/metisa_interproscan/interproscan/interproscan-5.78-109.0/interproscan.sh

InterProScan running...
Processing 76 protein sequences.
This may take some time.

Command:
wsl bash /home/nurly/metisa_interproscan/interproscan/interproscan-5.78-109.0/inter


## Downstream processing within the initial analysis stage

The preceding analysis performs the initial sequence-processing workflow required to prepare reliable inputs for candidate discovery.

The resulting outputs provide the foundation for the following stages:

1. quality-controlled protein sequences
2. homology-search results
3. filtered and taxonomically screened hits
4. functional-domain annotations

These outputs are carried forward into candidate mining.



# 05. Candidate Mining

Candidate mining integrates the available evidence to identify proteins with potential value as targeted pesticide candidates.

The analysis considers the evidence generated from the preceding homology, filtering, taxonomy and functional-domain analyses.

Candidates are prioritised according to the project's predefined biological criteria, allowing the large predicted proteome to be reduced to a more manageable set of proteins for detailed downstream analysis.


In [6]:

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment
    from openpyxl.utils import get_column_letter
    OPENPYXL_AVAILABLE = True
except ImportError:
    OPENPYXL_AVAILABLE = False

DEFAULT_MAX_CANDIDATES = 500

MIN_IDENTITY_MINING = 30.0
MIN_QUERY_COVERAGE_MINING = 70.0
MAX_EVALUE_MINING = 1e-10

STRONG_IDENTITY_MINING = 40.0
STRONG_COVERAGE_MINING = 80.0
STRONG_EVALUE_MINING = 1e-20

MIN_ARTHROPOD_FRACTION_KEEP = 0.70
MIN_ARTHROPOD_FRACTION_STRONG = 0.90

MAX_HITS_PER_GENE = 50

WEIGHT_HOMOLOGY = 30.0
WEIGHT_TAXONOMY = 25.0
WEIGHT_ANNOTATION = 15.0
WEIGHT_INTERPRO = 15.0
WEIGHT_TARGET_CLASS = 15.0

CANDIDATE_COLUMNS = [
    "gene_id", "rank", "final_score", "decision", "reason",
    "sequence_length", "best_hit_accession", "best_hit_species",
    "best_hit_taxid", "best_hit_lineage", "best_identity_pct",
    "best_query_coverage_pct", "best_evalue", "best_bitscore",
    "qualifying_hits", "total_blast_hits", "arthropod_fraction",
    "bacterial_fraction", "fungal_fraction",
    "non_arthropod_eukaryote_fraction", "other_prokaryote_fraction",
    "taxonomy_classification", "homology_score", "taxonomy_score",
    "annotation_score", "interpro_score", "target_class_score",
    "functional_description", "interpro_accessions",
    "interpro_descriptions", "pfam", "panther", "go_terms",
    "functional_keywords", "target_class", "best_hit_description",
]

EVIDENCE_COLUMNS = [
    "gene_id", "hit_rank", "sseqid", "pident", "qcov", "evalue",
    "bitscore", "length", "qlen", "slen", "qualifies", "strong_hit",
    "stitle",
]

REJECTED_COLUMNS = [
    x for x in CANDIDATE_COLUMNS
    if x != "rank"
]

def clean_text(value):
    if value is None:
        return ""
    return str(value).strip()


def safe_float(value, default=0.0):
    try:
        return float(str(value).strip())
    except Exception:
        return default


def safe_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except Exception:
        return default


def normalize_gene_id(value):
    value = clean_text(value)

    if value.startswith(">"):
        value = value[1:]

    if not value:
        return ""

    return value.split()[0]


def normalize_column_name(name):
    name = clean_text(name).lower()
    name = name.replace(" ", "")
    name = name.replace("-", "_")
    name = name.replace(".", "_")
    return name


def find_column(fieldnames, candidates):
    if not fieldnames:
        return None

    normalized = {
        normalize_column_name(x): x
        for x in fieldnames
    }

    for candidate in candidates:
        key = normalize_column_name(candidate)

        if key in normalized:
            return normalized[key]

    return None


def safe_filename(name):
    name = clean_text(name)
    name = os.path.basename(name)

    for suffix in [
        ".tsv",
        ".txt",
        ".xlsx",
        ".csv"
    ]:
        if name.lower().endswith(suffix):
            name = name[:-len(suffix)]

    if not name:
        name = "metisa_results"

    return name


def ask_existing_file(prompt):
    while True:
        path = input(prompt).strip().strip('"')

        if not path:
            print("Please enter a file path.")
            continue

        path = os.path.abspath(
            os.path.expanduser(path)
        )

        if os.path.isfile(path):
            return path

        print("File not found:")
        print(path)
        print()


def ask_output_basename():
    while True:
        value = input(
            "Output base name [metisa_results]: "
        ).strip().strip('"')

        if not value:
            value = "metisa_results"

        value = safe_filename(value)

        if value:
            return value

        print("Please enter a valid filename.")

def ask_max_candidates():
    while True:
        value = input(
            f"Number of final candidates [{DEFAULT_MAX_CANDIDATES}]: "
        ).strip()

        if not value:
            return DEFAULT_MAX_CANDIDATES

        try:
            value = int(value)

            if value <= 0:
                print("Please enter a number greater than 0.")
                continue

            return value

        except ValueError:
            print("Please enter a whole number.")

def print_header(title):
    print()
    print("=" * 78)
    print(title)
    print("=" * 78)


def join_unique(values, separator="; "):
    seen = set()
    result = []

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        if value in seen:
            continue

        seen.add(value)
        result.append(value)

    return separator.join(result)


def truncate_text(value, maximum=1000):
    value = clean_text(value)

    if len(value) <= maximum:
        return value

    return value[:maximum - 3] + "..."


def write_tsv(path, rows, fieldnames):
    with open(
        path,
        "w",
        encoding="utf-8",
        newline=""
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            delimiter="\t",
            extrasaction="ignore"
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


def write_text(path, text):
    with open(
        path,
        "w",
        encoding="utf-8"
    ) as handle:
        handle.write(text)

def read_fasta(path):
    sequences = {}

    current_id = None
    chunks = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as handle:

        for raw_line in handle:

            line = raw_line.strip()

            if not line:
                continue

            if line.startswith(">"):

                if current_id is not None:
                    sequences[current_id] = "".join(chunks)

                current_id = normalize_gene_id(line)
                chunks = []

            else:
                chunks.append(
                    line.replace(" ", "")
                )

    if current_id is not None:
        sequences[current_id] = "".join(chunks)

    return sequences


def validate_sequences(sequences):

    valid_amino_acids = set(
        "ACDEFGHIKLMNPQRSTVWY"
        "BJOUXZ"
    )

    stats = {
        "total": len(sequences),
        "empty": 0,
        "internal_stop": 0,
        "invalid_characters": 0,
    }

    for sequence in sequences.values():

        sequence = sequence.upper()

        if not sequence:
            stats["empty"] += 1
            continue

        if "*" in sequence[:-1]:
            stats["internal_stop"] += 1

        invalid = (
            set(sequence)
            - valid_amino_acids
            - {"*"}
        )

        if invalid:
            stats["invalid_characters"] += 1

    return stats

def parse_blast(path):
    hits = defaultdict(list)

    with open(
        path, "r", encoding="utf-8-sig",
        errors="replace", newline=""
    ) as handle:

        reader = csv.DictReader(
            handle, delimiter="\t"
        )

        if not reader.fieldnames:
            raise ValueError(
                "BLAST file does not contain a header."
            )

        fields = reader.fieldnames

        qseqid_col = find_column(
            fields,
            ["qseqid", "query", "query_id", "gene_id"]
        )

        sseqid_col = find_column(
            fields,
            ["sseqid", "subject", "subject_id", "accession"]
        )

        pident_col = find_column(
            fields,
            ["pident", "identity", "identity_pct"]
        )

        length_col = find_column(
            fields,
            ["length", "alignment_length"]
        )

        qlen_col = find_column(
            fields,
            ["qlen", "query_length"]
        )

        slen_col = find_column(
            fields,
            ["slen", "subject_length"]
        )

        qcov_col = find_column(
            fields,
            ["qcov", "query_coverage", "best_query_coverage"]
        )

        evalue_col = find_column(
            fields,
            ["evalue", "e_value"]
        )

        bitscore_col = find_column(
            fields,
            ["bitscore", "bit_score"]
        )

        title_col = find_column(
            fields,
            ["stitle", "description",
             "subject_title", "best_hit_description"]
        )

        required = {
            "qseqid": qseqid_col,
            "sseqid": sseqid_col,
            "pident": pident_col,
            "evalue": evalue_col,
            "bitscore": bitscore_col,
        }

        missing = [
            name
            for name, column in required.items()
            if column is None
        ]

        if missing:
            raise ValueError(
                "BLAST file is missing required columns: "
                + ", ".join(missing)
            )

        total_rows = 0

        for row in reader:
            total_rows += 1

            gene_id = normalize_gene_id(
                row.get(qseqid_col)
            )

            if not gene_id:
                continue

            hit = {
                "qseqid": gene_id,

                "sseqid": clean_text(
                    row.get(sseqid_col)
                ),

                "pident": safe_float(
                    row.get(pident_col)
                ),

                "length": (
                    safe_int(row.get(length_col))
                    if length_col else 0
                ),

                "qlen": (
                    safe_int(row.get(qlen_col))
                    if qlen_col else 0
                ),

                "slen": (
                    safe_int(row.get(slen_col))
                    if slen_col else 0
                ),

                "qcov": (
                    safe_float(row.get(qcov_col))
                    if qcov_col else 0.0
                ),

                "evalue": safe_float(
                    row.get(evalue_col)
                ),

                "bitscore": safe_float(
                    row.get(bitscore_col)
                ),

                "stitle": (
                    clean_text(row.get(title_col))
                    if title_col else ""
                ),
            }

            hits[gene_id].append(hit)

    return hits, total_rows

def qualify_blast_hit(hit):
    return (
        hit["pident"] >= MIN_IDENTITY_MINING
        and
        hit["qcov"] >= MIN_QUERY_COVERAGE_MINING
        and
        hit["evalue"] <= MAX_EVALUE_MINING
    )


def strong_blast_hit(hit):
    return (
        hit["pident"] >= STRONG_IDENTITY_MINING
        and
        hit["qcov"] >= STRONG_COVERAGE_MINING
        and
        hit["evalue"] <= STRONG_EVALUE_MINING
    )

def blast_hit_strength(hit):
    identity = hit["pident"]
    coverage = hit["qcov"]
    evalue = hit["evalue"]

    identity_score = (
        (identity - MIN_IDENTITY_MINING)
        / (70.0 - MIN_IDENTITY_MINING)
        * 100.0
    )

    identity_score = max(
        0.0,
        min(100.0, identity_score)
    )

    coverage_score = (
        (coverage - MIN_QUERY_COVERAGE_MINING)
        / (100.0 - MIN_QUERY_COVERAGE_MINING)
        * 100.0
    )

    coverage_score = max(
        0.0,
        min(100.0, coverage_score)
    )

    if evalue <= 0:
        evalue_score = 100.0
    else:
        neglog = -math.log10(
            max(evalue, 1e-300)
        )

        # 1e-10 -> ~33
        # 1e-20 -> ~67
        # 1e-30 -> 100
        evalue_score = (
            neglog
            / 30.0
            * 100.0
        )

        evalue_score = max(
            0.0,
            min(100.0, evalue_score)
        )

    return (
        identity_score * 0.50
        + coverage_score * 0.30
        + evalue_score * 0.20
    )


def select_best_hit(hits):
    qualifying = [
        hit
        for hit in hits
        if qualify_blast_hit(hit)
    ]

    if not qualifying:
        return None, []

    qualifying.sort(
        key=lambda x: (
            x["bitscore"],
            x["pident"],
            x["qcov"],
            -math.log10(
                max(x["evalue"], 1e-300)
            )
        ),
        reverse=True
    )

    return qualifying[0], qualifying

def parse_interpro(path):
    annotations = defaultdict(
        lambda: {
            "interpro_accessions": [],
            "interpro_descriptions": [],
            "signature_accessions": [],
            "signature_descriptions": [],
            "pfam": [],
            "panther": [],
            "go_terms": [],
            "pathways": [],
        }
    )

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as handle:

        reader = csv.DictReader(
            handle,
            delimiter="\t"
        )

        if not reader.fieldnames:
            raise ValueError(
                "InterPro file does not contain a header."
            )

        fields = reader.fieldnames

        protein_col = find_column(
            fields,
            [
                "Protein Accession",
                "protein_accession",
                "protein",
                "accession"
            ]
        )

        analysis_col = find_column(
            fields,
            [
                "Analysis",
                "analysis"
            ]
        )

        signature_col = find_column(
            fields,
            [
                "Signature accession",
                "signature_accession"
            ]
        )

        signature_desc_col = find_column(
            fields,
            [
                "Signature description",
                "signature_description"
            ]
        )

        start_col = find_column(
            fields,
            [
                "Start location",
                "start"
            ]
        )

        stop_col = find_column(
            fields,
            [
                "Stop location",
                "stop"
            ]
        )

        score_col = find_column(
            fields,
            [
                "Score",
                "score"
            ]
        )

        interpro_col = find_column(
            fields,
            [
                "InterPro accession",
                "interpro_accession"
            ]
        )

        interpro_desc_col = find_column(
            fields,
            [
                "InterPro description",
                "interpro_description"
            ]
        )

        go_col = find_column(
            fields,
            [
                "GO terms",
                "go_terms",
                "go"
            ]
        )

        pathway_col = find_column(
            fields,
            [
                "Pathways",
                "pathways",
                "pathway"
            ]
        )

        if protein_col is None:
            raise ValueError(
                "InterPro file is missing "
                "'Protein Accession'."
            )

        for row in reader:
            gene_id = normalize_gene_id(
                row.get(protein_col)
            )

            if not gene_id:
                continue

            annotation = annotations[gene_id]

            analysis = (
                clean_text(row.get(analysis_col))
                if analysis_col else ""
            )

            signature = (
                clean_text(row.get(signature_col))
                if signature_col else ""
            )

            signature_desc = (
                clean_text(
                    row.get(signature_desc_col)
                )
                if signature_desc_col else ""
            )

            interpro = (
                clean_text(row.get(interpro_col))
                if interpro_col else ""
            )

            interpro_desc = (
                clean_text(
                    row.get(interpro_desc_col)
                )
                if interpro_desc_col else ""
            )

            go = (
                clean_text(row.get(go_col))
                if go_col else ""
            )

            pathway = (
                clean_text(row.get(pathway_col))
                if pathway_col else ""
            )

            if interpro and interpro != "-":
                annotation[
                    "interpro_accessions"
                ].append(interpro)

            if interpro_desc and interpro_desc != "-":
                annotation[
                    "interpro_descriptions"
                ].append(interpro_desc)

            if signature and signature != "-":
                annotation[
                    "signature_accessions"
                ].append(signature)

            if signature_desc and signature_desc != "-":
                annotation[
                    "signature_descriptions"
                ].append(signature_desc)

            if analysis.upper() == "PFAM":
                if signature and signature != "-":
                    annotation["pfam"].append(
                        signature
                    )

                if signature_desc and signature_desc != "-":
                    annotation["pfam"].append(
                        signature_desc
                    )

            if analysis.upper() == "PANTHER":
                if signature and signature != "-":
                    annotation["panther"].append(
                        signature
                    )

                if signature_desc and signature_desc != "-":
                    annotation["panther"].append(
                        signature_desc
                    )

            if go and go != "-":
                annotation["go_terms"].append(go)

            if pathway and pathway != "-":
                annotation["pathways"].append(
                    pathway
                )

    return annotations

def has_real_interpro_evidence(annotation):

    if not annotation:
        return False

    evidence_keys = [
        "interpro_accessions",
        "interpro_descriptions",
        "signature_accessions",
        "signature_descriptions",
        "pfam",
        "panther",
        "go_terms",
        "pathways",
    ]

    for key in evidence_keys:
        values = annotation.get(key, [])

        for value in values:
            value = clean_text(value)

            if value and value != "-":
                return True

    return False

def parse_taxonomy(path):

    taxonomy = {}

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as handle:

        reader = csv.DictReader(
            handle,
            delimiter="\t"
        )

        if not reader.fieldnames:
            raise ValueError(
                "Taxonomy file does not contain a header."
            )

        fields = reader.fieldnames

        gene_col = find_column(
            fields,
            [
                "gene_id",
                "qseqid",
                "protein_accession"
            ]
        )

        if gene_col is None:
            raise ValueError(
                "Taxonomy file is missing gene_id."
            )

        for row in reader:

            gene_id = normalize_gene_id(
                row.get(gene_col)
            )

            if not gene_id:
                continue

            item = {}

            for field in fields:
                item[
                    normalize_column_name(field)
                ] = clean_text(
                    row.get(field)
                )

            taxonomy[gene_id] = item

    return taxonomy


def taxonomy_value(item, names, default=""):

    if not item:
        return default

    for name in names:

        key = normalize_column_name(name)

        if key in item:
            return item[key]

    return default


def taxonomy_fraction(item, names):

    return safe_float(
        taxonomy_value(
            item,
            names,
            "0"
        ),
        0.0
    )

def classify_taxonomy(
    item,
    blast_qualifying_hits
):
    if item:
        arth_frac = taxonomy_fraction(
            item, ["arthropod_fraction"]
        )

        bacteria_frac = taxonomy_fraction(
            item,
            [
                "bacterial_fraction",
                "bacteria_fraction"
            ]
        )

        fungal_frac = taxonomy_fraction(
            item,
            [
                "fungal_fraction",
                "fungi_fraction"
            ]
        )

        nonarth_frac = taxonomy_fraction(
            item,
            ["non_arthropod_eukaryote_fraction"]
        )

        other_prok_frac = taxonomy_fraction(
            item,
            ["other_prokaryote_fraction"]
        )

        classification = taxonomy_value(
            item,
            [
                "taxonomy_classification",
                "classification"
            ]
        )

        decision = taxonomy_value(
            item, ["decision"]
        )

        return {
            "arthropod_fraction": arth_frac,
            "bacterial_fraction": bacteria_frac,
            "fungal_fraction": fungal_frac,
            "non_arthropod_eukaryote_fraction": nonarth_frac,
            "other_prokaryote_fraction": other_prok_frac,
            "classification": classification,
            "decision": decision,
        }

    arthropod = 0
    bacterial = 0
    fungal = 0
    nonarth = 0
    other_prok = 0
    unknown = 0

    for hit in blast_qualifying_hits:
        title = hit["stitle"].lower()

        if any(
            word in title
            for word in [
                "bacter", "escherichia", "bacillus",
                "streptococcus", "staphylococcus",
                "pseudomonas", "mycobacterium"
            ]
        ):
            bacterial += 1

        elif any(
            word in title
            for word in [
                "fung", "candida", "aspergillus",
                "saccharomyces", "yeast"
            ]
        ):
            fungal += 1

        elif any(
            word in title
            for word in [
                "insect", "lepidoptera", "diptera",
                "coleoptera", "hemiptera", "hymenoptera",
                "blattodea", "orthoptera", "arachnid",
                "arthropod"
            ]
        ):
            arthropod += 1

        elif any(
            word in title
            for word in [
                "plant", "arabidopsis", "rice",
                "maize", "soybean", "human", "mouse",
                "fish", "vertebrate", "mammal"
            ]
        ):
            nonarth += 1

        else:
            unknown += 1

    total = (
        arthropod
        + bacterial
        + fungal
        + nonarth
        + other_prok
        + unknown
    )

    if total == 0:
        total = 1

    arth_frac = arthropod / total
    bact_frac = bacterial / total
    fungal_frac = fungal / total
    nonarth_frac = nonarth / total
    other_frac = other_prok / total

    if arth_frac >= MIN_ARTHROPOD_FRACTION_KEEP:
        classification = "KEEP_ARTHROPOD"
    elif bact_frac >= 0.50:
        classification = "LIKELY_BACTERIAL"
    elif fungal_frac >= 0.50:
        classification = "LIKELY_FUNGAL"
    elif nonarth_frac >= 0.50:
        classification = "NON_ARTHROPOD_EUKARYOTE"
    else:
        classification = "MIXED_OR_UNKNOWN"

    return {
        "arthropod_fraction": arth_frac,
        "bacterial_fraction": bact_frac,
        "fungal_fraction": fungal_frac,
        "non_arthropod_eukaryote_fraction": nonarth_frac,
        "other_prokaryote_fraction": other_frac,
        "classification": classification,
        "decision": classification,
    }

def annotation_quality(annotation):

    if not annotation:
        return 0.0

    score = 0.0

    if annotation["interpro_accessions"]:
        score += 35.0

    if annotation["interpro_descriptions"]:
        score += 20.0

    if annotation["pfam"]:
        score += 20.0

    if annotation["panther"]:
        score += 15.0

    if annotation["go_terms"]:
        score += 7.0

    if annotation["pathways"]:
        score += 3.0

    return min(100.0, score)

def annotation_keywords(annotation):
    if not annotation:
        return []

    text_parts = []

    for key in [
        "interpro_descriptions",
        "signature_descriptions",
        "pfam",
        "panther"
    ]:
        text_parts.extend(
            annotation.get(key, [])
        )

    text = " ".join(text_parts).lower()

    groups = {
        "enzyme": [
            "enzyme",
            "hydrolase",
            "transferase",
            "oxidoreductase",
            "protease",
            "peptidase",
            "phosphatase",
            "kinase"
        ],

        "transcription_factor": [
            "transcription factor",
            "transcriptional regulator",
            "dna-binding",
            "myb",
            "madf",
            "zinc finger"
        ],

        "receptor": [
            "receptor",
            "ligand-binding"
        ],

        "transporter": [
            "transporter",
            "transport",
            "channel"
        ],

        "membrane": [
            "transmembrane",
            "membrane protein"
        ],

        "development": [
            "development",
            "developmental",
            "growth",
            "cell differentiation"
        ],

        "digestion": [
            "digest",
            "amylase",
            "lipase",
            "trypsin",
            "chymotrypsin",
            "protease",
            "peptidase",
            "carbohydrase",
            "glycosidase"
        ],

        "hormone": [
            "hormone",
            "ecdysone",
            "juvenile hormone"
        ],

        "signal": [
            "signal peptide",
            "signaling",
            "signalling"
        ],
    }

    result = []

    for group, terms in groups.items():
        if any(
            term in text
            for term in terms
        ):
            result.append(group)

    return result

def calculate_homology_score(
    best_hit,
    qualifying_hits
):

    if best_hit is None:
        return 0.0

    best_strength = blast_hit_strength(
        best_hit
    )

    n = len(qualifying_hits)

    hit_support = (
        1.0
        - math.exp(-n / 3.0)
    )

    hit_count_score = (
        hit_support * 100.0
    )

    strong_hits = sum(
        1
        for hit in qualifying_hits
        if strong_blast_hit(hit)
    )

    strong_support = (
        1.0
        - math.exp(-strong_hits / 2.0)
    )

    strong_score = (
        strong_support * 100.0
    )

    # Best hit remains dominant.
    score = (
        best_strength * 0.70
        +
        hit_count_score * 0.15
        +
        strong_score * 0.15
    )

    return min(
        100.0,
        score
    )

def classify_target_class(
    best_hit,
    annotation
):

    text_parts = []

    # BLAST description
    if best_hit:
        text_parts.append(
            clean_text(
                best_hit.get("stitle", "")
            )
        )

    # InterPro / Pfam / Panther descriptions
    if annotation:
        for key in [
            "interpro_descriptions",
            "signature_descriptions",
            "pfam",
            "panther",
        ]:
            text_parts.extend(
                annotation.get(key, [])
            )

    text = " ".join(
        clean_text(x)
        for x in text_parts
    ).lower()

    chitin_terms = [
        "chitin synthase",
        "chitinase",
        "chitin binding",
        "chitin-binding",
        "chitin metabolism",
        "chitin biosynthesis",
        "chitin degradation",
        "chitinase activity",
        "chitin synthase activity",
        "chitin",
    ]

    if any(
        term in text
        for term in chitin_terms
    ):
        return "CHITIN"

    development_terms = [
        "ecdysone",
        "ecdysteroid",
        "juvenile hormone",
        "juvenile hormone binding",
        "juvenile hormone esterase",
        "juvenile hormone receptor",
        "development",
        "developmental",
        "growth",
        "molting",
        "moulting",
        "metamorphosis",
        "cell differentiation",
    ]

    if any(
        term in text
        for term in development_terms
    ):
        return "GROWTH_DEVELOPMENT"

    digestion_terms = [
        "digestive enzyme",
        "digestion",
        "amylase",
        "alpha-amylase",
        "lipase",
        "trypsin",
        "trypsin-like",
        "chymotrypsin",
        "chymotrypsin-like",
        "peptidase",
        "protease",
        "carboxypeptidase",
        "aminopeptidase",
        "glycosidase",
        "carbohydrase",
        "cellulase",
    ]

    if any(
        term in text
        for term in digestion_terms
    ):
        return "DIGESTION"

    detox_terms = [
        "cytochrome p450",
        "cytochrome p450 monooxygenase",
        "glutathione s-transferase",
        "glutathione transferase",
        "gst",
        "carboxylesterase",
        "esterase",
        "detoxification",
        "detoxification enzyme",
        "xenobiotic",
        "xenobiotic metabolism",
        "udp-glucuronosyltransferase",
        "udp-glycosyltransferase",
        "ug t",
        "aldo-keto reductase",
    ]

    if any(
        term in text
        for term in detox_terms
    ):
        return "DETOXIFICATION"

    signaling_terms = [
        "receptor",
        "ligand-binding receptor",
        "g-protein coupled receptor",
        "gpcr",
        "signaling",
        "signalling",
        "signal transduction",
        "protein kinase",
        "kinase",
    ]

    if any(
        term in text
        for term in signaling_terms
    ):
        return "SIGNALING"

    transport_terms = [
        "transporter",
        "transport protein",
        "ion channel",
        "channel protein",
        "abc transporter",
        "atp-binding cassette",
        "membrane transporter",
    ]

    if any(
        term in text
        for term in transport_terms
    ):
        return "TRANSPORT"

    regulation_terms = [
        "transcription factor",
        "transcriptional regulator",
        "dna-binding",
        "transcriptional activator",
        "transcriptional repressor",
        "zinc finger",
    ]

    if any(
        term in text
        for term in regulation_terms
    ):
        return "TRANSCRIPTION_REGULATION"

    return "OTHER"


TARGET_CLASS_SCORES = {
    "CHITIN": 100.0,
    "GROWTH_DEVELOPMENT": 100.0,
    "DIGESTION": 95.0,
    "SIGNALING": 85.0,
    "DETOXIFICATION": 80.0,
    "TRANSPORT": 70.0,
    "TRANSCRIPTION_REGULATION": 65.0,
    "OTHER": 30.0,
}

def calculate_target_class_score(target_class):
    return TARGET_CLASS_SCORES.get(
        target_class,
        30.0
    )

def calculate_taxonomy_score(tax):

    if not tax:
        return 0.0

    arth = tax[
        "arthropod_fraction"
    ]

    bact = tax[
        "bacterial_fraction"
    ]

    fungal = tax[
        "fungal_fraction"
    ]

    nonarth = tax[
        "non_arthropod_eukaryote_fraction"
    ]

    other_prok = tax[
        "other_prokaryote_fraction"
    ]

    # Arthropod support.
    score = arth * 100.0

    # Strong contamination penalties.
    score -= bact * 100.0
    score -= fungal * 90.0
    score -= other_prok * 90.0

    # Non-arthropod eukaryotes are less alarming
    # than bacterial/fungal contamination.
    score -= nonarth * 45.0

    return max(
        0.0,
        min(100.0, score)
    )

def calculate_annotation_score(
    best_hit,
    annotation
):

    score = 0.0

    if best_hit:

        description = best_hit[
            "stitle"
        ].lower()

        if description:

            if "hypothetical protein" in description:
                score += 3.0

            elif "uncharacterized" in description:
                score += 5.0

            elif any(
                term in description
                for term in [
                    "trypsin",
                    "chymotrypsin",
                    "peptidase",
                    "protease",
                    "amylase",
                    "lipase",
                    "chitinase",
                    "chitin synthase",
                    "cytochrome p450",
                    "glutathione s-transferase",
                    "ecdysone",
                    "ecdysteroid",
                    "juvenile hormone",
                    "receptor",
                    "transporter",
                    "kinase",
                    "transcription factor",
                    "enzyme"
                ]
            ):
                score += 30.0

            else:
                score += 10.0

    return min(
        100.0,
        score
    )

def calculate_final_score(
    homology_score,
    taxonomy_score,
    annotation_score,
    interpro_score,
    target_class_score,
    tax
):

    score = (
        homology_score
        * WEIGHT_HOMOLOGY
        / 100.0
        +
        taxonomy_score
        * WEIGHT_TAXONOMY
        / 100.0
        +
        annotation_score
        * WEIGHT_ANNOTATION
        / 100.0
        +
        interpro_score
        * WEIGHT_INTERPRO
        / 100.0
        +
        target_class_score
        * WEIGHT_TARGET_CLASS
        / 100.0
    )

    # Additional contamination penalty.
    contamination = (
        tax["bacterial_fraction"] * 30.0
        +
        tax["fungal_fraction"] * 25.0
        +
        tax["other_prokaryote_fraction"] * 30.0
        +
        tax["non_arthropod_eukaryote_fraction"] * 10.0
    )

    score -= contamination

    return max(
        0.0,
        min(100.0, score)
    )

def determine_decision(
    best_hit,
    qualifying_hits,
    tax,
    annotation,
    final_score
):

    if best_hit is None:
        return (
            "REJECT",
            "No BLAST hit passed the identity, "
            "coverage and E-value filter."
        )

    arth = tax[
        "arthropod_fraction"
    ]

    bact = tax[
        "bacterial_fraction"
    ]

    fungal = tax[
        "fungal_fraction"
    ]

    nonarth = tax[
        "non_arthropod_eukaryote_fraction"
    ]

    classification = tax.get(
        "classification",
        ""
    ).upper()

    if bact >= 0.70:
        return (
            "REJECT",
            "Strong bacterial support among "
            "qualifying hits."
        )

    if fungal >= 0.70:
        return (
            "REJECT",
            "Strong fungal support among "
            "qualifying hits."
        )

    if classification in {
        "LIKELY_BACTERIAL",
        "LIKELY_FUNGAL"
    }:
        return (
            "REJECT",
            "Taxonomy classification indicates "
            "likely microbial contamination."
        )

    if (
        arth >= MIN_ARTHROPOD_FRACTION_STRONG
        and
        final_score >= 70
        and
        len(qualifying_hits) >= 2
    ):
        return (
            "KEEP",
            "Strong arthropod homology with multiple "
            "qualifying hits and strong combined evidence."
        )

    if (
        arth >= MIN_ARTHROPOD_FRACTION_KEEP
        and
        final_score >= 60
    ):
        return (
            "KEEP",
            "Good arthropod support and sufficient "
            "combined evidence."
        )

    if nonarth >= 0.60:
        return (
            "REVIEW",
            "Most qualifying hits are non-arthropod "
            "eukaryotes."
        )

    if arth >= 0.40:
        return (
            "REVIEW",
            "Arthropod support exists but taxonomy "
            "is mixed."
        )

    return (
        "REVIEW",
        "Evidence is insufficient for confident "
        "arthropod classification."
    )

def choose_functional_description(
    best_hit,
    annotation
):

    if annotation:

        descriptions = annotation.get(
            "interpro_descriptions",
            []
        )

        for description in descriptions:

            description = clean_text(
                description
            )

            if (
                description
                and
                description != "-"
            ):
                return description

        descriptions = annotation.get(
            "signature_descriptions",
            []
        )

        for description in descriptions:

            description = clean_text(
                description
            )

            if (
                description
                and
                description != "-"
            ):
                return description

    if best_hit:
        return best_hit["stitle"]

    return ""

def extract_species(title):

    title = clean_text(title)

    if not title:
        return ""

    # UniProt:
    # Protein name OS=Species OX=TaxID
    match = re.search(
        r"\bOS=([^=]+?)(?:\s+OX=|\s+GN=|\s+PE=|\s+SV=|$)",
        title
    )

    if match:
        return match.group(1).strip()

    # Generic fallback.
    match = re.search(
        r"\bOS=([A-Z][^=]+)",
        title
    )

    if match:
        return match.group(1).strip()

    return ""

def build_candidate(
    gene_id,
    blast_hits,
    annotation,
    taxonomy,
    sequences
):

    all_hits = sorted(
        blast_hits,
        key=lambda x: (
            x["bitscore"],
            x["pident"],
            x["qcov"],
            -math.log10(max(x["evalue"], 1e-300))
        ),
        reverse=True
    )[:MAX_HITS_PER_GENE]

    best_hit, qualifying_hits = select_best_hit(all_hits)

    tax_item = taxonomy.get(gene_id, {})
    tax = classify_taxonomy(tax_item, qualifying_hits)

    homology_score = calculate_homology_score(
        best_hit,
        qualifying_hits
    )

    taxonomy_score = calculate_taxonomy_score(tax)
    interpro_score = annotation_quality(annotation)

    annotation_score = calculate_annotation_score(
        best_hit,
        annotation
    )

    target_class = classify_target_class(
        best_hit,
        annotation
    )

    target_class_score = calculate_target_class_score(
        target_class
    )

    final_score = calculate_final_score(
        homology_score,
        taxonomy_score,
        annotation_score,
        interpro_score,
        target_class_score,
        tax
    )

    decision, reason = determine_decision(
        best_hit,
        qualifying_hits,
        tax,
        annotation,
        final_score
    )

    sequence = sequences.get(gene_id, "")
    description = choose_functional_description(
        best_hit,
        annotation
    )

    # --------------------------------------------------------
    # BEST HIT DETAILS
    # --------------------------------------------------------

    if best_hit:

        best_accession = best_hit["sseqid"]

        best_species = taxonomy_value(
            tax_item,
            ["best_hit_species"]
        ) or extract_species(
            best_hit["stitle"]
        )

        best_taxid = taxonomy_value(
            tax_item,
            ["best_hit_taxid"]
        )

        best_lineage = taxonomy_value(
            tax_item,
            ["best_hit_lineage"]
        )

        best_identity = best_hit["pident"]
        best_coverage = best_hit["qcov"]
        best_evalue = best_hit["evalue"]
        best_bitscore = best_hit["bitscore"]
        best_description = best_hit["stitle"]

    else:

        best_accession = ""
        best_species = ""
        best_taxid = ""
        best_lineage = ""
        best_identity = ""
        best_coverage = ""
        best_evalue = ""
        best_bitscore = ""
        best_description = ""

    row = {
        "gene_id": gene_id,
        "rank": "",
        "final_score": round(final_score, 3),
        "decision": decision,
        "reason": reason,
        "sequence_length": len(sequence),

        "best_hit_accession": best_accession,
        "best_hit_species": best_species,
        "best_hit_taxid": best_taxid,
        "best_hit_lineage": best_lineage,

        "best_identity_pct": best_identity,
        "best_query_coverage_pct": best_coverage,
        "best_evalue": best_evalue,
        "best_bitscore": best_bitscore,

        "qualifying_hits": len(qualifying_hits),
        "total_blast_hits": len(blast_hits),

        "arthropod_fraction": round(
            tax["arthropod_fraction"], 4
        ),
        "bacterial_fraction": round(
            tax["bacterial_fraction"], 4
        ),
        "fungal_fraction": round(
            tax["fungal_fraction"], 4
        ),
        "non_arthropod_eukaryote_fraction": round(
            tax["non_arthropod_eukaryote_fraction"], 4
        ),
        "other_prokaryote_fraction": round(
            tax["other_prokaryote_fraction"], 4
        ),

        "taxonomy_classification": tax.get(
            "classification",
            ""
        ),

        "homology_score": round(
            homology_score, 3
        ),
        "taxonomy_score": round(
            taxonomy_score, 3
        ),
        "annotation_score": round(
            annotation_score, 3
        ),
        "interpro_score": round(
            interpro_score, 3
        ),

        "functional_description": description,

        "interpro_accessions": join_unique(
            annotation.get(
                "interpro_accessions",
                []
            )
        ),
        "interpro_descriptions": join_unique(
            annotation.get(
                "interpro_descriptions",
                []
            )
        ),
        "pfam": join_unique(
            annotation.get("pfam", [])
        ),
        "panther": join_unique(
            annotation.get("panther", [])
        ),
        "go_terms": join_unique(
            annotation.get("go_terms", [])
        ),

        "functional_keywords": "; ".join(
            annotation_keywords(annotation)
        ),

        "target_class_score": round(
            target_class_score, 3
        ),
        "target_class": target_class,

        "best_hit_description": best_description,
    }

    return row

def build_evidence_rows(
    blast_hits
):

    rows = []

    for gene_id, hits in blast_hits.items():

        limited_hits = sorted(
            hits,
            key=lambda x: (
                x["bitscore"],
                x["pident"],
                x["qcov"],
                -math.log10(
                    max(x["evalue"], 1e-300)
                )
            ),
            reverse=True
        )[:MAX_HITS_PER_GENE]

        sorted_hits = sorted(
            limited_hits,
            key=lambda x: (
                qualify_blast_hit(x),
                x["bitscore"]
            ),
            reverse=True
        )

        for rank, hit in enumerate(
            sorted_hits,
            start=1
        ):

            rows.append({
                "gene_id": gene_id,
                "hit_rank": rank,
                "sseqid": hit["sseqid"],
                "pident": hit["pident"],
                "qcov": hit["qcov"],
                "evalue": hit["evalue"],
                "bitscore": hit["bitscore"],
                "length": hit["length"],
                "qlen": hit["qlen"],
                "slen": hit["slen"],
                "qualifies":
                    "YES"
                    if qualify_blast_hit(hit)
                    else "NO",
                "strong_hit":
                    "YES"
                    if strong_blast_hit(hit)
                    else "NO",
                "stitle": hit["stitle"],
            })

    return rows

def write_rows_to_sheet(ws, rows, columns):

    # Header
    for col_num, column in enumerate(columns, start=1):

        cell = ws.cell(
            row=1,
            column=col_num,
            value=column
        )

        cell.font = Font(bold=True)
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Data
    for row_num, row in enumerate(rows, start=2):

        for col_num, column in enumerate(columns, start=1):

            value = row.get(column, "")
            ws.cell(
                row=row_num,
                column=col_num,
                value="" if value is None else value
            )

    ws.freeze_panes = "A2"

    if rows:
        ws.auto_filter.ref = ws.dimensions

    # Column widths
    for col_num, column in enumerate(columns, start=1):

        max_len = len(str(column))

        for row_num in range(
            2,
            min(ws.max_row, 501) + 1
        ):
            value = ws.cell(
                row=row_num,
                column=col_num
            ).value

            if value is not None:
                max_len = max(
                    max_len,
                    len(str(value))
                )

        ws.column_dimensions[
            get_column_letter(col_num)
        ].width = min(
            max(max_len + 2, 12),
            45
        )

    # Wrap text
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )


def write_excel(
    path,
    candidates,
    evidence,
    rejected,
    summary_text
):

    if not OPENPYXL_AVAILABLE:
        return False

    workbook = Workbook()
    workbook.remove(workbook.active)

    sheets = [
        ("Candidates", candidates, CANDIDATE_COLUMNS),
        ("BLAST_Evidence", evidence, EVIDENCE_COLUMNS),
        ("Rejected", rejected, REJECTED_COLUMNS),
    ]

    for sheet_name, rows, columns in sheets:

        ws = workbook.create_sheet(sheet_name)

        write_rows_to_sheet(
            ws,
            rows,
            columns
        )

    # Summary
    ws = workbook.create_sheet("Summary")

    for row_num, line in enumerate(
        summary_text.splitlines(),
        start=1
    ):
        ws.cell(
            row=row_num,
            column=1,
            value=line
        )

    ws.column_dimensions["A"].width = 110

    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

    workbook.save(path)

    return True

def make_summary(
    input_blast,
    input_interpro,
    input_taxonomy,
    input_fasta,
    sequences,
    blast_hits,
    candidates,
    rejected,
    total_blast_rows
):

    keep = sum(row["decision"] == "KEEP" for row in candidates)
    review = sum(row["decision"] == "REVIEW" for row in candidates)

    sequence_stats = validate_sequences(sequences)

    lines = [
        "METISA PLANA TARGET MINING SUMMARY",
        "=" * 60,
        "",
        "INPUT FILES",
        f"BLAST: {input_blast}",
        f"InterPro: {input_interpro}",
        f"Taxonomy: {input_taxonomy}",
        f"FASTA: {input_fasta}",
        "",
        "INPUT STATISTICS",
        f"FASTA proteins: {len(sequences)}",
        f"BLAST rows: {total_blast_rows}",
        f"BLAST query proteins: {len(blast_hits)}",
        f"Sequence empty: {sequence_stats['empty']}",
        f"Internal stop: {sequence_stats['internal_stop']}",
        f"Invalid amino-acid characters: {sequence_stats['invalid_characters']}",
        "",
        "SCORING",
        f"Homology weight: {WEIGHT_HOMOLOGY}%",
        f"Taxonomy weight: {WEIGHT_TAXONOMY}%",
        f"Annotation weight: {WEIGHT_ANNOTATION}%",
        f"InterPro weight: {WEIGHT_INTERPRO}%",
        f"Target class weight: {WEIGHT_TARGET_CLASS}%",
        "",
        "BLAST FILTER",
        f"Identity >= {MIN_IDENTITY_MINING}%",
        f"Query coverage >= {MIN_QUERY_COVERAGE_MINING}%",
        f"E-value <= {MAX_EVALUE_MINING}",
        "",
        "RESULTS",
        f"Final candidates: {len(candidates)}",
        f"KEEP: {keep}",
        f"REVIEW: {review}",
        f"REJECTED: {len(rejected)}",
    ]

    return "\n".join(lines) + "\n"

def main():

    print_header("METISA PLANA TARGET MINING")
    print("Combines BLAST, taxonomy, InterPro and sequence evidence.")

    print_header("INPUT FILES")

    blast_path = ask_existing_file("Filtered blast path [qualifying hits] : ")
    interpro_path = ask_existing_file("InterProScan TSV path: ")
    taxonomy_path = ask_existing_file("Screening TSV path: ")
    fasta_path = ask_existing_file("Filtered fasta path: ")

    print_header("OUTPUT")

    output_base = ask_output_basename()
    max_candidates = ask_max_candidates()
    output_dir = os.path.dirname(blast_path)

    candidate_path = os.path.join(
        output_dir, output_base + "_candidates.tsv"
    )
    evidence_path = os.path.join(
        output_dir, output_base + "_blast.tsv"
    )
    rejected_path = os.path.join(
        output_dir, output_base + "_rejected.tsv"
    )
    summary_path = os.path.join(
        output_dir, output_base + "_summary.txt"
    )
    excel_path = os.path.join(
        output_dir, output_base + ".xlsx"
    )

    print("\nOutput files:")
    for path in [
        candidate_path,
        evidence_path,
        rejected_path,
        excel_path,
        summary_path
    ]:
        print(" ", path)

    print_header("READING FASTA")

    sequences = read_fasta(fasta_path)
    sequence_stats = validate_sequences(sequences)

    print(f"Proteins loaded: {len(sequences):,}")
    print(f"Empty sequences: {sequence_stats['empty']:,}")
    print(f"Internal stops: {sequence_stats['internal_stop']:,}")
    print(
        f"Invalid characters: "
        f"{sequence_stats['invalid_characters']:,}"
    )

    print_header("READING BLAST")

    blast_hits, total_blast_rows = parse_blast(blast_path)

    print(f"BLAST rows: {total_blast_rows:,}")
    print(f"BLAST queries: {len(blast_hits):,}")

    print_header("READING INTERPRO")

    interpro = parse_interpro(interpro_path)

    print(
        f"Proteins with InterPro annotations: "
        f"{len(interpro):,}"
    )

    print_header("READING TAXONOMY")

    taxonomy = parse_taxonomy(taxonomy_path)

    print(f"Taxonomy records: {len(taxonomy):,}")

    print_header("CALCULATING CANDIDATE SCORES")

    gene_ids = {
        gene_id
        for gene_id, hits in blast_hits.items()
        if gene_id in sequences
        and any(qualify_blast_hit(hit) for hit in hits)
    }

    print(
        f"Proteins passing BLAST filter: "
        f"{len(gene_ids):,}"
    )

    all_candidates = []

    for index, gene_id in enumerate(gene_ids, start=1):

        all_candidates.append(
            build_candidate(
                gene_id,
                blast_hits.get(gene_id, []),
                interpro.get(gene_id, {}),
                taxonomy,
                sequences
            )
        )

        if index % 1000 == 0:
            print(f"Processed {index:,} proteins...")

    candidates = [
        row for row in all_candidates
        if row["decision"] in {"KEEP", "REVIEW"}
    ]

    candidates.sort(
        key=lambda row: (
            row["final_score"],
            row["homology_score"],
            row["taxonomy_score"]
        ),
        reverse=True
    )

    candidates = candidates[:max_candidates]

    for rank, row in enumerate(candidates, start=1):
        row["rank"] = rank

    rejected = [
        row for row in all_candidates
        if row["decision"] == "REJECT"
    ]

    rejected.sort(
        key=lambda row: row["final_score"],
        reverse=True
    )

    evidence_rows = build_evidence_rows(blast_hits)

    summary_text = make_summary(
        blast_path,
        interpro_path,
        taxonomy_path,
        fasta_path,
        sequences,
        blast_hits,
        candidates,
        rejected,
        total_blast_rows
    )

    print_header("WRITING OUTPUT")

    write_tsv(candidate_path, candidates, CANDIDATE_COLUMNS)
    print("Created:", candidate_path)

    write_tsv(evidence_path, evidence_rows, EVIDENCE_COLUMNS)
    print("Created:", evidence_path)

    write_tsv(rejected_path, rejected, REJECTED_COLUMNS)
    print("Created:", rejected_path)

    write_text(summary_path, summary_text)
    print("Created:", summary_path)

    if OPENPYXL_AVAILABLE:

        try:
            write_excel(
                excel_path,
                candidates,
                evidence_rows,
                rejected,
                summary_text
            )
            print("Created:", excel_path)

        except Exception as error:
            print("WARNING: Excel creation failed:")
            print(error)

    else:
        print("WARNING: openpyxl is not installed.")
        print("Excel file was not created.")
        print("Install with: pip install openpyxl")

    print_header("DONE")

    keep_count = sum(
        row["decision"] == "KEEP"
        for row in candidates
    )
    review_count = sum(
        row["decision"] == "REVIEW"
        for row in candidates
    )

    print(f"Top candidates exported: {len(candidates):,}")
    print(f"KEEP: {keep_count:,}")
    print(f"REVIEW: {review_count:,}")
    print(f"Rejected: {len(rejected):,}")
    print(f"\nAll outputs use the same base name: {output_base}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nProcess cancelled by user.")
    except Exception as error:
        print("\n" + "=" * 78)
        print("ERROR")
        print("=" * 78)
        print(error)


METISA PLANA TARGET MINING
Combines BLAST, taxonomy, InterPro and sequence evidence.

INPUT FILES

OUTPUT

Output files:
  C:\metp\output\testing_raw_blast_screening\testing_mining_candidates.tsv
  C:\metp\output\testing_raw_blast_screening\testing_mining_blast.tsv
  C:\metp\output\testing_raw_blast_screening\testing_mining_rejected.tsv
  C:\metp\output\testing_raw_blast_screening\testing_mining.xlsx
  C:\metp\output\testing_raw_blast_screening\testing_mining_summary.txt

READING FASTA
Proteins loaded: 76
Empty sequences: 0
Internal stops: 0
Invalid characters: 0

READING BLAST
BLAST rows: 749
BLAST queries: 77

READING INTERPRO
Proteins with InterPro annotations: 68

READING TAXONOMY
Taxonomy records: 99

CALCULATING CANDIDATE SCORES
Proteins passing BLAST filter: 72

WRITING OUTPUT
Created: C:\metp\output\testing_raw_blast_screening\testing_mining_candidates.tsv
Created: C:\metp\output\testing_raw_blast_screening\testing_mining_blast.tsv
Created: C:\metp\output\testing_raw_blast_scr


# 06. Expression Analysis

Expression evidence is incorporated to determine whether candidate genes are active across the available *M. plana* developmental stages.

Expression information provides biological context for candidate prioritisation.

Particular attention is given to expression patterns relevant to the target biological stage, allowing candidates with stronger evidence of activity in the relevant life stage to receive greater support during prioritisation.


In [7]:


# ==========================================================
# METISA PLANA
# RNA LIFE-STAGE EXPRESSION PRIORITIZATION
# ==========================================================

print("=" * 78)
print("METISA PLANA — RNA LIFE-STAGE EXPRESSION PRIORITIZATION")
print("=" * 78)


# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

def ask_existing_file(prompt):

    while True:

        path = input(prompt).strip().strip('"')

        if os.path.isfile(path):
            return path

        print("\nERROR: File not found.")
        print("Please enter a valid file path.\n")


def ask_integer(prompt, default):

    while True:

        value = input(
            f"{prompt} [{default}]: "
        ).strip()

        if value == "":
            return default

        try:

            value = int(value)

            if value > 0:
                return value

        except ValueError:
            pass

        print("Please enter a positive integer.")


def ask_float(prompt, default):

    while True:

        value = input(
            f"{prompt} [{default}]: "
        ).strip()

        if value == "":
            return default

        try:

            value = float(value)

            if 0 <= value <= 100:
                return value

        except ValueError:
            pass

        print("Please enter a number between 0 and 100.")


def ask_stage(prompt):

    valid_stages = {
        "larva": "Larva",
        "larval": "Larva",
        "third-instar": "Larva",
        "third instar": "Larva",
        "adult": "Adult",
        "pupa": "Pupa",
        "egg": "Egg",
    }

    while True:

        value = input(
            prompt
        ).strip().lower()

        if value in valid_stages:
            return valid_stages[value]

        print(
            "\nPlease enter one of:"
        )

        print(
            "Larva, Adult, Pupa, Egg\n"
        )


def clean_gene_id(value):

    value = str(value).strip()

    # Remove transcript suffix:
    # g45.t1 -> g45
    # g45.t2 -> g45
    value = re.sub(
        r"\.t\d+$",
        "",
        value,
        flags=re.IGNORECASE
    )

    return value


# ==========================================================
# INPUT FILES
# ==========================================================

print("\n" + "=" * 78)
print("INPUT FILES")
print("=" * 78)


candidate_file = ask_existing_file(
    "Top-500 candidate TSV path: "
)


print(
    "\nHow many RNA quant.sf files do you want to use?"
)

print(
    "You may provide 1 file or multiple files."
)

print(
    "At least one Larva sample is required."
)


rna_file_count = ask_integer(
    "Number of RNA files",
    1
)


rna_inputs = []


for i in range(
    1,
    rna_file_count + 1
):

    print(
        f"\n--- RNA SAMPLE {i} ---"
    )

    rna_path = ask_existing_file(
        "quant.sf path: "
    )

    stage = ask_stage(
        "Stage "
        "(Larva/Adult/Pupa/Egg): "
    )

    rna_inputs.append({
        "file": rna_path,
        "stage": stage
    })


# ==========================================================
# CHECK FOR LARVAL SAMPLE
# ==========================================================

larval_inputs = [
    x
    for x in rna_inputs
    if x["stage"] == "Larva"
]


if not larval_inputs:

    raise ValueError(
        "\nERROR: At least one Larva RNA sample "
        "is required because this pipeline prioritizes "
        "larval-stage targets."
    )


# ==========================================================
# DETERMINE RNA ANALYSIS MODE
# ==========================================================

non_larval_inputs = [
    x
    for x in rna_inputs
    if x["stage"] != "Larva"
]


if non_larval_inputs:

    RNA_MODE = "MULTI_STAGE"

    print(
        "\nRNA analysis mode: MULTI-STAGE"
    )

    print(
        "Larval expression, larval specificity, "
        "and stage dominance will be calculated."
    )

else:

    RNA_MODE = "LARVA_ONLY"

    print(
        "\nRNA analysis mode: LARVA-ONLY"
    )

    print(
        "Only larval expression will be used "
        "for RNA prioritization."
    )

    print(
        "Missing Adult/Pupa/Egg data will NOT "
        "be treated as zero expression."
    )


# ==========================================================
# OUTPUT SETTINGS
# ==========================================================

print("\n" + "=" * 78)
print("OUTPUT SETTINGS")
print("=" * 78)


output_dir = str(
    Path(candidate_file).parent
)


output_basename = input(
    "Output file name: "
).strip()


if output_basename == "":
    output_basename = "metisa_plana_rna"


top_n = ask_integer(
    "Number of candidates to retain",
    100
)


rna_weight = ask_float(
    "RNA evidence weight (%)",
    15.0
)


annotation_weight = (
    100.0 - rna_weight
)


# ==========================================================
# OUTPUT PATHS
# ==========================================================

top_file = os.path.join(
    output_dir,
    output_basename + "_top.tsv"
)


all_file = os.path.join(
    output_dir,
    output_basename + "_all500.tsv"
)


summary_file = os.path.join(
    output_dir,
    output_basename + "_summary.txt"
)


excel_file = os.path.join(
    output_dir,
    output_basename + ".xlsx"
)


# ==========================================================
# DISPLAY SETTINGS
# ==========================================================

print("\n" + "=" * 78)
print("SETTINGS")
print("=" * 78)


print(
    f"Candidate file:       {candidate_file}"
)


print(
    f"RNA files:            {len(rna_inputs)}"
)


for i, item in enumerate(
    rna_inputs,
    start=1
):

    print(
        f"  RNA {i}: "
        f"{item['stage']} — "
        f"{item['file']}"
    )


print(
    f"RNA analysis mode:    {RNA_MODE}"
)


print(
    f"Output directory:     {output_dir}"
)


print(
    f"Output basename:      {output_basename}"
)


print(
    f"Candidates retained:  {top_n}"
)


print(
    f"RNA weight:           {rna_weight:.1f}%"
)


print(
    f"Annotation weight:    "
    f"{annotation_weight:.1f}%"
)


# ==========================================================
# READ CANDIDATES
# ==========================================================

print("\n" + "=" * 78)
print("READING TOP-500 CANDIDATES")
print("=" * 78)


candidates = pd.read_csv(
    candidate_file,
    sep="\t"
)


print(
    f"Rows loaded: "
    f"{len(candidates):,}"
)


if len(candidates) > 500:

    print(
        "Input contains more than 500 rows."
    )

    print(
        "Only the first 500 candidates "
        "will be used."
    )

    candidates = (
        candidates
        .head(500)
        .copy()
    )


elif len(candidates) < 500:

    print(
        "WARNING: Input contains fewer "
        "than 500 candidates."
    )


# ==========================================================
# IDENTIFY CANDIDATE ID
# ==========================================================

gene_column = candidates.columns[0]


print(
    f"Candidate ID column: "
    f"{gene_column}"
)


if "gene_id" in candidates.columns:

    candidates["gene_id"] = (
        candidates["gene_id"]
        .apply(clean_gene_id)
    )

else:

    candidates["gene_id"] = (
        candidates[gene_column]
        .apply(clean_gene_id)
    )


# ==========================================================
# CHECK FINAL SCORE
# ==========================================================

if "final_score" not in candidates.columns:

    raise ValueError(
        "\nERROR: 'final_score' column "
        "was not found in the candidate file."
    )


# ==========================================================
# READ ALL RNA FILES
# ==========================================================

print("\n" + "=" * 78)
print("READING RNA QUANTIFICATION")
print("=" * 78)


rna_tables = []


for index, item in enumerate(
    rna_inputs,
    start=1
):

    path = item["file"]
    stage = item["stage"]


    print(
        f"\nReading RNA {index}: "
        f"{stage}"
    )


    rna = pd.read_csv(
        path,
        sep="\t"
    )


    required_columns = {
        "Name",
        "TPM",
        "NumReads"
    }


    missing = (
        required_columns
        - set(rna.columns)
    )


    if missing:

        raise ValueError(
            f"\nERROR: RNA file is missing "
            f"required columns:\n"
            f"{path}\n"
            +
            ", ".join(
                sorted(missing)
            )
        )


    print(
        f"Records: {len(rna):,}"
    )


    # ------------------------------------------------------
    # CLEAN IDS
    # ------------------------------------------------------

    rna["gene_id"] = (
        rna["Name"]
        .apply(clean_gene_id)
    )


    rna["TPM"] = pd.to_numeric(
        rna["TPM"],
        errors="coerce"
    ).fillna(0)


    rna["NumReads"] = pd.to_numeric(
        rna["NumReads"],
        errors="coerce"
    ).fillna(0)


    # ------------------------------------------------------
    # KEEP REQUIRED DATA
    # ------------------------------------------------------

    rna_small = rna[
        [
            "gene_id",
            "TPM",
            "NumReads"
        ]
    ].copy()


    # ------------------------------------------------------
    # HANDLE DUPLICATE GENE IDS
    # ------------------------------------------------------

    rna_small = (
        rna_small
        .groupby(
            "gene_id",
            as_index=False
        )
        .agg({
            "TPM": "sum",
            "NumReads": "sum"
        })
    )


    # ------------------------------------------------------
    # RENAME SAMPLE-SPECIFIC COLUMNS
    # ------------------------------------------------------

    sample_number = index


    rna_small.rename(
        columns={
            "TPM":
                f"TPM_sample_{sample_number}",
            "NumReads":
                f"NumReads_sample_{sample_number}"
        },
        inplace=True
    )


    rna_small[
        f"stage_sample_{sample_number}"
    ] = stage


    rna_tables.append(
        rna_small
    )


# ==========================================================
# MERGE RNA SAMPLES
# ==========================================================

print("\n" + "=" * 78)
print("COMBINING RNA SAMPLES")
print("=" * 78)


rna_combined = rna_tables[0].copy()


for table in rna_tables[1:]:

    rna_combined = rna_combined.merge(
        table,
        on="gene_id",
        how="outer"
    )


print(
    f"Combined RNA genes: "
    f"{len(rna_combined):,}"
)


# ==========================================================
# FILL ONLY SAMPLE-SPECIFIC NUMERIC COLUMNS
# ==========================================================

sample_tpm_columns = [
    c
    for c in rna_combined.columns
    if c.startswith("TPM_sample_")
]


sample_read_columns = [
    c
    for c in rna_combined.columns
    if c.startswith("NumReads_sample_")
]


for column in (
    sample_tpm_columns
    +
    sample_read_columns
):

    rna_combined[column] = (
        pd.to_numeric(
            rna_combined[column],
            errors="coerce"
        )
        .fillna(0)
    )


# ==========================================================
# CALCULATE LIFE-STAGE EXPRESSION
# ==========================================================

print("\n" + "=" * 78)
print("CALCULATING LIFE-STAGE EXPRESSION")
print("=" * 78)


stage_sample_columns = {}


for index, item in enumerate(
    rna_inputs,
    start=1
):

    stage = item["stage"]

    tpm_column = (
        f"TPM_sample_{index}"
    )


    stage_sample_columns.setdefault(
        stage,
        []
    ).append(
        tpm_column
    )


# ==========================================================
# CALCULATE MEAN TPM PER STAGE
# ==========================================================

for stage, columns in (
    stage_sample_columns.items()
):

    valid_columns = [
        c
        for c in columns
        if c in rna_combined.columns
    ]


    if not valid_columns:
        continue


    rna_combined[
        f"{stage}_TPM"
    ] = (
        rna_combined[
            valid_columns
        ]
        .mean(axis=1)
    )


# ==========================================================
# ONLY CREATE STAGES THAT WERE ACTUALLY PROVIDED
# ==========================================================

provided_stages = set(
    stage_sample_columns.keys()
)


print(
    "Stages provided: "
    +
    ", ".join(
        sorted(provided_stages)
    )
)


# ==========================================================
# MERGE WITH TOP-500
# ==========================================================

print("\n" + "=" * 78)
print("MATCHING CANDIDATES TO RNA")
print("=" * 78)


results = candidates.merge(
    rna_combined,
    on="gene_id",
    how="left"
)


# ==========================================================
# RNA MATCH STATUS
# ==========================================================

sample_tpm_columns = [
    c
    for c in rna_combined.columns
    if c.startswith("TPM_sample_")
]


results["RNA_matched"] = (
    results[sample_tpm_columns]
    .notna()
    .any(axis=1)
)


matched_count = (
    results["RNA_matched"]
    .sum()
)


unmatched_count = (
    len(results)
    - matched_count
)


match_rate = (
    matched_count
    /
    len(results)
    *
    100
)


print(
    f"Matched:    {matched_count:,}"
)


print(
    f"Unmatched:  {unmatched_count:,}"
)


print(
    f"Match rate: {match_rate:.2f}%"
)


# ==========================================================
# ENSURE LARVA TPM EXISTS
# ==========================================================

if "Larva_TPM" not in results.columns:

    raise ValueError(
        "\nERROR: Larva TPM could not be calculated."
    )


results["Larva_TPM"] = pd.to_numeric(
    results["Larva_TPM"],
    errors="coerce"
).fillna(0)


# ==========================================================
# LARVAL EXPRESSION
# ==========================================================

results["Larva_log2_TPM"] = np.log2(
    results["Larva_TPM"] + 1
)


# ==========================================================
# NORMALIZE LARVAL EXPRESSION
# ==========================================================

max_larva_log = (
    results["Larva_log2_TPM"]
    .max()
)


if max_larva_log > 0:

    results["Larva_expression_score"] = (

        results["Larva_log2_TPM"]
        /
        max_larva_log
        *
        100
    )

else:

    results[
        "Larva_expression_score"
    ] = 0.0


# ==========================================================
# MULTI-STAGE ANALYSIS
# ==========================================================

if RNA_MODE == "MULTI_STAGE":

    # ------------------------------------------------------
    # CREATE NON-LARVAL STAGE MEAN
    # ONLY USING ACTUALLY PROVIDED STAGES
    # ------------------------------------------------------

    non_larval_stages = [
        stage
        for stage in [
            "Adult",
            "Pupa",
            "Egg"
        ]
        if stage in provided_stages
    ]


    non_larval_columns = [
        f"{stage}_TPM"
        for stage in non_larval_stages
        if f"{stage}_TPM" in results.columns
    ]


    if non_larval_columns:

        results[
            "Non_larval_mean_TPM"
        ] = (
            results[
                non_larval_columns
            ]
            .mean(axis=1)
        )


        results[
            "Non_larval_log2_TPM"
        ] = np.log2(
            results[
                "Non_larval_mean_TPM"
            ]
            + 1
        )


        # --------------------------------------------------
        # LARVAL ENRICHMENT
        # --------------------------------------------------
        #
        # +1 pseudocount prevents division by zero.
        #
        # Example:
        #
        # Larva = 150
        # Other stages = 2
        #
        # -> strong enrichment
        #
        # Larva = 150
        # Other stages = 100
        #
        # -> weaker enrichment
        #

        results[
            "Larva_vs_nonLarva_ratio"
        ] = (

            (
                results["Larva_TPM"]
                + 1
            )

            /

            (
                results[
                    "Non_larval_mean_TPM"
                ]
                + 1
            )
        )


        results[
            "Larva_vs_nonLarva_ratio"
        ] = (
            results[
                "Larva_vs_nonLarva_ratio"
            ]
            .clip(
                lower=0,
                upper=100
            )
        )


        # --------------------------------------------------
        # FIXED SCALE SPECIFICITY SCORE
        # --------------------------------------------------
        #
        # 1x enrichment  -> 0
        # 2x enrichment  -> ~20
        # 4x enrichment  -> ~40
        # 8x enrichment  -> ~60
        # 16x enrichment -> ~80
        # 32x enrichment -> 100
        #

        results[
            "Larva_specificity_score"
        ] = (

            np.log2(
                results[
                    "Larva_vs_nonLarva_ratio"
                ]
            )
            .clip(
                lower=0,
                upper=5
            )
            /
            5
            *
            100
        )


        # --------------------------------------------------
        # LIFE-STAGE LOG EXPRESSION
        # --------------------------------------------------

        for stage in provided_stages:

            column = f"{stage}_TPM"

            if column not in results.columns:
                continue

            results[
                f"{stage}_log2_TPM"
            ] = np.log2(
                results[column] + 1
            )


        # --------------------------------------------------
        # STAGE DOMINANCE
        # --------------------------------------------------

        stage_log_columns = [
            f"{stage}_log2_TPM"
            for stage in provided_stages
            if f"{stage}_log2_TPM"
            in results.columns
        ]


        if stage_log_columns:

            results[
                "Highest_expression_stage"
            ] = (
                results[
                    stage_log_columns
                ]
                .idxmax(axis=1)
                .str.replace(
                    "_log2_TPM",
                    "",
                    regex=False
                )
            )


            results[
                "Larva_stage_dominant"
            ] = (
                results[
                    "Highest_expression_stage"
                ]
                == "Larva"
            )

        else:

            results[
                "Highest_expression_stage"
            ] = "Unknown"

            results[
                "Larva_stage_dominant"
            ] = False


        # --------------------------------------------------
        # MULTI-STAGE RNA SCORE
        # --------------------------------------------------

        results["RNA_score"] = (

            results[
                "Larva_expression_score"
            ]
            * 0.60

            +

            results[
                "Larva_specificity_score"
            ]
            * 0.40
        )


        # --------------------------------------------------
        # SMALL LARVAL DOMINANCE BONUS
        # --------------------------------------------------

        results["RNA_score"] = (

            results["RNA_score"]

            +

            np.where(
                results[
                    "Larva_stage_dominant"
                ],
                5.0,
                0.0
            )
        )


        results["RNA_score"] = (
            results["RNA_score"]
            .clip(
                lower=0,
                upper=100
            )
        )


else:

    # ======================================================
    # LARVA-ONLY MODE
    # ======================================================
    #
    # IMPORTANT:
    #
    # Adult/Pupa/Egg are NOT available.
    #
    # Therefore they are NOT assigned TPM = 0.
    #
    # RNA score is based ONLY on larval expression.
    #

    results[
        "Non_larval_mean_TPM"
    ] = np.nan


    results[
        "Non_larval_log2_TPM"
    ] = np.nan


    results[
        "Larva_vs_nonLarva_ratio"
    ] = np.nan


    results[
        "Larva_specificity_score"
    ] = np.nan


    results[
        "Highest_expression_stage"
    ] = "Larva_only_data"


    results[
        "Larva_stage_dominant"
    ] = np.nan


    results["RNA_score"] = (
        results[
            "Larva_expression_score"
        ]
    )


# ==========================================================
# EXPRESSION CATEGORY
# ==========================================================

def expression_category(tpm):

    if tpm >= 100:
        return "Very High"

    elif tpm >= 10:
        return "High"

    elif tpm >= 1:
        return "Moderate"

    elif tpm > 0:
        return "Low"

    else:
        return "Not detected"


results["Larva_expression"] = (
    results["Larva_TPM"]
    .apply(expression_category)
)


# ==========================================================
# COMBINE CODE 1 + RNA
# ==========================================================

results[
    "annotation_score_before_RNA"
] = results["final_score"]


results["final_score_RNA"] = (

    results["final_score"]
    *
    annotation_weight
    /
    100.0

    +

    results["RNA_score"]
    *
    rna_weight
    /
    100.0
)


# ==========================================================
# RANK
# ==========================================================

print("\n" + "=" * 78)
print("RANKING RNA-PRIORITIZED CANDIDATES")
print("=" * 78)


sort_columns = [
    "final_score_RNA",
    "RNA_score",
    "Larva_expression_score",
    "Larva_TPM"
]


if RNA_MODE == "MULTI_STAGE":

    sort_columns.insert(
        2,
        "Larva_specificity_score"
    )


# Additional biological tie-breakers

for column in [
    "final_score",
    "homology_score",
    "taxonomy_score",
    "best_identity_pct",
    "best_query_coverage_pct",
    "best_bitscore"
]:

    if column in results.columns:

        sort_columns.append(
            column
        )


results = results.sort_values(
    by=sort_columns,
    ascending=[
        False
    ] * len(sort_columns),
    na_position="last"
).reset_index(
    drop=True
)


results["combined_rank"] = (
    np.arange(
        len(results)
    )
    +
    1
)


# ==========================================================
# TOP N
# ==========================================================

top_results = (
    results
    .head(top_n)
    .copy()
)


# ==========================================================
# REORDER IMPORTANT COLUMNS
# ==========================================================

priority_columns = [

    "gene_id",

    "RNA_rank",

    "final_score_RNA",

    "annotation_score_before_RNA",

    "final_score",

    "RNA_score",

    "Larva_expression_score",

    "Larva_specificity_score",

    "Larva_TPM",

    "Adult_TPM",

    "Pupa_TPM",

    "Egg_TPM",

    "Non_larval_mean_TPM",

    "Larva_vs_nonLarva_ratio",

    "Larva_expression",

    "Highest_expression_stage",

    "Larva_stage_dominant",

    "RNA_matched"
]


priority_columns = [
    x
    for x in priority_columns
    if x in top_results.columns
]


remaining_columns = [
    x
    for x in top_results.columns
    if x not in priority_columns
]


top_results = top_results[
    priority_columns
    +
    remaining_columns
]


# ==========================================================
# RNA SUMMARY STATISTICS
# ==========================================================

very_high = (
    results["Larva_TPM"] >= 100
).sum()


high = (
    (results["Larva_TPM"] >= 10)
    &
    (results["Larva_TPM"] < 100)
).sum()


moderate = (
    (results["Larva_TPM"] >= 1)
    &
    (results["Larva_TPM"] < 10)
).sum()


low = (
    (results["Larva_TPM"] > 0)
    &
    (results["Larva_TPM"] < 1)
).sum()


not_detected = (
    results["Larva_TPM"] == 0
).sum()


if RNA_MODE == "MULTI_STAGE":

    larva_dominant_count = (
        results[
            "Larva_stage_dominant"
        ]
        .fillna(False)
        .sum()
    )

else:

    larva_dominant_count = np.nan


# ==========================================================
# WRITE TSV
# ==========================================================

print("\n" + "=" * 78)
print("WRITING OUTPUT")
print("=" * 78)


top_results.to_csv(
    top_file,
    sep="\t",
    index=False
)


results.to_csv(
    all_file,
    sep="\t",
    index=False
)


print(
    f"Created: {top_file}"
)


print(
    f"Created: {all_file}"
)


# ==========================================================
# SUMMARY FILE
# ==========================================================

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "METISA PLANA — RNA LIFE-STAGE "
        "PRIORITIZATION\n"
    )

    f.write(
        "=" * 60
        +
        "\n\n"
    )


    f.write(
        "RNA ANALYSIS MODE\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )

    f.write(
        f"{RNA_MODE}\n\n"
    )


    f.write(
        "RNA SAMPLES\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )


    for i, item in enumerate(
        rna_inputs,
        start=1
    ):

        f.write(
            f"RNA {i}: "
            f"{item['stage']}\n"
        )

        f.write(
            f"File: "
            f"{item['file']}\n\n"
        )


    f.write(
        "PIPELINE\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )

    f.write(
        "Input: Top-500 candidates "
        "from Code 1\n"
    )


    if RNA_MODE == "LARVA_ONLY":

        f.write(
            "RNA prioritization: "
            "Larval expression only\n"
        )

        f.write(
            "Specificity: Not calculated "
            "(non-larval data unavailable)\n\n"
        )

    else:

        f.write(
            "RNA prioritization: "
            "Larval expression + "
            "larval specificity + "
            "stage dominance\n\n"
        )


    f.write(
        "INPUT STATISTICS\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )

    f.write(
        f"Input candidates: "
        f"{len(candidates)}\n"
    )

    f.write(
        f"RNA files: "
        f"{len(rna_inputs)}\n"
    )

    f.write(
        f"RNA genes combined: "
        f"{len(rna_combined):,}\n"
    )

    f.write(
        f"Matched candidates: "
        f"{matched_count}\n"
    )

    f.write(
        f"Unmatched candidates: "
        f"{unmatched_count}\n"
    )

    f.write(
        f"Match rate: "
        f"{match_rate:.2f}%\n\n"
    )


    f.write(
        "LARVAL EXPRESSION\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )

    f.write(
        f"Very High (>=100 TPM): "
        f"{very_high}\n"
    )

    f.write(
        f"High (10-<100 TPM): "
        f"{high}\n"
    )

    f.write(
        f"Moderate (1-<10 TPM): "
        f"{moderate}\n"
    )

    f.write(
        f"Low (>0-<1 TPM): "
        f"{low}\n"
    )

    f.write(
        f"Not detected (0 TPM): "
        f"{not_detected}\n"
    )


    if RNA_MODE == "MULTI_STAGE":

        f.write(
            f"Larva highest-expression stage: "
            f"{larva_dominant_count}\n\n"
        )

    else:

        f.write(
            "Larva highest-expression stage: "
            "Not assessed\n\n"
        )


    f.write(
        "SCORING\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )

    f.write(
        f"Annotation weight: "
        f"{annotation_weight:.1f}%\n"
    )

    f.write(
        f"RNA weight: "
        f"{rna_weight:.1f}%\n"
    )


    if RNA_MODE == "LARVA_ONLY":

        f.write(
            "RNA score composition:\n"
        )

        f.write(
            "  Larval expression: 100%\n"
        )

    else:

        f.write(
            "RNA score composition:\n"
        )

        f.write(
            "  Larval expression: 60%\n"
        )

        f.write(
            "  Larval specificity: 40%\n"
        )

        f.write(
            "  Larval stage dominance bonus: "
            "+5 points\n"
        )


    f.write(
        "\nOUTPUT\n"
    )

    f.write(
        "-" * 40
        +
        "\n"
    )

    f.write(
        f"Final candidates retained: "
        f"{len(top_results)}\n"
    )


print(
    f"Created: {summary_file}"
)


# ==========================================================
# EXCEL
# ==========================================================

try:

    with pd.ExcelWriter(
        excel_file,
        engine="openpyxl"
    ) as writer:


        # --------------------------------------------------
        # TOP CANDIDATES
        # --------------------------------------------------

        top_results.to_excel(
            writer,
            sheet_name="Top_Candidates",
            index=False
        )


        # --------------------------------------------------
        # ALL CANDIDATES
        # --------------------------------------------------

        results.to_excel(
            writer,
            sheet_name="All_500",
            index=False
        )


        # --------------------------------------------------
        # RNA STAGE DATA
        # --------------------------------------------------

        stage_summary_columns = [

            "gene_id",

            "Larva_TPM",
            "Adult_TPM",
            "Pupa_TPM",
            "Egg_TPM",

            "Non_larval_mean_TPM",

            "Larva_expression_score",

            "Larva_specificity_score",

            "Larva_vs_nonLarva_ratio",

            "RNA_score",

            "Highest_expression_stage",

            "Larva_stage_dominant"

        ]


        stage_summary_columns = [
            x
            for x in stage_summary_columns
            if x in results.columns
        ]


        results[
            stage_summary_columns
        ].to_excel(
            writer,
            sheet_name="RNA_Stage_Expression",
            index=False
        )


        # --------------------------------------------------
        # SUMMARY
        # --------------------------------------------------

        summary_table = pd.DataFrame({

            "Metric": [

                "RNA analysis mode",

                "Input candidates",

                "RNA files",

                "RNA genes combined",

                "Matched candidates",

                "Unmatched candidates",

                "Match rate (%)",

                "Very High Larval TPM",

                "High Larval TPM",

                "Moderate Larval TPM",

                "Low Larval TPM",

                "Not detected",

                "Larva highest-expression stage",

                "Annotation weight (%)",

                "RNA weight (%)",

                "Final candidates"

            ],

            "Value": [

                RNA_MODE,

                len(candidates),

                len(rna_inputs),

                len(rna_combined),

                matched_count,

                unmatched_count,

                match_rate,

                very_high,

                high,

                moderate,

                low,

                not_detected,

                (
                    larva_dominant_count
                    if RNA_MODE == "MULTI_STAGE"
                    else "Not assessed"
                ),

                annotation_weight,

                rna_weight,

                len(top_results)

            ]

        })


        summary_table.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )


    print(
        f"Created: {excel_file}"
    )


except Exception as e:

    print(
        "\nWARNING: Excel output "
        "could not be created."
    )

    print(e)


# ==========================================================
# SHOW TOP 20
# ==========================================================

print("\n" + "=" * 78)
print("TOP 20 AFTER RNA PRIORITIZATION")
print("=" * 78)


display_columns = [

    "gene_id",

    "combined_rank",

    "final_score_RNA",

    "final_score",

    "RNA_score",

    "Larva_expression_score",

    "Larva_specificity_score",

    "Larva_TPM",

    "Adult_TPM",

    "Pupa_TPM",

    "Egg_TPM",

    "Highest_expression_stage",

    "Larva_expression"

]


for column in [

    "best_hit_species",

    "functional_description",

    "target_class",

    "decision"

]:

    if column in top_results.columns:

        display_columns.append(
            column
        )


print(
    top_results[
        [
            c
            for c in display_columns
            if c in top_results.columns
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ==========================================================
# FINAL
# ==========================================================

print("\n" + "=" * 78)
print("DONE")
print("=" * 78)


print(
    f"Top {len(top_results)} "
    "candidates retained."
)


print(
    "\nRNA prioritization mode:"
)


if RNA_MODE == "LARVA_ONLY":

    print(
        "  Larva-only expression ranking"
    )

    print(
        "  High larval expression → higher priority"
    )

    print(
        "  Low/no larval expression → lower priority"
    )

else:

    print(
        "  1. Larval expression"
    )

    print(
        "  2. Larval specificity "
        "relative to available non-larval stages"
    )

    print(
        "  3. Whether larva is the "
        "highest-expression stage"
    )


print(
    "\nNext step: inspect the "
    "RNA-prioritized candidates "
    "before MAFFT/phylogenetic analysis."
)


METISA PLANA — RNA LIFE-STAGE EXPRESSION PRIORITIZATION

INPUT FILES

How many RNA quant.sf files do you want to use?
You may provide 1 file or multiple files.
At least one Larva sample is required.

--- RNA SAMPLE 1 ---

--- RNA SAMPLE 2 ---

--- RNA SAMPLE 3 ---

--- RNA SAMPLE 4 ---

--- RNA SAMPLE 5 ---

RNA analysis mode: MULTI-STAGE
Larval expression, larval specificity, and stage dominance will be calculated.

OUTPUT SETTINGS

SETTINGS
Candidate file:       C:\metp\output\testing_raw_blast_screening\testing_mining_candidates.tsv
RNA files:            5
  RNA 1: Pupa — C:\metp\rna_quant\SRR8905601\quant.sf
  RNA 2: Adult — C:\metp\rna_quant\SRR8905603\quant.sf
  RNA 3: Larva — C:\metp\rna_quant\SRR8905604\quant.sf
  RNA 4: Larva — C:\metp\rna_quant\SRR8905605\quant.sf
  RNA 5: Egg — C:\metp\rna_quant\SRR8905607\quant.sf
RNA analysis mode:    MULTI_STAGE
Output directory:     C:\metp\output\testing_raw_blast_screening
Output basename:      testing_rna
Candidates retained:  100
RNA


# 07. Phylogenetic Analysis

Phylogenetic analysis is performed for selected candidate proteins to examine their evolutionary relationships with homologous proteins.

Multiple sequence alignment is performed before phylogenetic reconstruction.

The resulting phylogenetic relationships provide additional evidence for:

- conservation of the candidate protein
- evolutionary relationships with homologues
- possible lineage-specific patterns
- confidence in functional interpretation

This analysis provides an evolutionary layer of evidence for candidate evaluation.


In [8]:



DEFAULT_HOMOLOGS_PER_TREE = 10
MIN_SEQUENCE_LENGTH = 30

MAFFT_TIMEOUT = 1800      # 30 minutes
TRIMAL_TIMEOUT = 1800     # 30 minutes
IQTREE_TIMEOUT = 7200     # 2 hours
BLASTDBCMD_TIMEOUT = 600   # 10 minutes

UFBBOOT_REPLICATES = 1000
SH_ALRT_REPLICATES = 1000


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)


def clean_path(path_string: str) -> Path:
    value = path_string.strip().strip('"')

    # Expand ~
    if value.startswith("~"):
        value = str(Path.home()) + value[1:]

    match = re.match(r"^/([a-zA-Z])/(.*)$", value)

    if match:
        drive = match.group(1).upper()
        remainder = match.group(2)
        value = f"{drive}:/{remainder}"

    return Path(value)


def ask_existing_file(prompt: str) -> Path:
    while True:
        value = input(prompt).strip()

        if not value:
            print("ERROR: Path cannot be empty.")
            continue

        path = clean_path(value)

        if not path.is_file():
            print("\nERROR: File not found:")
            print(f"  {path}\n")
            continue

        return path

def ask_output_folder() -> Path:
    """Ask for output folder and create it if necessary."""

    while True:
        value = input(
            "Path to OUTPUT folder (will be created if necessary): "
        ).strip()

        if not value:
            print("ERROR: Output folder path cannot be empty.")
            continue

        path = clean_path(value)

        try:
            path.mkdir(parents=True, exist_ok=True)
            return path
        except Exception as e:
            print(f"\nERROR: Cannot create output folder:\n  {e}\n")


def ask_positive_integer(prompt: str, default: int) -> int:
    """
    Ask for a positive integer.
    """

    while True:
        value = input(f"{prompt} [{default}]: ").strip()

        if not value:
            return default

        try:
            number = int(value)

            if number < 1:
                raise ValueError

            return number

        except ValueError:
            print("ERROR: Please enter a positive integer.")

def windows_to_gitbash_path(path: Path) -> str:
    """
    Convert a Windows path to Git Bash format.
    """

    path = Path(path)
    drive = path.drive

    if not drive:
        return path.as_posix()

    drive_letter = drive[0].lower()
    remainder = path.as_posix()[2:]

    return f"/{drive_letter}{remainder}"

def find_git_bash() -> Path | None:
    """Find Git Bash from the system PATH."""

    bash = shutil.which("bash")

    if bash and Path(bash).is_file():
        return Path(bash)

    return None
    
def test_executable(executable: Path, version_arguments=None) -> bool:
    args = version_arguments or ["--version"]

    print("\n" + "=" * 70)
    print("EXECUTABLE DIAGNOSTIC")
    print("=" * 70)

    # 1 — PATH
    print(f"\n[1/5] Checking path:\n      {executable}")
    path = executable.absolute()

    if not path.is_file():
        print("      FAILED: File does not exist.")
        return False

    print("      OK")

    # 2 — FILE
    try:
        size = path.stat().st_size
        print(
            f"\n[2/5] File check:"
            f"\n      Name: {path.name}"
            f"\n      Size: {size:,} bytes"
            f"\n      OK"
        )
    except Exception as e:
        print(f"      FAILED: {e}")
        return False

    # 3 — EXECUTE
    command = [str(path)] + args
    print(f"\n[3/5] Running:\n      {command}")

    try:
        result = subprocess.run(
            command, capture_output=True, text=True, timeout=30
        )

    except FileNotFoundError:
        print("      FAILED: Executable could not be launched.")
        return False

    except PermissionError:
        print("      FAILED: Permission denied.")
        return False

    except subprocess.TimeoutExpired:
        print("      FAILED: Timeout.")
        return False

    except Exception as e:
        print(f"      FAILED: {type(e).__name__}: {e}")
        return False

    # 4 — RESULT
    print(f"\n[4/5] Result: {result.returncode}")

    if result.stdout.strip():
        print("      STDOUT:")
        print("      " + result.stdout.strip().replace("\n", "\n      "))

    if result.stderr.strip():
        print("      STDERR:")
        print("      " + result.stderr.strip().replace("\n", "\n      "))

    # 5 — DIAGNOSIS
    print("\n[5/5] FINAL DIAGNOSIS")

    if result.returncode == 0:
        print("      EXECUTABLE WORKS.")
        print("=" * 70)
        return True

    error = result.stderr.lower()

    if "usage" in error or "version" in error:
        print("      EXECUTABLE LAUNCHED.")
        print("      Argument/version check returned an error.")
        print("=" * 70)
        return True

    print("      EXECUTABLE FAILED.")
    print("=" * 70)
    return False

def test_trimal_executable(
    trimal: Path
) -> tuple[bool, Path | None]:
    """Test Windows trimAl through Git Bash."""

    print("\nTesting trimAl through Git Bash...")
    print(f"  Executable: {trimal}")

    if not trimal.is_file():
        print("  trimAl: FAILED — file does not exist.")
        return False, None

    git_bash = find_git_bash()

    if git_bash is None:
        print("  trimAl: FAILED — Git Bash not found.")
        return False, None

    print(f"  Git Bash: {git_bash}")

    trimal_dir = windows_to_gitbash_path(trimal.parent)
    command = f'cd "{trimal_dir}" && ./trimal.exe --version'

    try:
        result = subprocess.run(
            [str(git_bash), "-lc", command],
            capture_output=True,
            text=True,
            timeout=30
        )

    except subprocess.TimeoutExpired:
        print("  trimAl: FAILED — timeout.")
        return False, None

    except Exception as e:
        print(f"  trimAl: FAILED — {e}")
        return False, None

    output = (
        (result.stdout or "") +
        (result.stderr or "")
    ).strip()

    if result.returncode == 0:
        print("  trimAl: WORKING")

        if output:
            print(output)

        return True, git_bash

    print(f"  trimAl: FAILED — return code {result.returncode}")

    if output:
        print(output)

    return False, None


def test_trimal_wsl(wsl_path: str) -> bool:
    print("=" * 70)
    print("WSL trimAl EXECUTABLE DIAGNOSTIC")
    print("=" * 70)

    print("\n[1/4] Checking WSL and trimAl file...")

    try:
        result = subprocess.run(
            [
                "wsl.exe",
                "sh",
                "-c",
                f'test -f "{wsl_path}" && test -x "{wsl_path}"'
            ],
            capture_output=True,
            text=True,
            timeout=30
        )

        if result.returncode != 0:
            print("      FAILED: WSL or trimAl file check.")
            return False

        print("      WSL: WORKING")
        print("      trimAl: EXISTS + EXECUTABLE")

    except Exception as e:
        print(f"      FAILED: {e}")
        return False

    print("\n[2/4] Checking trimAl dependencies...")

    result = subprocess.run(
        [
            "wsl.exe",
            "sh",
            "-c",
            f'ldd "{wsl_path}" 2>&1'
        ],
        capture_output=True,
        text=True,
        timeout=30
    )

    if "not found" in result.stdout.lower():
        print("      FAILED: Missing dependency.")
        print(result.stdout)
        return False

    print("      Dependencies: OK")

    print("\n[3/4] Checking WSL environment...")

    result = subprocess.run(
        [
            "wsl.exe",
            "sh",
            "-c",
            "uname -m"
        ],
        capture_output=True,
        text=True,
        timeout=30
    )

    print(f"      Architecture: {result.stdout.strip()}")

    print("\n[4/4] Running trimAl...")

    try:
        result = subprocess.run(
            [
                "wsl.exe",
                "sh",
                "-c",
                f'"{wsl_path}" -h'
            ],
            capture_output=True,
            text=True,
            timeout=30
        )

    except subprocess.TimeoutExpired:
        print("      FAILED: trimAl timed out.")
        return False

    output = (
        (result.stdout or "") +
        (result.stderr or "")
    ).strip()

    if "trimal" not in output.lower():
        print("      FAILED: No recognizable trimAl output.")
        print(output[:1000])
        return False

    print("      trimAl: WORKING")
    print(f"      Return code: {result.returncode}")

    print("\n" + "=" * 70)
    print("FINAL DIAGNOSIS")
    print("trimAl WORKS INSIDE WSL.")
    print("=" * 70)

    return True

def ask_executable(
    tool_name: str
) -> tuple[Path, Path | None]:

    while True:

        value = input(
            f"Path to {tool_name} executable: "
        ).strip()

        if not value:

            print(
                "ERROR: Path cannot be empty."
            )

            continue

        path = clean_path(value)

        if tool_name.lower() == "trimal":

            original_value = value.strip().strip('"')

            if original_value.startswith("/"):

                print()
                print("=" * 70)
                print("WSL trimAl path detected")
                print("=" * 70)

                print()
                print("Path:")
                print(f"  {original_value}")

                print()
                print("Testing trimAl INSIDE WSL...")
                print()

                success = test_trimal_wsl(original_value)

                if success:

                    print()
                    print("=" * 70)
                    print("FINAL DIAGNOSIS")
                    print()
                    print("        trimAl WORKS INSIDE WSL.")
                    print("=" * 70)

                    return Path(original_value), None

                print()
                print("=" * 70)
                print("FINAL DIAGNOSIS")
                print()
                print("        trimAl FAILED INSIDE WSL.")
                print("=" * 70)

                print()
                print(
                    "The supplied WSL trimAl executable "
                    "could not be run."
                )

                print(
                    "Please provide another path."
                )

                print()

                continue

        if not path.is_file():
            print("\nERROR: Executable not found:")
            print(f"  {path}\n")
            continue

        if tool_name.lower() == "trimal":
            success, git_bash = test_trimal_executable(path)

            if success:
                return path, git_bash

            print("\nThe supplied trimAl executable could not be run.")
            print("Please provide another path.\n")
            continue

        print(f"\nTesting {tool_name}...")
        print(f"  Executable: {path}")

        if tool_name.lower() == "blastdbcmd":
            test_ok = test_executable(path, ["-version"])
        else:
            test_ok = test_executable(path)

        if test_ok:
            print(f"  {tool_name}: WORKING")
            return path, None

        print(f"  {tool_name}: FAILED")
        print(f"\nThe supplied {tool_name} executable could not be run.")
        print("Please provide another path.\n")

def detect_blast_database(db_root: Path):
    print("\nChecking BLAST protein database...")
    print(f"  Database root: {db_root}")

    single_pin = Path(str(db_root) + ".pin")
    single_phr = Path(str(db_root) + ".phr")
    single_psq = Path(str(db_root) + ".psq")

    if single_pin.is_file() and single_phr.is_file() and single_psq.is_file():
        print("\n  COMPLETE SINGLE-VOLUME DATABASE FOUND")
        print(f"    {single_pin.name}")
        print(f"    {single_phr.name}")
        print(f"    {single_psq.name}")

        return {
            "type": "single",
            "volumes": [db_root]
        }

    parent = db_root.parent
    prefix = db_root.name

    volumes = []

    if parent.exists():
        pattern = re.compile(
            re.escape(prefix) + r"\.(\d+)\.pin$",
            re.IGNORECASE
        )

        for pin_file in parent.iterdir():
            match = pattern.match(pin_file.name)

            if not match:
                continue

            volume_number = int(match.group(1))

            volume_root = (
                parent
                / f"{prefix}.{volume_number:02d}"
            )

            phr = Path(str(volume_root) + ".phr")
            psq = Path(str(volume_root) + ".psq")

            if phr.is_file() and psq.is_file():
                volumes.append(
                    (volume_number, volume_root)
                )

    volumes.sort(key=lambda x: x[0])

    if volumes:
        print()
        print("  SPLIT BLAST PROTEIN DATABASE DETECTED")
        print(f"  Number of volumes: {len(volumes)}")

        for number, root in volumes:
            print(
                f"    Volume {number:02d}: "
                f"{root.name}"
            )

        print()

        return {
            "type": "split",
            "volumes": [
                root
                for number, root in volumes
            ]
        }

    possible_files = (
        list(parent.glob(prefix + ".*"))
        if parent.exists()
        else []
    )

    if possible_files:
        print()
        print(
            "WARNING: Database files were found "
            "but the database appears incomplete."
        )

        print()

        for file in sorted(possible_files):
            print(f"  {file.name}")

    else:
        print()
        print("No BLAST database files found.")

    return None

def ask_blast_database():

    while True:

        value = input(
            "Path to Uniprot database "
            "(without .pin/.phr/.psq): "
        ).strip()

        if not value:

            print(
                "ERROR: Database path "
                "cannot be empty."
            )

            continue

        db_root = clean_path(value)

        database_info = (
            detect_blast_database(
                db_root
            )
        )

        if database_info is not None:

            return (
                db_root,
                database_info
            )

        print()
        print(
            "ERROR: Could not detect a "
            "complete protein BLAST database."
        )

        print()

def load_fasta(fasta_file: Path) -> dict:
    """
    Load FASTA into:

        {header: sequence}
    """

    sequences = {}
    current_header = None
    current_sequence = []

    with open(
        fasta_file,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as handle:
        for raw_line in handle:
            line = raw_line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if current_header is not None:
                    sequences[current_header] = "".join(current_sequence)

                current_header = line[1:].strip()
                current_sequence = []

            else:
                current_sequence.append(line)

        if current_header is not None:
            sequences[current_header] = "".join(current_sequence)

    return sequences


def get_fasta_id(header: str) -> str:
    return header.split()[0]


def clean_protein_sequence(sequence: str) -> str:
    """
    Clean protein sequence.

    Removes whitespace and terminal *.
    """

    sequence = (
        str(sequence)
        .replace(" ", "")
        .replace("\r", "")
        .replace("\n", "")
        .strip()
        .rstrip("*")
        .upper()
    )

    return sequence


def build_fasta_lookup(fasta_file: Path):
    """
    Build lookup using both:

        full header
        first FASTA ID
    """

    raw = load_fasta(fasta_file)
    lookup = {}

    for header, sequence in raw.items():
        fasta_id = get_fasta_id(header)
        sequence = clean_protein_sequence(sequence)

        lookup[fasta_id] = sequence
        lookup[header] = sequence

    logger.info(
        f"Loaded {len(raw)} Metisa "
        f"protein sequences"
    )

    return lookup

def load_blast_tsv(blast_tsv: Path) -> pd.DataFrame:
    logger.info(f"Loading BLAST TSV: {blast_tsv}")

    df = pd.read_csv(blast_tsv, sep="\t", dtype=str)

    df.columns = [str(c).strip() for c in df.columns]

    required = ["qseqid", "sseqid"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            "BLAST TSV is missing required "
            f"columns: {missing}"
        )

    for column in ["pident", "length", "qlen", "slen", "qcovs", "evalue", "bitscore"]:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    df["qseqid"] = (
        df["qseqid"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["sseqid"] = (
        df["sseqid"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df = df[
        (df["qseqid"] != "")
        &
        (df["sseqid"] != "")
    ].copy()

    logger.info(
        f"Loaded {len(df)} BLAST rows"
    )

    logger.info(
        f"Unique Metisa proteins: "
        f"{df['qseqid'].nunique()}"
    )

    return df

def extract_accession(
    sseqid: str
) -> str:

    if pd.isna(sseqid):

        return ""

    value = str(
        sseqid
    ).strip()

    if not value:

        return ""

    if "|" in value:

        parts = value.split("|")

        if len(parts) >= 2:

            accession = (
                parts[1]
                .strip()
            )

            if accession:

                return accession

    return value.split()[0]

def select_top_homologs(
    blast_df: pd.DataFrame,
    max_hits: int = DEFAULT_HOMOLOGS_PER_TREE
) -> pd.DataFrame:

    df = blast_df.copy()

    df["accession"] = (
        df["sseqid"]
        .apply(extract_accession)
    )

    df = df[
        df["accession"].str.len() > 0
    ].copy()

    df = df.drop_duplicates(
        subset=[
            "qseqid",
            "accession"
        ],
        keep="first"
    )

    sort_columns = []
    ascending = []

    if "bitscore" in df.columns:

        sort_columns.append(
            "bitscore"
        )

        ascending.append(False)

    if "qcovs" in df.columns:

        sort_columns.append(
            "qcovs"
        )

        ascending.append(False)

    if "pident" in df.columns:

        sort_columns.append(
            "pident"
        )

        ascending.append(False)

    if "evalue" in df.columns:

        sort_columns.append(
            "evalue"
        )

        ascending.append(True)

    if sort_columns:

        df = df.sort_values(
            sort_columns,
            ascending=ascending,
            na_position="last"
        )

    selected = (
        df
        .groupby(
            "qseqid",
            sort=False,
            group_keys=False
        )
        .head(max_hits)
        .reset_index(drop=True)
    )

    selected["homolog_rank"] = (
        selected
        .groupby("qseqid")
        .cumcount()
        + 1
    )

    return selected

def parse_fasta_records(fasta_text: str):
    """
    Parse FASTA text into:

        [(header, sequence), ...]
    """

    records = []
    current_header = None
    current_sequence = []

    for line in fasta_text.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if current_header is not None:
                sequence = clean_protein_sequence(
                    "".join(current_sequence)
                )

                records.append(
                    (current_header, sequence)
                )

            current_header = line[1:].strip()
            current_sequence = []

        else:
            current_sequence.append(line)

    if current_header is not None:
        sequence = clean_protein_sequence(
            "".join(current_sequence)
        )

        records.append(
            (current_header, sequence)
        )

    return records

def run_blastdbcmd_batch(
    blastdbcmd: Path,
    db_root: Path,
    accessions: list[str],
    output_fasta: Path
) -> bool:

    if not accessions:

        return False

    accession_file = (
        output_fasta.parent
        /
        (
            output_fasta.stem
            +
            "_accessions.txt"
        )
    )

    try:

        accession_file.write_text(
            "\n".join(
                accessions
            )
            +
            "\n",
            encoding="utf-8"
        )

        command = [

            str(blastdbcmd),

            "-db",
            str(db_root),

            "-entry_batch",
            str(accession_file),

            "-outfmt",
            "%f",

            "-out",
            str(output_fasta)
        ]

        logger.info(
            f"Retrieving {len(accessions)} "
            f"sequences with blastdbcmd"
        )

        result = subprocess.run(

            command,

            cwd=str(
                blastdbcmd.parent
            ),

            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,

            text=True,

            timeout=BLASTDBCMD_TIMEOUT
        )

        if result.returncode != 0:

            logger.error(
                "blastdbcmd failed:"
            )

            if result.stderr:

                logger.error(
                    result.stderr.strip()
                )

            return False

        if not output_fasta.is_file():

            logger.error(
                "blastdbcmd did not "
                "produce an output FASTA."
            )

            return False

        if output_fasta.stat().st_size == 0:

            logger.error(
                "blastdbcmd produced "
                "an empty FASTA."
            )

            return False

        return True

    except subprocess.TimeoutExpired:

        logger.error(
            "blastdbcmd timed out."
        )

        return False

    except Exception as e:

        logger.error(
            f"blastdbcmd error: {e}"
        )

        return False

    finally:

        accession_file.unlink(
            missing_ok=True
        )

def safe_filename(
    value: str
) -> str:

    value = str(value)

    value = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        value
    )

    value = value.strip()

    if not value:

        value = "unknown"

    return value

def retrieve_homolog_sequences(
    blastdbcmd: Path,
    db_root: Path,
    candidate_df: pd.DataFrame,
    homolog_dir: Path
):


    homolog_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    retrieval_records = []

    for gene_id, group in candidate_df.groupby(
        "qseqid",
        sort=False
    ):

        logger.info("")
        logger.info(
            f"{gene_id}: retrieving "
            f"{len(group)} homologs"
        )

        safe_gene = safe_filename(
            gene_id
        )

        output_fasta = (
            homolog_dir
            /
            f"{safe_gene}_homologs.fasta"
        )

        accessions = (
            group["accession"]
            .dropna()
            .astype(str)
            .tolist()
        )

        accessions = list(
            dict.fromkeys(
                accessions
            )
        )

        if not accessions:

            logger.warning(
                f"{gene_id}: no accessions"
            )

            continue

        success = run_blastdbcmd_batch(
            blastdbcmd,
            db_root,
            accessions,
            output_fasta
        )

        if not success:

            for accession in accessions:

                retrieval_records.append({

                    "qseqid":
                        gene_id,

                    "accession":
                        accession,

                    "retrieved":
                        False
                })

            continue

        try:

            text = (
                output_fasta
                .read_text(
                    encoding="utf-8",
                    errors="replace"
                )
            )

            records = parse_fasta_records(
                text
            )

        except Exception as e:

            logger.error(
                f"{gene_id}: could not "
                f"parse retrieved FASTA: {e}"
            )

            records = []

        retrieved_accessions = set()

        for header, sequence in records:

            accession = extract_accession(
                header
            )

            if accession:

                retrieved_accessions.add(
                    accession
                )

                retrieval_records.append({

                    "qseqid":
                        gene_id,

                    "accession":
                        accession,

                    "retrieved":
                        True,

                    "sequence_length":
                        len(sequence)
                })

        for accession in accessions:

            if accession not in retrieved_accessions:

                retrieval_records.append({

                    "qseqid":
                        gene_id,

                    "accession":
                        accession,

                    "retrieved":
                        False,

                    "sequence_length":
                        0
                })

        logger.info(
            f"{gene_id}: retrieved "
            f"{len(records)} sequences"
        )

    return pd.DataFrame(
        retrieval_records
    )

def filter_homolog_fasta(
    homolog_fasta: Path,
    min_length: int = MIN_SEQUENCE_LENGTH
):


    if not homolog_fasta.is_file():

        return "", 0

    text = (
        homolog_fasta
        .read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

    records = parse_fasta_records(
        text
    )

    valid_records = []

    for header, sequence in records:

        if len(sequence) < min_length:

            logger.warning(
                f"Removing short homolog "
                f"{header} "
                f"({len(sequence)} aa)"
            )

            continue

        allowed = set(
            "ACDEFGHIKLMNPQRSTVWYBXZJUO"
        )

        invalid = set(sequence) - allowed

        if invalid:

            logger.warning(
                f"Removing sequence "
                f"{header}: invalid "
                f"characters {invalid}"
            )

            continue

        valid_records.append(
            (
                header,
                sequence
            )
        )

    if not valid_records:

        return "", 0

    output_lines = []

    for header, sequence in valid_records:

        output_lines.append(
            f">{header}"
        )

        # Wrap sequence at 80 characters

        for i in range(
            0,
            len(sequence),
            80
        ):

            output_lines.append(
                sequence[
                    i:i + 80
                ]
            )

    return (
        "\n".join(output_lines)
        +
        "\n",
        len(valid_records)
    )

def build_phylo_fasta(
    gene_id: str,
    homolog_fasta: Path,
    metisa_lookup: dict,
    protein_id: str,
    output_fasta: Path
) -> tuple[bool, int]:

    if not homolog_fasta.is_file():

        return False, 0

    metisa_sequence = None

    if protein_id in metisa_lookup:

        metisa_sequence = (
            metisa_lookup[
                protein_id
            ]
        )

    elif gene_id in metisa_lookup:

        metisa_sequence = (
            metisa_lookup[
                gene_id
            ]
        )

    if not metisa_sequence:

        logger.warning(
            f"{gene_id}: Metisa protein "
            f"{protein_id} not found."
        )

        return False, 0

    metisa_sequence = clean_protein_sequence(
        metisa_sequence
    )

    if len(metisa_sequence) < MIN_SEQUENCE_LENGTH:

        logger.warning(
            f"{gene_id}: Metisa sequence "
            f"is too short "
            f"({len(metisa_sequence)} aa)."
        )

        return False, 0

    homolog_text, homolog_count = (
        filter_homolog_fasta(
            homolog_fasta
        )
    )

    if homolog_count == 0:

        logger.warning(
            f"{gene_id}: no valid homolog "
            f"sequences available."
        )

        return False, 0

    with open(
        output_fasta,
        "w",
        encoding="utf-8"
    ) as handle:

        handle.write(
            f">METISA_PLANA|{protein_id}\n"
        )

        for i in range(
            0,
            len(metisa_sequence),
            80
        ):

            handle.write(
                metisa_sequence[
                    i:i + 80
                ]
                +
                "\n"
            )

        handle.write(
            homolog_text
        )

    total_sequences = (
        homolog_count + 1
    )

    logger.info(
        f"{gene_id}: phylogenetic FASTA "
        f"contains {total_sequences} sequences "
        f"(1 Metisa + {homolog_count} homologs)"
    )

    return True, total_sequences

def run_mafft(
    mafft: Path,
    input_fasta: Path,
    output_alignment: Path
) -> bool:

    command = [

        str(mafft),

        "--auto",

        str(input_fasta)
    ]

    logger.info(
        f"Running MAFFT: "
        f"{input_fasta.name}"
    )

    try:

        with open(
            output_alignment,
            "w",
            encoding="utf-8"
        ) as output_handle:

            result = subprocess.run(

                command,

                cwd=str(
                    mafft.parent
                ),

                stdout=output_handle,

                stderr=subprocess.PIPE,

                text=True,

                timeout=MAFFT_TIMEOUT
            )

        if result.returncode != 0:

            logger.error(
                "MAFFT failed:"
            )

            logger.error(
                result.stderr[-5000:]
            )

            return False

        if not output_alignment.is_file():

            return False

        if output_alignment.stat().st_size == 0:

            return False

        return True

    except subprocess.TimeoutExpired:

        logger.error(
            "MAFFT timed out."
        )

        return False

    except Exception as e:

        logger.error(
            f"MAFFT error: {e}"
        )

        return False

def windows_to_wsl_path(path: Path) -> str:
    """
    Convert a Windows path to a WSL /mnt/<drive>/... path.
    """

    path_str = str(path)

    # Normalize Windows backslashes
    path_str = path_str.replace("\\", "/")

    if len(path_str) >= 2 and path_str[1] == ":":
        drive = path_str[0].lower()
        remainder = path_str[2:].lstrip("/")

        return f"/mnt/{drive}/{remainder}"

    return path_str

def run_trimal(
    trimal: Path,
    git_bash: Path | None,
    input_alignment: Path,
    output_alignment: Path
) -> bool:
    """
    Run trimAl.

    Supports:
        - WSL trimAl
        - Native Windows trimAl

    WSL trimAl is executed through:
        wsl <trimal_path>

    Git Bash is not required.
    """

    logger.info(
        f"Running trimAl: "
        f"{input_alignment.name}"
    )

    # ========================================================
    # DETERMINE IF THIS IS THE WSL TRIMAL
    # ========================================================

    trimal_str = str(trimal)

    if (
        trimal_str.startswith("/")
        or trimal_str.startswith("\\home\\")
        or trimal_str.startswith("\\usr\\")
    ):

        # ----------------------------------------------------
        # Convert Python's path representation back to WSL
        # ----------------------------------------------------

        trimal_wsl = trimal_str.replace("\\", "/")

        # Make sure it starts with /
        if not trimal_wsl.startswith("/"):
            trimal_wsl = "/" + trimal_wsl

        # ----------------------------------------------------
        # Convert Windows alignment paths to WSL paths
        # ----------------------------------------------------

        input_wsl = windows_to_wsl_path(
            input_alignment
        )

        output_wsl = windows_to_wsl_path(
            output_alignment
        )

        command = [
            "wsl",
            trimal_wsl,
            "-in",
            input_wsl,
            "-out",
            output_wsl,
            "-automated1"
        ]

        logger.info(
            f"Using WSL trimAl: {trimal_wsl}"
        )

    # ========================================================
    # NATIVE WINDOWS TRIMAL
    # ========================================================

    else:

        command = [
            str(trimal),
            "-in",
            str(input_alignment),
            "-out",
            str(output_alignment),
            "-automated1"
        ]

        logger.info(
            f"Using Windows trimAl: {trimal}"
        )

    # ========================================================
    # RUN TRIMAL
    # ========================================================

    try:

        result = subprocess.run(

            command,

            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,

            text=True,

            timeout=TRIMAL_TIMEOUT
        )

        # ----------------------------------------------------
        # Check exit status
        # ----------------------------------------------------

        if result.returncode != 0:

            logger.error(
                "trimAl failed."
            )

            if result.stdout:

                logger.error(
                    result.stdout
                )

            if result.stderr:

                logger.error(
                    result.stderr
                )

            return False

        # ----------------------------------------------------
        # Check output file
        # ----------------------------------------------------

        if not output_alignment.is_file():

            logger.error(
                "trimAl did not produce "
                "the output alignment."
            )

            return False

        if output_alignment.stat().st_size == 0:

            logger.error(
                "trimAl produced an "
                "empty alignment."
            )

            return False

        logger.info(
            f"trimAl completed successfully: "
            f"{output_alignment.name}"
        )

        return True

    except subprocess.TimeoutExpired:

        logger.error(
            "trimAl timed out."
        )

        return False

    except FileNotFoundError:

        logger.error(
            f"trimAl executable not found: "
            f"{trimal}"
        )

        return False

    except Exception as e:

        logger.error(
            f"trimAl error: {e}"
        )

        return False


# ============================================================
# IQ-TREE
# ============================================================

def run_iqtree(
    iqtree: Path,
    input_alignment: Path,
    output_prefix: Path
) -> bool:

    command = [

        str(iqtree),

        "-s",
        str(input_alignment),

        "-m",
        "MFP",

        "-B",
        str(UFBBOOT_REPLICATES),

        "-alrt",
        str(SH_ALRT_REPLICATES),

        "-nt",
        "AUTO",

        "-pre",
        str(output_prefix)
    ]

    logger.info(
        f"Running IQ-TREE: "
        f"{input_alignment.name}"
    )

    logger.info(
        "Model: ModelFinder Plus (MFP)"
    )

    logger.info(
        f"UFBoot: {UFBBOOT_REPLICATES}"
    )

    logger.info(
        f"SH-aLRT: {SH_ALRT_REPLICATES}"
    )

    try:

        result = subprocess.run(

            command,

            cwd=str(
                iqtree.parent
            ),

            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,

            text=True,

            timeout=IQTREE_TIMEOUT
        )

        if result.returncode != 0:

            logger.error(
                "IQ-TREE failed:"
            )

            if result.stdout:

                logger.error(
                    result.stdout[-5000:]
                )

            if result.stderr:

                logger.error(
                    result.stderr[-5000:]
                )

            return False

        treefile = Path(
            str(output_prefix)
            +
            ".treefile"
        )

        iqtree_report = Path(
            str(output_prefix)
            +
            ".iqtree"
        )

        if not treefile.is_file():

            logger.error(
                "IQ-TREE finished but "
                ".treefile was not produced."
            )

            return False

        if treefile.stat().st_size == 0:

            logger.error(
                "IQ-TREE produced "
                "an empty treefile."
            )

            return False

        if not iqtree_report.is_file():

            logger.warning(
                "IQ-TREE tree was produced "
                "but .iqtree report was not found."
            )

        return True

    except subprocess.TimeoutExpired:

        logger.error(
            "IQ-TREE timed out."
        )

        return False

    except Exception as e:

        logger.error(
            f"IQ-TREE error: {e}"
        )

        return False


# ============================================================
# READ IQ-TREE REPORT
# ============================================================

def extract_iqtree_model(
    iqtree_report: Path
) -> str:
    """
    Try to extract the selected model
    from the IQ-TREE report.
    """

    if not iqtree_report.is_file():

        return ""

    try:

        text = (
            iqtree_report
            .read_text(
                encoding="utf-8",
                errors="replace"
            )
        )

    except Exception:

        return ""

    patterns = [

        r"Best-fit model:\s*(\S+)",

        r"Best-fit model according to BIC:\s*(\S+)",

        r"Best-fit model according to AICc:\s*(\S+)",

        r"Model of substitution:\s*(\S+)"
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:

            return match.group(1)

    return ""


# ============================================================
# TREE SUPPORT
# ============================================================

def extract_support_values(
    treefile: Path
):
    """
    Extract support values from IQ-TREE
    Newick labels.

    IQ-TREE commonly writes branch support
    as:

        SH-aLRT/UFBoot

    e.g.

        98.5/100

    This function returns both components
    when available.
    """

    if not treefile.is_file():

        return {
            "ufboot_values": [],
            "sh_alrt_values": []
        }

    try:

        tree = (
            treefile
            .read_text(
                encoding="utf-8",
                errors="replace"
            )
            .strip()
        )

    except Exception:

        return {
            "ufboot_values": [],
            "sh_alrt_values": []
        }

    # --------------------------------------------------------
    # Match support labels such as:
    #
    # 98/100
    # 98.5/100
    # --------------------------------------------------------

    pair_matches = re.findall(
        r"\)(\d+(?:\.\d+)?)\/"
        r"(\d+(?:\.\d+)?)"
        r"(?=:|,|\))",
        tree
    )

    sh_alrt_values = []
    ufboot_values = []

    for sh_value, ufboot_value in pair_matches:

        try:

            sh_alrt_values.append(
                float(sh_value)
            )

            ufboot_values.append(
                float(ufboot_value)
            )

        except ValueError:

            continue

    # --------------------------------------------------------
    # If only one support value exists
    # --------------------------------------------------------

    if not pair_matches:

        single_matches = re.findall(
            r"\)(\d+(?:\.\d+)?)(?=:|,|\))",
            tree
        )

        for value in single_matches:

            try:

                ufboot_values.append(
                    float(value)
                )

            except ValueError:

                continue

    return {
        "ufboot_values":
            ufboot_values,

        "sh_alrt_values":
            sh_alrt_values
    }


def summarize_values(values):
    if not values:
        return {"mean": None, "minimum": None, "maximum": None, "nodes": 0}

    return {
        "mean": sum(values) / len(values),
        "minimum": min(values),
        "maximum": max(values),
        "nodes": len(values)
    }


# ============================================================
# PROCESS ONE CANDIDATE
# ============================================================

def process_candidate(
    gene_id: str,
    protein_id: str,
    homolog_fasta: Path,
    metisa_lookup: dict,
    mafft: Path,
    trimal: Path,
    git_bash: Path,
    iqtree: Path,
    phylo_fasta_dir: Path,
    alignments_dir: Path,
    trees_dir: Path
):

    logger.info("")
    logger.info(
        "=" * 75
    )

    logger.info(
        f"PROCESSING CANDIDATE: {gene_id}"
    )

    logger.info(
        "=" * 75
    )

    result = {

        "gene_id":
            gene_id,

        "protein_id":
            protein_id,

        "success":
            False,

        "sequence_count":
            0,

        "alignment_length":
            None,

        "trimmed_alignment_length":
            None,

        "model":
            "",

        "ufboot_mean_pct":
            None,

        "ufboot_min_pct":
            None,

        "ufboot_max_pct":
            None,

        "ufboot_nodes":
            0,

        "sh_alrt_mean_pct":
            None,

        "sh_alrt_min_pct":
            None,

        "sh_alrt_max_pct":
            None,

        "sh_alrt_nodes":
            0,

        "error":
            ""
    }

    safe_gene = safe_filename(
        gene_id
    )

    combined_fasta = (
        phylo_fasta_dir
        /
        f"{safe_gene}_phylo.fasta"
    )

    alignment = (
        alignments_dir
        /
        f"{safe_gene}.aln"
    )

    trimmed = (
        alignments_dir
        /
        f"{safe_gene}.trimmed.aln"
    )

    tree_prefix = (
        trees_dir
        /
        safe_gene
    )

    treefile = Path(
        str(tree_prefix)
        +
        ".treefile"
    )

    iqtree_report = Path(
        str(tree_prefix)
        +
        ".iqtree"
    )

    fasta_success, sequence_count = (
        build_phylo_fasta(

            gene_id,

            homolog_fasta,

            metisa_lookup,

            protein_id,

            combined_fasta
        )
    )

    if not fasta_success:

        result["error"] = (
            "Could not build "
            "phylogenetic FASTA"
        )

        return result

    result[
        "sequence_count"
    ] = sequence_count


    if not alignment.is_file():

        if not run_mafft(

            mafft,

            combined_fasta,

            alignment

        ):

            result["error"] = (
                "MAFFT failed"
            )

            return result

    else:

        logger.info(
            f"MAFFT output already exists: "
            f"{alignment.name}"
        )

    try:

        alignment_text = (
            alignment
            .read_text(
                encoding="utf-8",
                errors="replace"
            )
        )

        alignment_records = (
            parse_fasta_records(
                alignment_text
            )
        )

        if alignment_records:

            result[
                "alignment_length"
            ] = len(
                alignment_records[0][1]
            )

    except Exception:

        pass

    if not trimmed.is_file():

        if not run_trimal(

            trimal,

            git_bash,

            alignment,

            trimmed

        ):

            result["error"] = (
                "trimAl failed"
            )

            return result

    else:

        logger.info(
            f"trimAl output already exists: "
            f"{trimmed.name}"
        )

    try:

        trimmed_text = (
            trimmed
            .read_text(
                encoding="utf-8",
                errors="replace"
            )
        )

        trimmed_records = (
            parse_fasta_records(
                trimmed_text
            )
        )

        if trimmed_records:

            result[
                "trimmed_alignment_length"
            ] = len(
                trimmed_records[0][1]
            )

    except Exception:

        pass

    if not treefile.is_file():

        if not run_iqtree(

            iqtree,

            trimmed,

            tree_prefix

        ):

            result["error"] = (
                "IQ-TREE failed"
            )

            return result

    else:

        logger.info(
            f"IQ-TREE tree already exists: "
            f"{treefile.name}"
        )

    result[
        "model"
    ] = extract_iqtree_model(
        iqtree_report
    )

    support = extract_support_values(
        treefile
    )

    ufboot_summary = summarize_values(
        support["ufboot_values"]
    )

    sh_alrt_summary = summarize_values(
        support["sh_alrt_values"]
    )

    result[
        "ufboot_mean_pct"
    ] = ufboot_summary["mean"]

    result[
        "ufboot_min_pct"
    ] = ufboot_summary["minimum"]

    result[
        "ufboot_max_pct"
    ] = ufboot_summary["maximum"]

    result[
        "ufboot_nodes"
    ] = ufboot_summary["nodes"]

    result[
        "sh_alrt_mean_pct"
    ] = sh_alrt_summary["mean"]

    result[
        "sh_alrt_min_pct"
    ] = sh_alrt_summary["minimum"]

    result[
        "sh_alrt_max_pct"
    ] = sh_alrt_summary["maximum"]

    result[
        "sh_alrt_nodes"
    ] = sh_alrt_summary["nodes"]

    result["success"] = True

    logger.info(
        f"{gene_id}: phylogenetic analysis complete"
    )

    if result[
        "model"
    ]:

        logger.info(
            f"{gene_id}: model = "
            f"{result['model']}"
        )

    if result[
        "ufboot_mean_pct"
    ] is not None:

        logger.info(
            f"{gene_id}: mean UFBoot = "
            f"{result['ufboot_mean_pct']:.2f}%"
        )

    if result[
        "sh_alrt_mean_pct"
    ] is not None:

        logger.info(
            f"{gene_id}: mean SH-aLRT = "
            f"{result['sh_alrt_mean_pct']:.2f}%"
        )

    return result

def load_interpro(
    interpro_file: Path
) -> pd.DataFrame:

    try:

        df = pd.read_csv(

            interpro_file,

            sep="\t",

            header=None,

            dtype=str,

            comment="#"
        )

        logger.info(
            f"Loaded InterProScan data: "
            f"{len(df)} rows"
        )

        return df

    except Exception as e:

        logger.warning(
            f"Could not parse InterProScan TSV: "
            f"{e}"
        )

        return pd.DataFrame()

def get_interpro_hits(interpro_df, protein_id):

    if interpro_df.empty:
        return ""

    matches = []

    for _, row in interpro_df.iterrows():

        values = [
            str(x) for x in row.tolist()
            if pd.notna(x)
        ]

        if not values:
            continue

        first = values[0]

        if first == protein_id or first.startswith(protein_id + "."):

            if len(values) >= 12:

                matches.append(
                    f"{values[4]}: {values[5]}"
                )

                if values[11] and values[11] != "-":
                    matches.append(
                        f"InterPro={values[11]}"
                    )

    return "; ".join(dict.fromkeys(matches))

def build_annotation_summary(
    top_hits: pd.DataFrame,
    results_df: pd.DataFrame,
    interpro_df: pd.DataFrame,
    metisa_lookup: dict
) -> pd.DataFrame:

    rows = []

    candidate_ids = (
        top_hits[
            "qseqid"
        ]
        .drop_duplicates()
        .tolist()
    )

    for gene_id in candidate_ids:

        group = top_hits[
            top_hits["qseqid"]
            ==
            gene_id
        ]

        if group.empty:

            continue

        first = group.iloc[0]

        protein_id = gene_id

        if (
            "protein_id" in group.columns
            and
            pd.notna(
                first.get(
                    "protein_id"
                )
            )
        ):

            protein_id = str(
                first[
                    "protein_id"
                ]
            )

        interpro_hits = (
            get_interpro_hits(
                interpro_df,
                protein_id
            )
        )

        row = {

            "gene_id":
                gene_id,

            "protein_id":
                protein_id,

            "top_hit_sseqid":
                first.get(
                    "sseqid",
                    ""
                ),

            "top_hit_accession":
                first.get(
                    "accession",
                    ""
                ),

            "top_hit_identity_pct":
                first.get(
                    "pident",
                    ""
                ),

            "top_hit_query_coverage_pct":
                first.get(
                    "qcovs",
                    ""
                ),

            "top_hit_evalue":
                first.get(
                    "evalue",
                    ""
                ),

            "top_hit_bitscore":
                first.get(
                    "bitscore",
                    ""
                ),

            "top_hit_description":
                first.get(
                    "stitle",
                    ""
                ),

            "interpro_evidence":
                interpro_hits
        }

        if not results_df.empty:

            matching = results_df[
                results_df[
                    "gene_id"
                ].astype(str)
                ==
                str(gene_id)
            ]

        else:

            matching = pd.DataFrame()

        if not matching.empty:

            phylo = matching.iloc[0]

            row[
                "phylo_success"
            ] = phylo.get(
                "success",
                False
            )

            row[
                "phylo_sequence_count"
            ] = phylo.get(
                "sequence_count",
                ""
            )

            row[
                "alignment_length"
            ] = phylo.get(
                "alignment_length",
                ""
            )

            row[
                "trimmed_alignment_length"
            ] = phylo.get(
                "trimmed_alignment_length",
                ""
            )

            row[
                "iqtree_model"
            ] = phylo.get(
                "model",
                ""
            )

            row[
                "ufboot_mean_pct"
            ] = phylo.get(
                "ufboot_mean_pct",
                ""
            )

            row[
                "ufboot_min_pct"
            ] = phylo.get(
                "ufboot_min_pct",
                ""
            )

            row[
                "ufboot_max_pct"
            ] = phylo.get(
                "ufboot_max_pct",
                ""
            )

            row[
                "ufboot_nodes"
            ] = phylo.get(
                "ufboot_nodes",
                0
            )

            row[
                "sh_alrt_mean_pct"
            ] = phylo.get(
                "sh_alrt_mean_pct",
                ""
            )

            row[
                "sh_alrt_min_pct"
            ] = phylo.get(
                "sh_alrt_min_pct",
                ""
            )

            row[
                "sh_alrt_max_pct"
            ] = phylo.get(
                "sh_alrt_max_pct",
                ""
            )

            row[
                "sh_alrt_nodes"
            ] = phylo.get(
                "sh_alrt_nodes",
                0
            )

            row[
                "phylo_error"
            ] = phylo.get(
                "error",
                ""
            )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )

def main():

    print()
    print(
        "=" * 80
    )

    print(
        "METISA PLANA — SCRIPT 2 v2"
    )

    print(
        "MAFFT + trimAl + IQ-TREE"
    )

    print(
        "=" * 80
    )

    print()

    print(
        "Workflow:"
    )

    print(
        "TOP-10 RNA candidates → "
        "RAW BLAST matching candidates → "
        "Top-N homologs → blastdbcmd → "
        "MAFFT → trimAl → IQ-TREE"
    )

    print()

    print(
        "Default homologs per tree: "
        f"{DEFAULT_HOMOLOGS_PER_TREE}"
    )

    print()

    print(
        "=" * 80
    )

    print(
        "STEP 1 — INPUT FILES"
    )

    print(
        "=" * 80
    )

    print()

    rna_candidates_tsv = ask_existing_file(
        "Path to TOP RNAi CANDIDATES TSV: "
    )

    blast_tsv = ask_existing_file(
        "Path to RAW BLAST TSV: "
    )

    metisa_fasta = ask_existing_file(
        "Path to filtered fasta: "
    )

    interpro_tsv = ask_existing_file(
        "Path to InterProScan TSV: "
    )

    print()
    print(
        "=" * 80
    )

    print(
        "STEP 2 — PHYLOGENETIC HOMOLOG COUNT"
    )

    print(
        "=" * 80
    )

    print()

    print(
        "Your BLAST file contains the top 10 hits."
    )

    print(
        "The script will select the best N "
        "homologs for each Metisa protein."
    )

    print()

    homologs_per_tree = ask_positive_integer(
        "Number of homologs per tree",
        DEFAULT_HOMOLOGS_PER_TREE
    )

    print()
    print(
        "=" * 80
    )

    print(
        "STEP 3 — BLAST PROTEIN DATABASE"
    )

    print(
        "=" * 80
    )

    db_root, db_info = (
        ask_blast_database()
    )

    print()
    print(
        "=" * 80
    )

    print(
        "STEP 4 — SOFTWARE EXECUTABLES"
    )

    print(
        "=" * 80
    )

    print()

    mafft, _ = ask_executable(
        "MAFFT"
    )

    trimal, git_bash = ask_executable(
        "trimAl"
    )

    iqtree, _ = ask_executable(
        "IQ-TREE"
    )

    blastdbcmd, _ = ask_executable(
        "BLASTDBCMD"
    )

    print()
    print(
        "=" * 80
    )

    print(
        "STEP 5 — OUTPUT"
    )

    print(
        "=" * 80
    )

    print()

    output_folder = (
        ask_output_folder()
    )

    homolog_dir = (
        output_folder
        /
        "homolog_sequences"
    )

    phylo_fasta_dir = (
        output_folder
        /
        "phylo_fastas"
    )

    alignments_dir = (
        output_folder
        /
        "alignments"
    )

    trees_dir = (
        output_folder
        /
        "trees"
    )

    work_dir = (
        output_folder
        /
        "temp"
    )

    for directory in [

        homolog_dir,

        phylo_fasta_dir,

        alignments_dir,

        trees_dir,

        work_dir

    ]:

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    print()

    print(
        "=" * 80
    )

    print(
        "LOADING TOP RNA CANDIDATES"
    )

    print(
        "=" * 80
    )

    try:

        rna_candidates_df = pd.read_csv(
            rna_candidates_tsv,
            sep="\t",
            dtype=str
        )

    except Exception as e:

        logger.error(
            f"Cannot load RNA candidates TSV: {e}"
        )

        sys.exit(1)

    if "gene_id" not in rna_candidates_df.columns:

        logger.error(
            "RNA candidates TSV must contain "
            "a 'gene_id' column."
        )

        sys.exit(1)

    rna_candidate_ids = (
        rna_candidates_df[
            "gene_id"
        ]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    logger.info(
        f"RNA candidates loaded: "
        f"{len(rna_candidate_ids)}"
    )

    print("\n" + "=" * 80)
    print("LOADING BLAST DATA")
    print("=" * 80)

    try:
        blast_df = load_blast_tsv(blast_tsv)
    except Exception as e:
        logger.error(f"Cannot load BLAST TSV: {e}")
        sys.exit(1)

    print("\nLoading Metisa protein FASTA...")
    metisa_lookup = build_fasta_lookup(metisa_fasta)

    print("\nLoading InterProScan...")
    interpro_df = load_interpro(interpro_tsv)

    print("\n" + "=" * 80)
    print("SELECTING TOP HOMOLOGS FOR RNA CANDIDATES")
    print("=" * 80)

    blast_df["_gene_id"] = (
        blast_df["qseqid"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.t\d+$", "", regex=True)
    )

    blast_candidate_df = blast_df[
        blast_df["_gene_id"].isin(rna_candidate_ids)
    ].copy()

    logger.info(
        f"BLAST rows belonging to TOP RNA candidates: "
        f"{len(blast_candidate_df)}"
    )

    blast_candidate_ids = (
        blast_candidate_df["qseqid"]
        .drop_duplicates()
        .tolist()
    )

    logger.info(
        f"RNA candidates found in BLAST: "
        f"{len(blast_candidate_ids)}"
    )

    top_hits = select_top_homologs(
        blast_candidate_df,
        max_hits=homologs_per_tree
    )

    logger.info(
        f"Selected {len(top_hits)} "
        f"homolog records"
    )

    candidate_ids = (
        top_hits[
            "qseqid"
        ]
        .drop_duplicates()
        .tolist()
    )

    logger.info(
        f"Candidates with selected homologs: "
        f"{len(candidate_ids)}"
    )

    missing_candidates = [
        gene_id
        for gene_id in rna_candidate_ids
        if gene_id not in blast_candidate_ids
    ]

    if missing_candidates:

        logger.warning(
            f"{len(missing_candidates)} RNA candidates "
            f"were not found in the BLAST TSV."
        )

        logger.warning(
            "Missing RNA candidate IDs: "
            +
            ", ".join(missing_candidates)
        )

    selected_tsv = output_folder / "selected_homologs.tsv"
    top_hits.to_csv(selected_tsv, sep="\t", index=False)

    print("\nSelected homolog table saved:")
    print(f"  {selected_tsv}")

    print("\n" + "=" * 80)
    print("RETRIEVING HOMOLOG SEQUENCES")
    print("=" * 80)

    retrieval_df = retrieve_homolog_sequences(
        blastdbcmd,
        db_root,
        top_hits,
        homolog_dir
    )

    retrieval_tsv = output_folder / "homolog_retrieval.tsv"
    retrieval_df.to_csv(retrieval_tsv, sep="\t", index=False)

    print("\n" + "=" * 80)
    print("PHYLOGENETIC ANALYSIS")
    print("=" * 80)

    results = []

    for index, gene_id in enumerate(candidate_ids, start=1):
        print("\n" + "-" * 80)
        print(f"Candidate {index}/{len(candidate_ids)}: {gene_id}")
        print("-" * 80)

        group = top_hits[top_hits["qseqid"] == gene_id]

        if group.empty:
            continue

        protein_id = gene_id

        if "protein_id" in group.columns:

            candidate_ids_from_column = (

                group[
                    "protein_id"
                ]

                .dropna()

                .astype(str)

                .tolist()
            )

            if candidate_ids_from_column:

                protein_id = (
                    candidate_ids_from_column[0]
                )

        homolog_fasta = homolog_dir / f"{safe_filename(gene_id)}_homologs.fasta"

        if not homolog_fasta.is_file():
            logger.warning(
                f"{gene_id}: homolog FASTA not found. Skipping."
            )

            results.append({
                "gene_id": gene_id,
                "protein_id": protein_id,
                "success": False,
                "sequence_count": 0,
                "error": "No homolog FASTA"
            })

            continue

        result = process_candidate(
            gene_id=gene_id,
            protein_id=protein_id,
            homolog_fasta=homolog_fasta,
            metisa_lookup=metisa_lookup,
            mafft=mafft,
            trimal=trimal,
            git_bash=git_bash,
            iqtree=iqtree,
            phylo_fasta_dir=phylo_fasta_dir,
            alignments_dir=alignments_dir,
            trees_dir=trees_dir
        )

        results.append(result)

    results_df = pd.DataFrame(results)
    results_tsv = output_folder / "phylo_results.tsv"
    results_df.to_csv(results_tsv, sep="\t", index=False)

    print()
    print("=" * 80)
    print("BUILDING ANNOTATION SUMMARY")
    print("=" * 80)

    annotation_df = build_annotation_summary(
        top_hits,
        results_df,
        interpro_df,
        metisa_lookup
    )

    annotation_tsv = output_folder / "phylo_annotation_summary.tsv"
    annotation_df.to_csv(annotation_tsv, sep="\t", index=False)

    successful = 0
    if not results_df.empty:
        successful = int((results_df["success"] == True).sum())

    failed = len(candidate_ids) - successful

    print("\n" + "=" * 80)
    print("SCRIPT 2 v2 COMPLETE")
    print("=" * 80 + "\n")

    print(f"Input BLAST rows: {len(blast_df)}")
    print(f"RNA candidates processed: {len(candidate_ids)}")
    print(f"Homologs per tree: {homologs_per_tree}")
    print(f"Selected homolog rows: {len(top_hits)}")
    print(f"Successful phylogenies: {successful}")
    print(f"Failed phylogenies: {failed}\n")

    print("BLAST database:")
    if db_info["type"] == "split":
        print("  Type: SPLIT")
        print(f"  Volumes: {len(db_info['volumes'])}")
    else:
        print("  Type: SINGLE")

    print("\nPhylogenetic settings:")
    print("  MAFFT: --auto")
    print("  trimAl: -automated1")
    print("  IQ-TREE model: MFP")
    print(f"  UFBoot: {UFBBOOT_REPLICATES}")
    print(f"  SH-aLRT: {SH_ALRT_REPLICATES}")
    print("  Threads: AUTO\n")

    print("Output files:")
    print(f"  Selected homologs:\n    {selected_tsv}\n")
    print(f"  Homolog retrieval:\n    {retrieval_tsv}\n")
    print(f"  Phylogenetic results:\n    {results_tsv}\n")
    print(f"  Annotation summary:\n    {annotation_tsv}\n")

    print("Output directories:")
    print(f"  Homolog sequences:\n    {homolog_dir}")
    print(f"  Phylogenetic FASTA:\n    {phylo_fasta_dir}")
    print(f"  Alignments:\n    {alignments_dir}")
    print(f"  Trees:\n    {trees_dir}\n")

    print("NEXT STEP:")
    print(
        "Use the phylogenetic results together "
        "with BLAST and InterPro evidence "
        "for candidate prioritization."
    )

    print("\n" + "=" * 80)

if __name__ == "__main__":
    try:
        main()

    except KeyboardInterrupt:
        print("\nProcess cancelled by user.")
        sys.exit(1)

    except Exception as e:
        print("\nFATAL ERROR:")
        print(f"  {e}")
        logger.exception("Full error:")
        sys.exit(1)


METISA PLANA — SCRIPT 2 v2
MAFFT + trimAl + IQ-TREE

Workflow:
TOP-10 RNA candidates → RAW BLAST matching candidates → Top-N homologs → blastdbcmd → MAFFT → trimAl → IQ-TREE

Default homologs per tree: 10

STEP 1 — INPUT FILES


STEP 2 — PHYLOGENETIC HOMOLOG COUNT

Your BLAST file contains the top 10 hits.
The script will select the best N homologs for each Metisa protein.


STEP 3 — BLAST PROTEIN DATABASE

Checking BLAST protein database...
  Database root: C:\BLAST\db\uniprot_combined

  SPLIT BLAST PROTEIN DATABASE DETECTED
  Number of volumes: 7
    Volume 00: uniprot_combined.00
    Volume 01: uniprot_combined.01
    Volume 02: uniprot_combined.02
    Volume 03: uniprot_combined.03
    Volume 04: uniprot_combined.04
    Volume 05: uniprot_combined.05
    Volume 06: uniprot_combined.06


STEP 4 — SOFTWARE EXECUTABLES


Testing MAFFT...
  Executable: C:\BLAST\tools\mafft-win\mafft.bat

EXECUTABLE DIAGNOSTIC

[1/5] Checking path:
      C:\BLAST\tools\mafft-win\mafft.bat
      OK

[2

2026-09-05 18:20:39,254 | INFO | RNA candidates loaded: 72
2026-09-05 18:20:39,255 | INFO | Loading BLAST TSV: C:\metp\output\testing_raw_blast.tsv
2026-09-05 18:20:39,277 | INFO | Loaded 1156 BLAST rows
2026-09-05 18:20:39,280 | INFO | Unique Metisa proteins: 99
2026-09-05 18:20:39,283 | INFO | Loaded 76 Metisa protein sequences
2026-09-05 18:20:39,334 | INFO | Loaded InterProScan data: 675 rows
2026-09-05 18:20:39,358 | INFO | BLAST rows belonging to TOP RNA candidates: 862
2026-09-05 18:20:39,360 | INFO | RNA candidates found in BLAST: 72
2026-09-05 18:20:39,383 | INFO | Selected 720 homolog records
2026-09-05 18:20:39,384 | INFO | Candidates with selected homologs: 72
2026-09-05 18:20:39,385 | WARNING | 72 RNA candidates were not found in the BLAST TSV.
2026-09-05 18:20:39,385 | WARNING | Missing RNA candidate IDs: g94, g96, g11, g4, g95, g97, g27, g9, g7, g8, g5, g10, g74, g54, g58, g53, g33, g42, g6, g34, g75, g50, g71, g21, g69, g13, g43, g72, g70, g36, g52, g41, g31, g1, g51, g


LOADING TOP RNA CANDIDATES

LOADING BLAST DATA

Loading Metisa protein FASTA...

Loading InterProScan...

SELECTING TOP HOMOLOGS FOR RNA CANDIDATES

Selected homolog table saved:
  C:\metp\output\phylogeny\selected_homologs.tsv

RETRIEVING HOMOLOG SEQUENCES


2026-09-05 18:20:39,981 | INFO | g34.t1: retrieved 10 sequences
2026-09-05 18:20:39,982 | INFO | 
2026-09-05 18:20:39,982 | INFO | g98.t1: retrieving 10 homologs
2026-09-05 18:20:39,985 | INFO | Retrieving 10 sequences with blastdbcmd
2026-09-05 18:20:40,834 | INFO | g98.t1: retrieved 10 sequences
2026-09-05 18:20:40,835 | INFO | 
2026-09-05 18:20:40,836 | INFO | g96.t1: retrieving 10 homologs
2026-09-05 18:20:40,839 | INFO | Retrieving 10 sequences with blastdbcmd
2026-09-05 18:20:41,272 | INFO | g96.t1: retrieved 10 sequences
2026-09-05 18:20:41,273 | INFO | 
2026-09-05 18:20:41,273 | INFO | g50.t3: retrieving 10 homologs
2026-09-05 18:20:41,276 | INFO | Retrieving 10 sequences with blastdbcmd
2026-09-05 18:20:41,680 | INFO | g50.t3: retrieved 10 sequences
2026-09-05 18:20:41,681 | INFO | 
2026-09-05 18:20:41,682 | INFO | g41.t1: retrieving 10 homologs
2026-09-05 18:20:41,686 | INFO | Retrieving 10 sequences with blastdbcmd
2026-09-05 18:20:42,568 | INFO | g41.t1: retrieved 10 sequen


PHYLOGENETIC ANALYSIS

--------------------------------------------------------------------------------
Candidate 1/72: g34.t1
--------------------------------------------------------------------------------


...


--------------------------------------------------------------------------------
Candidate 71/72: g92.t1
--------------------------------------------------------------------------------


2026-09-05 19:10:42,292 | INFO | Running trimAl: g92.t1.aln
2026-09-05 19:10:42,294 | INFO | Using WSL trimAl: /home/nurly/trimal-1.5.1/source/trimal.exe
2026-09-05 19:10:44,564 | INFO | trimAl completed successfully: g92.t1.trimmed.aln
2026-09-05 19:10:44,567 | INFO | Running IQ-TREE: g92.t1.trimmed.aln
2026-09-05 19:10:44,569 | INFO | Model: ModelFinder Plus (MFP)
2026-09-05 19:10:44,569 | INFO | UFBoot: 1000
2026-09-05 19:10:44,570 | INFO | SH-aLRT: 1000
2026-09-05 19:11:18,275 | INFO | g92.t1: phylogenetic analysis complete
2026-09-05 19:11:18,276 | INFO | g92.t1: model = DAYHOFF+I+G4
2026-09-05 19:11:18,277 | INFO | g92.t1: mean UFBoot = 64.88%
2026-09-05 19:11:18,277 | INFO | g92.t1: mean SH-aLRT = 73.56%
2026-09-05 19:11:18,283 | INFO | 
2026-09-05 19:11:18,284 | INFO | ===========================================================================
2026-09-05 19:11:18,285 | INFO | PROCESSING CANDIDATE: g29.t1
2026-09-05 19:11:18,287 | INFO | =========================================


--------------------------------------------------------------------------------
Candidate 72/72: g29.t1
--------------------------------------------------------------------------------


2026-09-05 19:11:20,109 | INFO | Running trimAl: g29.t1.aln
2026-09-05 19:11:20,110 | INFO | Using WSL trimAl: /home/nurly/trimal-1.5.1/source/trimal.exe
2026-09-05 19:11:23,194 | INFO | trimAl completed successfully: g29.t1.trimmed.aln
2026-09-05 19:11:23,197 | INFO | Running IQ-TREE: g29.t1.trimmed.aln
2026-09-05 19:11:23,198 | INFO | Model: ModelFinder Plus (MFP)
2026-09-05 19:11:23,198 | INFO | UFBoot: 1000
2026-09-05 19:11:23,199 | INFO | SH-aLRT: 1000
2026-09-05 19:12:00,071 | INFO | g29.t1: phylogenetic analysis complete
2026-09-05 19:12:00,073 | INFO | g29.t1: model = MTZOA+G4
2026-09-05 19:12:00,073 | INFO | g29.t1: mean UFBoot = 57.75%
2026-09-05 19:12:00,074 | INFO | g29.t1: mean SH-aLRT = 79.65%



BUILDING ANNOTATION SUMMARY

SCRIPT 2 v2 COMPLETE

Input BLAST rows: 1156
RNA candidates processed: 72
Homologs per tree: 10
Selected homolog rows: 720
Successful phylogenies: 72
Failed phylogenies: 0

BLAST database:
  Type: SPLIT
  Volumes: 7

Phylogenetic settings:
  MAFFT: --auto
  trimAl: -automated1
  IQ-TREE model: MFP
  UFBoot: 1000
  SH-aLRT: 1000
  Threads: AUTO

Output files:
  Selected homologs:
    C:\metp\output\phylogeny\selected_homologs.tsv

  Homolog retrieval:
    C:\metp\output\phylogeny\homolog_retrieval.tsv

  Phylogenetic results:
    C:\metp\output\phylogeny\phylo_results.tsv

  Annotation summary:
    C:\metp\output\phylogeny\phylo_annotation_summary.tsv

Output directories:
  Homolog sequences:
    C:\metp\output\phylogeny\homolog_sequences
  Phylogenetic FASTA:
    C:\metp\output\phylogeny\phylo_fastas
  Alignments:
    C:\metp\output\phylogeny\alignments
  Trees:
    C:\metp\output\phylogeny\trees

NEXT STEP:
Use the phylogenetic results together with BLAST


# 08. Final Candidate Ranking

The final stage integrates the available evidence to produce the prioritised candidate list.

Evidence from sequence quality, homology, similarity thresholds, taxonomy, functional annotation, expression and phylogenetic analysis is considered together.

The final ranking provides a transparent prioritisation of the most promising *M. plana* protein candidates for targeted pesticide development.

The highest-ranked candidates can then be selected for detailed biological interpretation and reporting.


In [9]:

TOP_N = 10

WEIGHT_RNA = 30
WEIGHT_HOMOLOGY = 20
WEIGHT_ANNOTATION = 20
WEIGHT_PHYLO = 15
WEIGHT_TARGET = 15

def ask_file(label):

    while True:

        value = input(
            f"\nEnter {label} path:\n> "
        ).strip().strip('"')

        path = Path(value)

        if path.is_file():
            return path

        print(
            f"ERROR: File not found:\n{path}"
        )


def ask_output_directory():

    value = input(
        "\nEnter OUTPUT DIRECTORY "
        "(press Enter to use the same folder as the RNA file):\n> "
    ).strip().strip('"')

    if not value:
        return None

    path = Path(value)

    path.mkdir(
        parents=True,
        exist_ok=True
    )

    return path

def load_tsv(path, label):

    print(
        f"\nLoading {label}..."
    )

    try:

        df = pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            low_memory=False
        )

    except Exception as e:

        raise RuntimeError(
            f"Could not read {label}:\n{e}"
        )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    print(
        f"{label} rows loaded: "
        f"{len(df):,}"
    )

    return df

def require_columns(
    df,
    columns,
    label
):

    missing = [
        c
        for c in columns
        if c not in df.columns
    ]

    if missing:

        raise ValueError(
            f"\n{label} is missing required columns:\n"
            + "\n".join(
                f"  - {x}"
                for x in missing
            )
        )

def numeric(df, column):

    if column not in df.columns:

        return pd.Series(
            np.nan,
            index=df.index
        )

    return pd.to_numeric(
        df[column],
        errors="coerce"
    )

def normalize_gene_id(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    match = re.match(
        r"^(g\d+)",
        value,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1)

    return value

def clean_text(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in {
        "nan",
        "none",
        "null",
        "-"
    }:
        return ""

    return value

def extract_accession(value):

    value = clean_text(value)

    if not value:
        return ""

    parts = value.split("|")

    if len(parts) >= 2:

        accession = parts[1].strip()

        if accession:
            return accession

    return value

def extract_species(description):

    description = clean_text(
        description
    )

    if not description:
        return ""

    match = re.search(
        r"OS=(.*?)(?:\s+OX=|\s+GN=|\s+PE=|\s+SV=|$)",
        description
    )

    if match:
        return match.group(1).strip()

    return ""

def extract_function(description):

    description = clean_text(
        description
    )

    if not description:
        return ""

    description = re.sub(
        r"^(?:sp|tr)\|[^|]+\|[^ ]+\s+",
        "",
        description,
        flags=re.IGNORECASE
    )

    description = re.split(
        r"\s+OS=",
        description,
        maxsplit=1
    )[0]

    return description.strip()

def extract_pfam(interpro_rows):

    if interpro_rows.empty:
        return ""

    values = []

    if "Signature accession" in interpro_rows.columns:

        for value in interpro_rows[
            "Signature accession"
        ]:

            value = clean_text(value)

            if re.match(
                r"^PF\d+$",
                value
            ):

                values.append(value)

    return ";".join(
        dict.fromkeys(values)
    )

def extract_interpro(interpro_rows):

    if interpro_rows.empty:
        return ""

    values = []

    if "InterPro accession" in interpro_rows.columns:

        for value in interpro_rows[
            "InterPro accession"
        ]:

            value = clean_text(value)

            if re.match(
                r"^IPR\d+$",
                value
            ):

                values.append(value)

    return ";".join(
        dict.fromkeys(values)
    )

def extract_interpro_descriptions(
    interpro_rows
):

    if interpro_rows.empty:
        return ""

    values = []

    if "InterPro description" in interpro_rows.columns:

        for value in interpro_rows[
            "InterPro description"
        ]:

            value = clean_text(value)

            if value:
                values.append(value)

    return "; ".join(
        dict.fromkeys(values)
    )

def determine_gene_family(
    functional_annotation,
    interpro_description,
    pfam
):

    text = (
        clean_text(functional_annotation)
        + " "
        + clean_text(interpro_description)
        + " "
        + clean_text(pfam)
    )

    lower = text.lower()

    rules = [

        ("chitin synthase", "Chitin synthase"),

        ("trypsin", "Trypsin / Peptidase S1"),

        ("chymotrypsin", "Chymotrypsin / Peptidase S1"),

        ("serine protease", "Serine protease"),

        ("collagenase", "Collagenase / Serine protease"),

        ("pancreatic triacylglycerol lipase",
         "Triacylglycerol lipase"),

        ("triacylglycerol lipase",
         "Triacylglycerol lipase"),

        ("lipase", "Lipase"),

        ("nucleoside diphosphate kinase",
         "Nucleoside diphosphate kinase"),

        ("mitochondrial-processing peptidase",
         "Mitochondrial processing peptidase"),

        ("mitochondrial processing peptidase",
         "Mitochondrial processing peptidase"),

        ("presequence protease",
         "Mitochondrial presequence protease"),

        ("serine protease inhibitor",
         "Serine protease inhibitor"),

        ("protease inhibitor",
         "Protease inhibitor"),

        ("sh3 domain",
         "SH3-domain protein"),

        ("immunoglobulin-like",
         "Immunoglobulin-like protein"),

        ("leucine-rich repeat",
         "Leucine-rich repeat protein"),

        ("ef-hand",
         "EF-hand protein"),

        ("transcription factor",
         "Transcription factor"),
    ]

    for keyword, family in rules:

        if keyword in lower:
            return family

    ipr = clean_text(
        interpro_description
    )

    if ipr:

        first = ipr.split(";")[0].strip()

        if first:
            return first[:120]

    annotation = clean_text(
        functional_annotation
    )

    if annotation:
        return annotation[:120]

    return ""

def determine_target_class(
    functional_annotation,
    gene_family,
    interpro_description,
    existing_target_class
):

    annotation = (
        clean_text(functional_annotation)
        + " "
        + clean_text(gene_family)
        + " "
        + clean_text(interpro_description)
    )

    lower = annotation.lower()

    # CHITIN / STRUCTURAL

    if (
        "chitin synthase" in lower
        or "chitin pathway" in lower
    ):

        return "STRUCTURAL/CHITIN"

    # DIGESTION

    if any(
        x in lower
        for x in [
            "trypsin",
            "chymotrypsin",
            "serine protease",
            "collagenase",
            "lipase",
            "peptidase"
        ]
    ):

        if (
            "mitochondrial" not in lower
            and "presequence" not in lower
        ):

            return "DIGESTION"

    # TARGET SITE

    if any(
        x in lower
        for x in [
            "ion channel",
            "ligand-gated",
            "voltage-gated",
            "acetylcholine receptor",
            "nicotinic acetylcholine receptor",
            "gaba receptor",
            "gaba-gated",
            "glutamate-gated",
            "ryanodine receptor"
        ]
    ):

        return "TARGET-SITE"

    # SIGNALLING

    if any(
        x in lower
        for x in [
            "signaling",
            "signalling",
            "signal transduction",
            "sh3 domain"
        ]
    ):

        return "SIGNALING"

    existing = clean_text(
        existing_target_class
    )

    if existing:
        return existing.upper()

    return ""

def calculate_phylo_score(row):

    if not row["phylo_success"]:
        return 0.0

    values = []

    uf = row.get(
        "ufboot_mean_pct",
        np.nan
    )

    sh = row.get(
        "sh_alrt_mean_pct",
        np.nan
    )

    if pd.notna(uf):
        values.append(float(uf))

    if pd.notna(sh):
        values.append(float(sh))

    if not values:
        return 0.0

    return min(
        100.0,
        np.mean(values)
    )

def calculate_homology_score(
    identity,
    coverage,
    evalue
):

    if pd.isna(identity):
        identity = 0

    if pd.isna(coverage):
        coverage = 0

    if pd.isna(evalue):
        evalue = 1.0

    identity_score = min(
        100,
        max(
            0,
            (identity - 20)
            / 60
            * 100
        )
    )

    coverage_score = min(
        100,
        max(
            0,
            coverage
        )
    )

    if evalue <= 1e-50:
        evalue_score = 100

    elif evalue <= 1e-20:
        evalue_score = 90

    elif evalue <= 1e-10:
        evalue_score = 80

    elif evalue <= 1e-5:
        evalue_score = 70

    elif evalue <= 0.05:
        evalue_score = 50

    else:
        evalue_score = 0

    return (
        identity_score * 0.40
        + coverage_score * 0.40
        + evalue_score * 0.20
    )

def calculate_rna_score(row):

    score = row.get(
        "RNA_score_numeric",
        np.nan
    )

    if pd.notna(score):

        return min(
            100,
            max(
                0,
                float(score)
            )
        )

    rank = row.get(
        "RNA_rank_numeric",
        np.nan
    )

    if pd.isna(rank):
        return 0

    if rank <= 10:
        return 100

    if rank <= 25:
        return 90

    if rank <= 50:
        return 80

    if rank <= 75:
        return 70

    return 60

def calculate_stage_expression_score(
    tpm,
    all_stage_tpm
):

    if pd.isna(tpm):
        return 0.0

    tpm = max(
        0.0,
        float(tpm)
    )

    if tpm <= 0:
        return 0.0

    other_values = [
        float(x)
        for x in all_stage_tpm
        if pd.notna(x)
        and float(x) >= 0
    ]

    if not other_values:
        return 0.0

    maximum = max(
        other_values
    )

    if maximum <= 0:
        return 0.0

    # Relative expression score.
    # 100 = highest expression among stages.

    score = (
        tpm / maximum
    ) * 100

    return round(
        min(100, score),
        2
    )

def calculate_annotation_score(
    interpro_count,
    pfam_count,
    functional_annotation
):

    score = 0

    if interpro_count > 0:
        score += 50

    if pfam_count > 0:
        score += 30

    if clean_text(
        functional_annotation
    ):

        score += 20

    return min(
        100,
        score
    )

def calculate_target_score(
    target_class,
    functional_annotation
):

    target = clean_text(
        target_class
    ).upper()

    annotation = clean_text(
        functional_annotation
    ).lower()

    if target == "STRUCTURAL/CHITIN":
        return 100

    if target == "TARGET-SITE":
        return 95

    if target == "DIGESTION":

        if any(
            x in annotation
            for x in [
                "trypsin",
                "chymotrypsin",
                "lipase",
                "collagenase",
                "serine protease"
            ]
        ):

            return 90

        return 70

    if target == "SIGNALING":
        return 75

    if target:
        return 50

    return 0

def create_functional_annotation(
    functional_annotation,
    gene_family,
    target_class
):

    source = clean_text(
        functional_annotation
    )

    family = clean_text(
        gene_family
    )

    target = clean_text(
        target_class
    ).upper()

    if not source:
        source = family

    if not source:
        return "Uncharacterized protein with limited functional annotation evidence"

    # Remove database-style extras.

    source = re.sub(
        r"\([^)]*\)",
        "",
        source
    )

    source = re.sub(
        r"\s+",
        " ",
        source
    ).strip()

    # Remove EC numbers and similar trailing information.

    source = re.sub(
        r"\s+EC=\S+.*$",
        "",
        source,
        flags=re.IGNORECASE
    )

    words = source.split()

    # If already within requested range.

    if 7 <= len(words) <= 15:
        return source

    # Build a concise biological description.

    if "chitin synthase" in source.lower():

        text = (
            "Chitin synthase involved in insect cuticle formation "
            "and developmental moulting"
        )

    elif "trypsin" in source.lower():

        text = (
            "Trypsin-like serine protease involved in larval "
            "dietary protein digestion"
        )

    elif "chymotrypsin" in source.lower():

        text = (
            "Chymotrypsin-like serine protease involved in "
            "larval digestive protein hydrolysis"
        )

    elif "lipase" in source.lower():

        text = (
            "Lipase involved in dietary lipid hydrolysis "
            "during larval feeding"
        )

    elif "collagenase" in source.lower():

        text = (
            "Collagenase-like serine protease associated with "
            "protein digestion during larval feeding"
        )

    elif "protease inhibitor" in source.lower():

        text = (
            "Protease inhibitor regulating proteolytic activity "
            "within insect physiological processes"
        )

    elif target == "TARGET-SITE":

        text = (
            f"{family or 'Receptor-like protein'} associated with "
            "insect neuronal signalling and physiological regulation"
        )

    elif target == "SIGNALING":

        text = (
            f"{family or 'Signalling protein'} involved in "
            "cellular signal transduction and developmental regulation"
        )

    else:

        # Use first 15 words if possible.

        text = " ".join(
            words[:15]
        )

    # Enforce 7–15 words.

    words = text.split()

    if len(words) < 7:

        additions = [
            "in insect biological processes",
            "during normal insect development",
            "with predicted biological activity"
        ]

        for addition in additions:

            words.extend(
                addition.split()
            )

            if len(words) >= 7:
                break

    return " ".join(
        words[:15]
    )

def determine_phylogenetic_group(row):

    group = clean_text(
        row.get(
            "phylogenetic_group",
            ""
        )
    )

    uf = row.get(
        "ufboot_mean_pct",
        np.nan
    )

    sh = row.get(
        "sh_alrt_mean_pct",
        np.nan
    )

    if group:

        if pd.notna(uf):

            return (
                f"{group} "
                f"(UFBoot {float(uf):.0f}%)"
            )

        return group

    if not row.get(
        "phylo_success",
        False
    ):

        return "Not resolved"

    if pd.notna(uf):

        if pd.notna(sh):

            return (
                "Supported phylogenetic placement "
                f"(UFBoot {float(uf):.0f}%; "
                f"SH-aLRT {float(sh):.0f}%)"
            )

        return (
            "Supported phylogenetic placement "
            f"(UFBoot {float(uf):.0f}%)"
        )

    return "Phylogenetic placement obtained"

def phylo_evidence(row):

    if not row["phylo_success"]:
        return ""

    model = clean_text(
        row.get(
            "iqtree_model",
            ""
        )
    )

    uf = row.get(
        "ufboot_mean_pct",
        np.nan
    )

    sh = row.get(
        "sh_alrt_mean_pct",
        np.nan
    )

    text = (
        "MAFFT + trimAl + IQ-TREE"
    )

    if model:
        text += (
            f"; model={model}"
        )

    if pd.notna(uf):

        text += (
            f"; UFBoot={float(uf):.2f}%"
        )

    if pd.notna(sh):

        text += (
            f"; SH-aLRT={float(sh):.2f}%"
        )

    return text

def generate_justification(
    gene_family,
    functional_annotation,
    target_class,
    larva_tpm,
    adult_tpm,
    pupa_tpm,
    egg_tpm,
    phylogenetic_group
):

    family = clean_text(
        gene_family
    )

    annotation = clean_text(
        functional_annotation
    )

    target = clean_text(
        target_class
    ).upper()

    stages = {
        "larval": larva_tpm,
        "adult": adult_tpm,
        "pupal": pupa_tpm,
        "egg": egg_tpm
    }

    valid = {
        k: float(v)
        for k, v in stages.items()
        if pd.notna(v)
    }

    larva = (
        float(larva_tpm)
        if pd.notna(larva_tpm)
        else 0.0
    )

    adult = (
        float(adult_tpm)
        if pd.notna(adult_tpm)
        else 0.0
    )

    pupa = (
        float(pupa_tpm)
        if pd.notna(pupa_tpm)
        else 0.0
    )

    egg = (
        float(egg_tpm)
        if pd.notna(egg_tpm)
        else 0.0
    )

    non_larval = [
        adult,
        pupa,
        egg
    ]

    max_non_larval = max(
        non_larval
    )

    if larva > max_non_larval * 2:

        expression_sentence = (
            f"RNA expression is strongly larva-biased "
            f"(Larva TPM {larva:.1f} versus Adult {adult:.1f}, "
            f"Pupa {pupa:.1f}, and Egg {egg:.1f})."
        )

    elif larva > max_non_larval:

        expression_sentence = (
            f"RNA expression is highest in larvae "
            f"(Larva TPM {larva:.1f}; Adult {adult:.1f}, "
            f"Pupa {pupa:.1f}, Egg {egg:.1f})."
        )

    else:

        expression_sentence = (
            f"RNA expression is detectable across stages "
            f"(Larva TPM {larva:.1f}, Adult {adult:.1f}, "
            f"Pupa {pupa:.1f}, Egg {egg:.1f})."
        )

    if (
        "chitin synthase" in family.lower()
        or "chitin synthase" in annotation.lower()
    ):

        return (
            "Cuticular chitin synthase supports chitin production "
            "required for insect cuticle formation and moulting. "
            f"{expression_sentence} "
            "Reducing chitin synthesis could compromise cuticle "
            "integrity and interfere with successful moulting. "
            "The combination of functional annotation, developmental "
            "relevance and phylogenetic support makes this a strong "
            "RNAi candidate."
        )

    if (
        "trypsin" in family.lower()
        or "trypsin" in annotation.lower()
    ):

        return (
            "Trypsin-like serine proteases contribute to dietary "
            "protein digestion in feeding insect larvae. "
            f"{expression_sentence} "
            "Knockdown could reduce proteolytic capacity and limit "
            "amino-acid acquisition during feeding. This provides "
            "a plausible mechanism for impaired larval growth "
            "and development."
        )

    if (
        "chymotrypsin" in family.lower()
        or "chymotrypsin" in annotation.lower()
    ):

        return (
            "Chymotrypsin-like serine proteases contribute to "
            "hydrolysis of dietary proteins in the larval gut. "
            f"{expression_sentence} "
            "RNAi-mediated reduction could decrease digestive "
            "capacity and nutrient acquisition. This makes the "
            "candidate biologically relevant for targeting the "
            "feeding stage of M. plana."
        )

    if (
        "lipase" in family.lower()
        or "lipase" in annotation.lower()
    ):

        return (
            "Lipases support hydrolysis of dietary lipids during "
            "larval feeding and nutrient acquisition. "
            f"{expression_sentence} "
            "Knockdown could reduce the availability of fatty acids "
            "needed for energy production and cellular processes. "
            "This provides a plausible digestive vulnerability "
            "for RNAi-based control."
        )

    if "collagenase" in annotation.lower():

        return (
            "The candidate is annotated as a collagenase-like "
            "serine protease with predicted proteolytic activity. "
            f"{expression_sentence} "
            "Knockdown could reduce digestive proteolysis and "
            "nutrient acquisition in feeding larvae. The predicted "
            "mechanism is plausible, although the specific substrate "
            "and phenotype require experimental validation."
        )

    if target == "TARGET-SITE":

        return (
            f"The candidate is annotated as {annotation or family}, "
            "consistent with a receptor or ion-channel function. "
            f"{expression_sentence} "
            "RNAi disruption could alter neuronal signalling or "
            "other physiological processes controlled by the target. "
            "The exact phenotype depends on the protein's biological "
            "role and requires experimental validation."
        )

    if target == "SIGNALING":

        return (
            f"The candidate is associated with {annotation or family} "
            "and may contribute to cellular signal transduction. "
            f"{expression_sentence} "
            "Knockdown could disrupt signalling processes required "
            "for normal development or physiological function. "
            "The specific downstream phenotype cannot be established "
            "from sequence evidence alone."
        )

    if "protease inhibitor" in family.lower():

        return (
            "The candidate encodes a protease inhibitor that may "
            "regulate proteolytic activity in the insect. "
            f"{expression_sentence} "
            "Knockdown could disturb the balance between proteases "
            "and their inhibitors, potentially affecting digestion "
            "or other physiological processes. Functional validation "
            "is needed to establish the relevant phenotype."
        )

    if target == "DIGESTION":

        return (
            f"The candidate is associated with {annotation or family} "
            "and is predicted to contribute to digestive function. "
            f"{expression_sentence} "
            "Knockdown could reduce the corresponding digestive "
            "activity and limit nutrient acquisition during feeding. "
            "This provides a plausible RNAi mechanism, although "
            "functional redundancy should be considered."
        )

    return (
        f"The candidate is annotated as {annotation or family or 'an uncharacterized protein'} "
        "with supporting sequence and functional evidence. "
        f"{expression_sentence} "
        f"The phylogenetic analysis provides {phylogenetic_group.lower() if phylogenetic_group else 'additional evolutionary support'}. "
        "The biological effect of RNAi remains to be confirmed experimentally."
    )

print("\n")
print("=" * 80)
print("FINAL TOP-10 CANDIDATE RANKING")
print("=" * 80)

print(
    "\nThe Top 100 RNA file is the authoritative candidate pool."
    "\nBLAST, InterPro and phylogeny provide supporting evidence."
)

rna_path = ask_file("TOP 100 RNA TSV")
blast_path = ask_file("TOP 500 BLAST-UniProt TSV")
interpro_path = ask_file("InterPro TSV")
phylo_path = ask_file("Phylogenetic annotation summary TSV")

output_dir = ask_output_directory()

if output_dir is None:
    output_dir = rna_path.parent

output_dir.mkdir(parents=True, exist_ok=True)

rna = load_tsv(rna_path, "Top 100 RNA")
blast = load_tsv(blast_path, "BLAST")
interpro = load_tsv(interpro_path, "InterPro")
phylo = load_tsv(
    phylo_path,
    "Phylogenetic annotation summary"
)

require_columns(
    rna,
    [
        "gene_id",
        "RNA_score",
        "Larva_TPM",
        "Adult_TPM",
        "Pupa_TPM",
        "Egg_TPM",
        "Larva_expression"
    ],
    "Top 100 RNA file"
)

require_columns(
    blast,
    [
        "gene_id",
        "hit_rank",
        "sseqid",
        "pident",
        "qcov",
        "evalue",
        "bitscore",
        "stitle"
    ],
    "BLAST file"
)

require_columns(
    interpro,
    [
        "Protein Accession",
        "Signature accession",
        "InterPro accession"
    ],
    "InterPro file"
)

require_columns(
    phylo,
    [
        "gene_id",
        "protein_id",
        "phylo_success",
        "phylo_sequence_count",
        "alignment_length",
        "trimmed_alignment_length",
        "iqtree_model",
        "ufboot_mean_pct",
        "sh_alrt_mean_pct"
    ],
    "Phylogenetic annotation summary"
)

if "phylogenetic_group" in phylo.columns:
    print("\n✓ phylogenetic_group detected.")
    print("  Named phylogenetic groups will be used.")
else:
    print("\nNOTE: phylogenetic_group column not found.")
    print(
        "      UFBoot/SH-aLRT support will be reported,"
        "\n      but no named clade will be invented."
    )

rna["gene_key"] = rna["gene_id"].map(normalize_gene_id)
blast["gene_key"] = blast["gene_id"].map(normalize_gene_id)
interpro["gene_key"] = (
    interpro["Protein Accession"].map(normalize_gene_id)
)
phylo["gene_key"] = phylo["gene_id"].map(normalize_gene_id)

rna = rna[
    rna["gene_key"].astype(str).str.strip() != ""
].copy()

rna = rna.drop_duplicates(
    subset=["gene_key"],
    keep="first"
).copy()

print(
    f"\nAuthoritative candidate pool: "
    f"{len(rna):,} genes"
)

candidate_genes = set(rna["gene_key"])

blast = blast[
    blast["gene_key"].isin(candidate_genes)
].copy()

interpro = interpro[
    interpro["gene_key"].isin(candidate_genes)
].copy()

phylo = phylo[
    phylo["gene_key"].isin(candidate_genes)
].copy()

rna["RNA_score_numeric"] = numeric(rna, "RNA_score")
rna["Larva_TPM_numeric"] = numeric(rna, "Larva_TPM")
rna["Adult_TPM_numeric"] = numeric(rna, "Adult_TPM")
rna["Pupa_TPM_numeric"] = numeric(rna, "Pupa_TPM")
rna["Egg_TPM_numeric"] = numeric(rna, "Egg_TPM")

blast["hit_rank_numeric"] = numeric(blast, "hit_rank")
blast["pident_numeric"] = numeric(blast, "pident")
blast["qcov_numeric"] = numeric(blast, "qcov")
blast["evalue_numeric"] = numeric(blast, "evalue")
blast["bitscore_numeric"] = numeric(blast, "bitscore")

phylo["ufboot_mean_pct"] = numeric(
    phylo,
    "ufboot_mean_pct"
)

phylo["sh_alrt_mean_pct"] = numeric(
    phylo,
    "sh_alrt_mean_pct"
)

blast = blast.sort_values(
    by=[
        "gene_key",
        "hit_rank_numeric",
        "evalue_numeric",
        "bitscore_numeric",
        "pident_numeric",
        "qcov_numeric"
    ],
    ascending=[
        True, True, True, False, False, False
    ],
    na_position="last"
)

best_blast = (
    blast
    .drop_duplicates(
        subset=["gene_key"],
        keep="first"
    )
    .copy()
)

print(f"\nBest BLAST hits selected: {len(best_blast):,}")

old_columns = [
    "best_hit_sseqid", "best_hit_accession",
    "best_hit_species", "best_hit_description",
    "best_identity_pct", "best_query_coverage_pct",
    "best_evalue", "best_bitscore",
    "functional_annotation", "gene_family",
    "subfamily_or_clade", "phylogenetic_group",
    "pfam_domains", "interpro_domains",
    "target_class", "offtarget_check"
]

rna = rna.drop(
    columns=[c for c in old_columns if c in rna.columns],
    errors="ignore"
)

best_blast_small = best_blast[
    [
        "gene_key",
        "sseqid",
        "pident_numeric",
        "qcov_numeric",
        "evalue_numeric",
        "bitscore_numeric",
        "stitle"
    ]
].copy()

best_blast_small = best_blast_small.rename(
    columns={
        "sseqid": "best_hit_sseqid",
        "pident_numeric": "best_identity_pct",
        "qcov_numeric": "best_query_coverage_pct",
        "evalue_numeric": "best_evalue",
        "bitscore_numeric": "best_bitscore",
        "stitle": "best_hit_description"
    }
)

merged = rna.merge(
    best_blast_small,
    on="gene_key",
    how="left",
    validate="one_to_one"
)

phylo_columns = [
    "gene_key",
    "protein_id",
    "phylo_success",
    "phylo_sequence_count",
    "alignment_length",
    "trimmed_alignment_length",
    "iqtree_model",
    "ufboot_mean_pct",
    "sh_alrt_mean_pct"
]

if "phylogenetic_group" in phylo.columns:

    phylo_columns.append(
        "phylogenetic_group"
    )


phylo_small = (
    phylo[
        phylo_columns
    ]
    .drop_duplicates(
        subset=["gene_key"],
        keep="first"
    )
    .copy()
)


merged = merged.merge(
    phylo_small,
    on="gene_key",
    how="left",
    validate="one_to_one"
)

interpro_records = {}

for gene, group in interpro.groupby(
    "gene_key"
):

    pfam = extract_pfam(
        group
    )

    ipr = extract_interpro(
        group
    )

    ipr_desc = extract_interpro_descriptions(
        group
    )

    interpro_records[gene] = {

        "pfam":
            pfam,

        "interpro":
            ipr,

        "interpro_desc":
            ipr_desc,

        "interpro_count":
            len([
                x
                for x in ipr.split(";")
                if x
            ]),

        "pfam_count":
            len([
                x
                for x in pfam.split(";")
                if x
            ])
    }

rows = []

for _, row in merged.iterrows():

    gene = row["gene_key"]

    ipr_info = interpro_records.get(
        gene,
        {
            "pfam": "",
            "interpro": "",
            "interpro_desc": "",
            "interpro_count": 0,
            "pfam_count": 0
        }
    )

    raw_function = extract_function(
        row.get("best_hit_description", "")
    )

    gene_family = determine_gene_family(
        raw_function,
        ipr_info["interpro_desc"],
        ipr_info["pfam"]
    )

    existing_target = row.get("target_class", "")

    target_class = determine_target_class(
        raw_function,
        gene_family,
        ipr_info["interpro_desc"],
        existing_target
    )

    functional_annotation = create_functional_annotation(
        raw_function,
        gene_family,
        target_class
    )

    identity = row.get("best_identity_pct", np.nan)
    coverage = row.get("best_query_coverage_pct", np.nan)
    evalue = row.get("best_evalue", np.nan)
    bitscore = row.get("best_bitscore", np.nan)

    homology_score = calculate_homology_score(
        identity,
        coverage,
        evalue
    )

    annotation_score = calculate_annotation_score(
        ipr_info["interpro_count"],
        ipr_info["pfam_count"],
        functional_annotation
    )

    phylo_success = (
        str(row.get("phylo_success", ""))
        .strip()
        .lower()
        in {"true", "1", "yes", "success"}
    )

    phylo_score = calculate_phylo_score(
        {
            "phylo_success": phylo_success,
            "ufboot_mean_pct": row.get(
                "ufboot_mean_pct",
                np.nan
            ),
            "sh_alrt_mean_pct": row.get(
                "sh_alrt_mean_pct",
                np.nan
            )
        }
    )

    phylogenetic_group = determine_phylogenetic_group(
        {
            "phylo_success": phylo_success,
            "phylogenetic_group": row.get(
                "phylogenetic_group",
                ""
            ),
            "ufboot_mean_pct": row.get(
                "ufboot_mean_pct",
                np.nan
            ),
            "sh_alrt_mean_pct": row.get(
                "sh_alrt_mean_pct",
                np.nan
            )
        }
    )

    phylo_text = phylo_evidence(
        {
            "phylo_success": phylo_success,
            "iqtree_model": row.get(
                "iqtree_model",
                ""
            ),
            "ufboot_mean_pct": row.get(
                "ufboot_mean_pct",
                np.nan
            ),
            "sh_alrt_mean_pct": row.get(
                "sh_alrt_mean_pct",
                np.nan
            )
        }
    )

    rna_score = calculate_rna_score(row)

    larva_tpm = row.get("Larva_TPM_numeric", np.nan)
    adult_tpm = row.get("Adult_TPM_numeric", np.nan)
    pupa_tpm = row.get("Pupa_TPM_numeric", np.nan)
    egg_tpm = row.get("Egg_TPM_numeric", np.nan)

    all_stage_tpm = [
        larva_tpm,
        adult_tpm,
        pupa_tpm,
        egg_tpm
    ]

    larva_expression_score = calculate_stage_expression_score(
        larva_tpm,
        all_stage_tpm
    )

    adult_expression_score = calculate_stage_expression_score(
        adult_tpm,
        all_stage_tpm
    )

    pupa_expression_score = calculate_stage_expression_score(
        pupa_tpm,
        all_stage_tpm
    )

    egg_expression_score = calculate_stage_expression_score(
        egg_tpm,
        all_stage_tpm
    )

    target_score = calculate_target_score(
        target_class,
        functional_annotation
    )

    final_score = (
        rna_score * WEIGHT_RNA / 100
        + homology_score * WEIGHT_HOMOLOGY / 100
        + annotation_score * WEIGHT_ANNOTATION / 100
        + phylo_score * WEIGHT_PHYLO / 100
        + target_score * WEIGHT_TARGET / 100
    )

    best_species = extract_species(
        row.get("best_hit_description", "")
    )

    protein_id = clean_text(
        row.get("protein_id", "")
    )

    if not protein_id:
        protein_id = clean_text(
            row.get("gene_id", "")
        )

    larval_expression = clean_text(
        row.get("Larva_expression", "")
    )

    justification = generate_justification(
        gene_family,
        functional_annotation,
        target_class,
        larva_tpm,
        adult_tpm,
        pupa_tpm,
        egg_tpm,
        phylogenetic_group
    )

    rows.append({
        "_gene_key": gene,

        "gene_id": clean_text(
            row.get("gene_id", "")
        ),

        "protein_id": protein_id,

        "protein_length_aa": clean_text(
            row.get("sequence_length", "")
        ),

        "gene_family": gene_family,
        "phylogenetic_group": phylogenetic_group,

        "best_hit_accession": extract_accession(
            row.get("best_hit_sseqid", "")
        ),

        "best_hit_species": best_species,

        "pfam_domains": ipr_info["pfam"],
        "interpro_domains": ipr_info["interpro"],

        "functional_annotation": functional_annotation,
        "target_class": target_class,

        "evalue": (
            evalue if pd.notna(evalue) else ""
        ),

        "identity_pct": (
            identity if pd.notna(identity) else ""
        ),

        "query_coverage_pct": (
            coverage if pd.notna(coverage) else ""
        ),

        "phylogenetic_evidence": phylo_text,

        "Larva_TPM": (
            larva_tpm if pd.notna(larva_tpm) else ""
        ),

        "Adult_TPM": (
            adult_tpm if pd.notna(adult_tpm) else ""
        ),

        "Pupa_TPM": (
            pupa_tpm if pd.notna(pupa_tpm) else ""
        ),

        "Egg_TPM": (
            egg_tpm if pd.notna(egg_tpm) else ""
        ),

        "larva_expression_score":
            larva_expression_score,

        "adult_expression_score":
            adult_expression_score,

        "pupa_expression_score":
            pupa_expression_score,

        "egg_expression_score":
            egg_expression_score,

        "Larva_expression": larval_expression,

        "rnai_or_chemistry": "RNAi",

        "justification": justification,

        # INTERNAL

        "_rna_score": rna_score,
        "_homology_score": homology_score,
        "_annotation_score": annotation_score,
        "_phylo_score": phylo_score,
        "_target_score": target_score,
        "_final_score": final_score,

        "_rna_rank": row.get(
            "RNA_rank_numeric",
            np.nan
        )
    })

results = pd.DataFrame(rows)

results = results.sort_values(
    by=[
        "_final_score",
        "_rna_score",
        "_homology_score",
        "_annotation_score",
        "_phylo_score",
        "_target_score",
        "larva_expression_score"
    ],

    ascending=[
        False,
        False,
        False,
        False,
        False,
        False,
        False
    ],

    na_position="last"
).reset_index(
    drop=True
)

results.insert(
    0,
    "rank",
    range(
        1,
        len(results) + 1
    )
)

top10 = results.head(
    TOP_N
).copy()

internal_columns = [
    c
    for c in results.columns
    if c.startswith("_")
]

results_public = results.drop(
    columns=internal_columns
)

top10_public = top10.drop(
    columns=internal_columns
)

final_columns = [

    "rank",
    "gene_id",
    "protein_id",
    "protein_length_aa",

    "gene_family",
    "phylogenetic_group",

    "best_hit_accession",
    "best_hit_species",

    "pfam_domains",
    "interpro_domains",

    "functional_annotation",
    "target_class",

    "evalue",
    "identity_pct",
    "query_coverage_pct",

    "phylogenetic_evidence",

    "Larva_TPM",
    "Adult_TPM",
    "Pupa_TPM",
    "Egg_TPM",

    "larva_expression_score",
    "adult_expression_score",
    "pupa_expression_score",
    "egg_expression_score",

    "Larva_expression",

    "rnai_or_chemistry",

    "justification"
]


top10_public = top10_public[
    final_columns
]

results_public = results_public[
    final_columns
]

top10_tsv = (
    output_dir
    / "FINAL_TOP10_CANDIDATES.tsv"
)

top10_csv = (
    output_dir
    / "FINAL_TOP10_CANDIDATES.csv"
)

full_tsv = (
    output_dir
    / "RANKED_TOP100_CANDIDATES.tsv"
)

top10_public.to_csv(
    top10_tsv,
    sep="\t",
    index=False
)

top10_public.to_csv(
    top10_csv,
    index=False
)

results_public.to_csv(
    full_tsv,
    sep="\t",
    index=False
)

print("\n" + "=" * 80)
print("FINAL TOP 10")
print("=" * 80)

display_columns = [
    "rank",
    "gene_id",
    "gene_family",
    "phylogenetic_group",
    "target_class"
]

print(
    top10_public[
        display_columns
    ].to_string(
        index=False
    )
)

print("\n" + "=" * 80)
print("TOP 10 SCORE BREAKDOWN")
print("=" * 80)

score_display = results[
    [
        "rank",
        "gene_id",
        "_rna_score",
        "_homology_score",
        "_annotation_score",
        "_phylo_score",
        "_target_score",
        "_final_score"
    ]
].head(
    TOP_N
).copy()

score_display.columns = [
    "rank",
    "gene_id",
    "RNA_score",
    "Homology_score",
    "Annotation_score",
    "Phylo_score",
    "Target_score",
    "Final_score"
]

print(
    score_display.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("\n" + "=" * 80)
print("OUTPUT COMPLETE")
print("=" * 80)

print(f"\nAuthoritative candidate pool: {len(rna)} genes")
print(f"Final ranked candidates: {len(results_public)}")
print(f"Final Top 10 candidates: {len(top10_public)}")

print("\nTop 10 TSV:")
print(top10_tsv)

print("\nTop 10 CSV:")
print(top10_csv)

print("\nFull ranked Top 100 TSV:")
print(full_tsv)

print("\n" + "=" * 80)
print("FINAL VALIDATION")
print("=" * 80)

# Candidate count
assert len(rna) == len(results_public), (
    f"RNA candidate count ({len(rna)}) does not match "
    f"ranked candidate count ({len(results_public)})"
)

# Top 10
assert len(top10_public) == 10, (
    f"Expected 10 final candidates, found {len(top10_public)}"
)

# Unique genes
assert results_public["gene_id"].nunique() == len(results_public), (
    "Duplicate gene IDs detected."
)

# Unique ranks
assert results_public["rank"].nunique() == len(results_public), (
    "Duplicate ranks detected."
)

# Rank sequence
assert list(results_public["rank"]) == list(
    range(1, len(results_public) + 1)
), "Rank sequence is invalid."

# Functional annotation length
annotation_word_counts = (
    results_public["functional_annotation"]
    .fillna("")
    .apply(lambda x: len(str(x).split()))
)

assert annotation_word_counts.between(7, 15).all(), (
    "Functional annotation contains entries outside the 7–15 word range."
)

# BLAST coverage
blast_coverage = merged["best_hit_sseqid"].notna().sum()

print(f"\n✓ RNA candidate pool = {len(rna)}")
print(f"✓ Ranked candidates = {len(results_public)}")
print(f"✓ Final Top 10 = {len(top10_public)}")
print(f"✓ Unique gene IDs = {results_public['gene_id'].nunique()}")
print(f"✓ Candidates with BLAST evidence = {blast_coverage}")
print("✓ BLAST hit_rank=1 used as best hit")
print("✓ InterPro/Pfam used as annotation evidence")
print("✓ MAFFT + trimAl + IQ-TREE used as phylogenetic evidence")
print("✓ UFBoot and SH-aLRT retained")
print("✓ Named phylogenetic groups used only when provided")
print("✓ No unsupported CHS-A/CHS-B assignments invented")
print("✓ Stage-specific RNA expression retained")
print("✓ Off-target column removed")
print("✓ Functional annotations constrained to 7–15 words")
print("✓ Justifications generated as natural 3–4 sentence explanations")
print("✓ Exactly 10 candidates exported")
print("\nDone.")



FINAL TOP-10 CANDIDATE RANKING

The Top 100 RNA file is the authoritative candidate pool.
BLAST, InterPro and phylogeny provide supporting evidence.

Loading Top 100 RNA...
Top 100 RNA rows loaded: 72

Loading BLAST...
BLAST rows loaded: 749

Loading InterPro...
InterPro rows loaded: 674

Loading Phylogenetic annotation summary...
Phylogenetic annotation summary rows loaded: 72

NOTE: phylogenetic_group column not found.
      UFBoot/SH-aLRT support will be reported,
      but no named clade will be invented.

Authoritative candidate pool: 72 genes

Best BLAST hits selected: 72

FINAL TOP 10
 rank gene_id                                                   gene_family                                         phylogenetic_group target_class
    1      g4                                                        Lipase Supported phylogenetic placement (UFBoot 91%; SH-aLRT 90%)    DIGESTION
    2     g94                           Peptidase M1 family aminopeptidases Supported phylogenetic plac


# Final Analysis Output

The completed workflow produces a prioritised set of *Metisa plana* protein candidates supported by multiple computational evidence layers.

The final candidate list can be used as the primary output for downstream biological interpretation and reporting.

## Evidence considered

- Protein sequence quality
- Homology evidence
- Similarity and alignment thresholds
- Taxonomic relevance
- Functional domain annotation
- Candidate-level evidence
- Developmental-stage expression
- Phylogenetic relationships
- Integrated candidate ranking

The final ranking represents the culmination of the computational screening workflow.
